# Прогноз швидкості адопції тварин за фото та описом профілю

**Виконав:** Олександр Коновалюк

## Мета

За профілем тварини на PetFinder потрібно передбачити, наскільки швидко її усиновлять. На вході є фотографії (одна або кілька на тварину), текстовий опис і, якщо вони присутні в таблиці, структуровані ознаки на кшталт віку, породи чи стану здоров'я. На виході — категорія швидкості адопції.

## Метрика

Офіційна метрика змагання — **quadratic weighted kappa** (квадратично зважена каппа). Вона враховує порядок класів: помилка на сусідню категорію дешевша, ніж прогноз «дуже швидко» замість «дуже повільно». Для максимального балу змагання потрібен результат **не нижче 0.5**. Якщо результат нижчий, бал рахується пропорційно до цього порогу.

Файл відповіді — `submission.csv` з колонками `PetID`, `AdoptionSpeed` і з одним рядком на кожен `PetID` з `test.csv`. `sample_submission.csv` задає лише назви колонок. Якщо в ньому лише кілька прикладів, а тест довший, відповідь усе одно пишеться на весь тест: інакше перевірка відхиляє файл через нестачу рядків.

## Як влаштований експеримент

Моделі порівнюються на одному стратифікованому розбитті, щоб різниця в метриці належала архітектурі, а не іншій вибірці.

1. **Нижня межа на тексті.** Мішок слів часто вже містить прямі сигнали: вік, здоров'я, характер, прохання забрати тварину.
2. **Лише зображення.** Перенесені ознаки згорткової мережі, навченої на великому наборі фотографій. Власну мережу з нуля на кількох тисячах знімків легко перенавчити під шум фону.
3. **Злиття.** Рішення усиновити тварину залежить і від того, як вона виглядає, і від того, що про неї написано. Окремі модальності цього не покривають.

Кожен експеримент лишається в ноутбуці й дописує рядок у журнал. Наступна архітектура або інший набір гіперпараметрів додається **новими комірками нижче**, без заміни попередніх блоків. Висновки після цього прогону заповнюються за надрукованими числами, а не заздалегідь.

Пошук гіперпараметрів окремим перебором тут свідомо не запускається: спочатку потрібно побачити, яка модальність узагалі несе сигнал. Перебирати швидкість навчання на слабкій архітектурі означає підігнати один валідаційний спліт.


## 1. Середовище, шляхи та гіперпараметри

Ноутбук розрахований на Kaggle з увімкненим GPU і Internet: ваги ResNet18 завантажуються під час першого прогону. Ознаки зображень кешуються у робочій директорії, щоб повторний запуск і наступні експерименти не рахували згорткову мережу знову.

Гіперпараметри першого прогону фіксовані й записані нижче, щоб пізніше можна було змінити одну величину в новому блоці й порівняти її з цим рядком журналу.

| Параметр | Значення | Навіщо |
| --- | --- | --- |
| Частка валідації | 20%, стратифіковано | У кожному класі зберігається та сама частка, інакше QWK стрибає через рідкісний клас |
| Максимум епох | 8 | Верхня межа, а не ціль |
| Patience | 2 епохи | Зупинка, якщо val QWK не зріс |
| Мінімальний приріст QWK | 0.001 | Дрібніший зсув на цьому розмірі вибірки не відрізняється від шуму |
| Dropout | 0.3 | Випадково вимикає частини вектора перед класифікатором, щоб голова не запам'ятовувала навчальні профілі |
| AdamW, lr, weight decay | 1e-3, 1e-4 | Вагове загасання тримає ваги малими на невеликій вибірці |
| Ембединг слова / прихований стан LSTM | 128 / 128, 1 шар | Порядку слів достатньо одного двонаправленого шару: текстів кілька тисяч |
| Довжина тексту | 128 токенів | Опис профілю рідко несе рішення в самому кінці довгого тексту |
| Фото на тварину | до 3 | Перші за номером кадри: головне фото і кілька ракурсів, без хвоста з десятків майже однакових знімків |
| Розмір зображення | 224, нормалізація ImageNet | Масштаб, на якому навчені ваги ResNet |


In [ ]:
import copy
import os
import random
import re
import time
import warnings
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from IPython.display import display
from PIL import Image
from sklearn.exceptions import UndefinedMetricWarning
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, cohen_kappa_score, confusion_matrix
from sklearn.model_selection import train_test_split
from torch.nn.utils.rnn import pack_padded_sequence, pad_sequence
from torch.utils.data import DataLoader, Dataset
from torchvision.models import ResNet18_Weights, resnet18
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=UndefinedMetricWarning)
sns.set_theme(style="whitegrid")

SEED = 42
VAL_SIZE = 0.2
MAX_EPOCHS = 8
PATIENCE = 2
MIN_DELTA = 0.001
DROPOUT = 0.3
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
EMBED_DIM = 128
LSTM_HIDDEN = 128
MAX_TEXT_LEN = 128
MIN_WORD_FREQ = 2
MAX_IMAGES = 3
BATCH_SIZE = 32
FEATURE_BATCH_SIZE = 64
TFIDF_MAX_FEATURES = 20000
TFIDF_MIN_DF = 2
GRAD_CLIP = 5.0
TABULAR_MAX_CARDINALITY = 400

DATA_ROOT = Path("/kaggle/input/competitions/deep-learning-for-computer-vision-and-nlp-2026-10")
TRAIN_CSV = DATA_ROOT / "train.csv"
TEST_CSV = DATA_ROOT / "test.csv"
SAMPLE_CSV = DATA_ROOT / "sample_submission.csv"
IMAGE_DIR = DATA_ROOT / "images"
WORKING = Path("/kaggle/working")
FEATURE_CACHE = WORKING / "resnet18_pet_features.npz"
LOG_PATH = WORKING / "experiment_log.csv"
SUBMISSION_PATH = WORKING / "submission.csv"

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
NUMERIC_NAME_HINTS = ("age", "fee", "quantity", "qty", "photoamt", "videoamt", "price", "amount")


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(SEED)
WORKING.mkdir(parents=True, exist_ok=True)

for path in (TRAIN_CSV, TEST_CSV, SAMPLE_CSV, IMAGE_DIR):
    if not path.exists():
        raise FileNotFoundError(
            f"Не знайдено {path}. Ноутбук має запускатися в Kaggle з підключеним змаганням."
        )

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_WORKERS = 2 if os.name != "nt" else 0
print("torch", torch.__version__)
print("device", device)
if device.type == "cuda":
    print(torch.cuda.get_device_name(0))
else:
    print("GPU не видно. Витяг ознак ResNet на CPU буде повільним; для прогону змагання увімкніть GPU.")


## 2. Метрика, журнал експериментів і рання зупинка

Журнал — список рядків, який кожен експеримент доповнює і зберігає в `experiment_log.csv`. Старі рядки не переписуються.

Рання зупинка дивиться на **val QWK**, а не на функцію втрат. Крос-ентропія карає впевненість у неправильному класі, але не знає, що клас 1 ближчий до 2, ніж до 4. Саме QWK є тим, що піде в залік.

На кожній епосі порівнюються два способи перетворити ймовірності на мітку:

- **argmax** — клас із найбільшою ймовірністю;
- **expectation** — округлене математичне сподівання індексу класу. Якщо модель сумнівається між сусідніми категоріями, очікування рідше стрибає через одну категорію, а QWK такий стрибок карає квадратом відстані.

Кращий із двох на епосі визначає, чи є покращення. Поріг покращення — 0.001. Якщо дві епохи поспіль приросту немає, цикл зупиняється. У ваги повертається найкраща епоха, навіть якщо після неї loss ще падав: падіння loss без росту QWK тут не є успіхом.

Коли обрану модель навчаємо на всіх розмічених рядках, валідації вже немає. Тому число епох дорівнює номеру найкращої епохи, а не верхній межі 8.


In [ ]:
EXPERIMENT_ROWS = []
RUNS = {}


def quadratic_weighted_kappa(y_true, y_pred):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    if len(y_true) == 0:
        return 0.0
    score = cohen_kappa_score(y_true, y_pred, weights="quadratic")
    if score != score:
        return 0.0
    return float(score)


def decode_probabilities(probs, n_classes):
    class_index = np.arange(n_classes, dtype=np.float64)
    expectation = np.rint(probs @ class_index)
    expectation = np.clip(expectation, 0, n_classes - 1).astype(int)
    return {
        "argmax": probs.argmax(axis=1).astype(int),
        "expectation": expectation,
    }


def log_experiment(name, val_qwk, decoder, epochs_ran, best_epoch, notes=""):
    row = {
        "Експеримент": name,
        "Val QWK": round(float(val_qwk), 4),
        "Декодер": decoder,
        "Епох навчено": int(epochs_ran) if epochs_ran == epochs_ran else 0,
        "Найкраща епоха": int(best_epoch) if best_epoch == best_epoch else 0,
        "Нотатки": notes,
    }
    EXPERIMENT_ROWS.append(row)
    frame = pd.DataFrame(EXPERIMENT_ROWS)
    frame.to_csv(LOG_PATH, index=False)
    display(frame)
    return row


def count_parameters(model):
    return sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)


def class_weights_from_labels(labels, n_classes):
    counts = np.bincount(np.asarray(labels).astype(int), minlength=n_classes).astype(np.float32)
    weights = counts.sum() / (n_classes * np.maximum(counts, 1.0))
    return torch.tensor(weights, dtype=torch.float32, device=device)


def batch_move(batch):
    if torch.is_tensor(batch):
        return batch.to(device)
    if isinstance(batch, (tuple, list)):
        return type(batch)(batch_move(item) for item in batch)
    if isinstance(batch, dict):
        return {key: batch_move(value) for key, value in batch.items()}
    return batch


@torch.no_grad()
def collect_probabilities(model, loader, logits_from_batch):
    model.eval()
    total_loss = 0.0
    seen = 0
    prob_chunks = []
    label_chunks = []
    for batch in loader:
        logits, targets, loss = logits_from_batch(model, batch, compute_loss=True)
        batch_size = targets.size(0)
        total_loss += loss.item() * batch_size
        seen += batch_size
        prob_chunks.append(torch.softmax(logits, dim=1).cpu().numpy())
        label_chunks.append(targets.detach().cpu().numpy())
    avg_loss = total_loss / max(seen, 1)
    return avg_loss, np.concatenate(prob_chunks), np.concatenate(label_chunks)


def train_until_plateau(model, train_loader, val_loader, optimizer, logits_from_batch, n_classes):
    history = {
        "epoch": [],
        "train_loss": [],
        "val_loss": [],
        "qwk_argmax": [],
        "qwk_expectation": [],
    }
    best_qwk = -1.0
    best_epoch = 0
    best_decoder = "argmax"
    best_state = copy.deepcopy(model.state_dict())
    wait = 0
    epochs_ran = 0

    for epoch in range(1, MAX_EPOCHS + 1):
        epochs_ran = epoch
        model.train()
        total_loss = 0.0
        seen = 0
        for batch in train_loader:
            optimizer.zero_grad(set_to_none=True)
            logits, targets, loss = logits_from_batch(model, batch, compute_loss=True)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()
            batch_size = targets.size(0)
            total_loss += loss.item() * batch_size
            seen += batch_size
        train_loss = total_loss / max(seen, 1)

        val_loss, probs, y_true = collect_probabilities(model, val_loader, logits_from_batch)
        decoded = decode_probabilities(probs, n_classes)
        qwk_argmax = quadratic_weighted_kappa(y_true, decoded["argmax"])
        qwk_expectation = quadratic_weighted_kappa(y_true, decoded["expectation"])
        if qwk_expectation > qwk_argmax:
            epoch_decoder = "expectation"
            epoch_qwk = qwk_expectation
        else:
            epoch_decoder = "argmax"
            epoch_qwk = qwk_argmax

        history["epoch"].append(epoch)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["qwk_argmax"].append(qwk_argmax)
        history["qwk_expectation"].append(qwk_expectation)
        print(
            f"Епоха {epoch:02d}/{MAX_EPOCHS} | train loss {train_loss:.4f} | "
            f"val loss {val_loss:.4f} | QWK argmax {qwk_argmax:.4f} | "
            f"QWK expectation {qwk_expectation:.4f}"
        )

        if epoch_qwk > best_qwk + MIN_DELTA:
            best_qwk = epoch_qwk
            best_epoch = epoch
            best_decoder = epoch_decoder
            best_state = copy.deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1
            if wait >= PATIENCE:
                print(
                    f"Рання зупинка на епосі {epoch}: val QWK не зріс на {MIN_DELTA} "
                    f"протягом {PATIENCE} епох. Найкраща епоха {best_epoch}, "
                    f"QWK {best_qwk:.4f} ({best_decoder})."
                )
                break
    else:
        print(
            f"Досягнуто межу {MAX_EPOCHS} епох. Найкраща епоха {best_epoch}, "
            f"QWK {best_qwk:.4f} ({best_decoder})."
        )

    model.load_state_dict(best_state)
    return {
        "history": history,
        "best_qwk": best_qwk,
        "best_epoch": best_epoch,
        "best_decoder": best_decoder,
        "epochs_ran": epochs_ran,
    }


def train_fixed_epochs(model, train_loader, optimizer, logits_from_batch, epochs):
    epochs = max(int(epochs), 1)
    model.train()
    for epoch in range(1, epochs + 1):
        total_loss = 0.0
        seen = 0
        for batch in train_loader:
            optimizer.zero_grad(set_to_none=True)
            logits, targets, loss = logits_from_batch(model, batch, compute_loss=True)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()
            batch_size = targets.size(0)
            total_loss += loss.item() * batch_size
            seen += batch_size
        print(
            f"Повне навчання, епоха {epoch}/{epochs}, "
            f"train loss {total_loss / max(seen, 1):.4f}"
        )


def plot_confusion(y_true, y_pred, title):
    labels = list(range(N_CLASSES))
    matrix = confusion_matrix(y_true, y_pred, labels=labels)
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(
        matrix,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=[str(IDX_TO_LABEL[i]) for i in labels],
        yticklabels=[str(IDX_TO_LABEL[i]) for i in labels],
        ax=ax,
    )
    ax.set_xlabel("Прогноз")
    ax.set_ylabel("Факт")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()


def plot_training_history(history):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history["epoch"], history["train_loss"], marker="o", label="train")
    axes[0].plot(history["epoch"], history["val_loss"], marker="o", label="val")
    axes[0].set_title("Функція втрат")
    axes[0].set_xlabel("Епоха")
    axes[0].legend()
    axes[1].plot(history["epoch"], history["qwk_argmax"], marker="o", label="argmax")
    axes[1].plot(history["epoch"], history["qwk_expectation"], marker="o", label="expectation")
    axes[1].set_title("Val QWK")
    axes[1].set_xlabel("Епоха")
    axes[1].legend()
    plt.tight_layout()
    plt.show()


demo_true = np.array([0, 0, 1, 1, 2, 2, 3, 3])
demo_near = np.array([0, 1, 1, 2, 2, 3, 3, 3])
demo_far = np.array([3, 3, 3, 3, 0, 0, 0, 0])
print("Перевірка метрики на штучному прикладі, не на даних змагання.")
print("Повний збіг:", round(quadratic_weighted_kappa(demo_true, demo_true), 4))
print("Сусідні класи:", round(quadratic_weighted_kappa(demo_true, demo_near), 4))
print("Далекі класи:", round(quadratic_weighted_kappa(demo_true, demo_far), 4))


## 3. Завантаження таблиць

Схему файлів заздалегідь не фіксуємо в коді назвами всіх колонок. Ноутбук знаходить ідентифікатор, ціль і текстовий опис за назвами, а решту колонок розбирає пізніше: частину як числа, частину як категорії, ідентифікатори з дуже великою кількістю значень відкидає. Якщо текстової колонки немає, опис вважається порожнім, і це буде видно в журналі як слабкий текстовий експеримент, а не як прихована помилка.

Мітки переводяться в індекси `0 ... K-1` у порядку зростання початкових значень. У файл відповіді повертається початкова шкала. Очікування класу рахується в просторі цих індексів, тому порядок міток має збігатися з порядком швидкості адопції.


In [ ]:
def find_column(columns, options):
    lookup = {str(column).lower(): column for column in columns}
    for option in options:
        if option.lower() in lookup:
            return lookup[option.lower()]
    return None


def normalize_id(value):
    if pd.isna(value):
        return ""
    text = str(value).strip()
    if text.endswith(".0"):
        text = text[:-2]
    return text


def read_table(path):
    frame = pd.read_csv(path)
    frame.columns = [str(column).strip() for column in frame.columns]
    return frame


train_df = read_table(TRAIN_CSV)
test_df = read_table(TEST_CSV)
sample_df = read_table(SAMPLE_CSV)

id_col = find_column(train_df.columns, ["PetID", "pet_id", "id"])
target_col = find_column(train_df.columns, ["AdoptionSpeed", "adoption_speed", "label", "target"])
text_col = find_column(train_df.columns, ["Description", "description", "desc", "text", "comments"])
if id_col is None or target_col is None:
    raise RuntimeError(
        f"Не вдалося знайти колонки ідентифікатора і цілі. Колонки train: {list(train_df.columns)}"
    )

test_id_col = find_column(test_df.columns, ["PetID", "pet_id", "id"]) or id_col
if test_id_col not in test_df.columns:
    raise RuntimeError(f"У test.csv немає колонки ідентифікатора. Колонки: {list(test_df.columns)}")

train_df[id_col] = train_df[id_col].map(normalize_id)
test_df[test_id_col] = test_df[test_id_col].map(normalize_id)
duplicate_count = int(train_df[id_col].duplicated().sum())
if duplicate_count:
    print(f"Дублікати {id_col} у train: {duplicate_count}. Залишаю перший рядок, щоб один профіль не потрапив і в навчання, і у валідацію.")
    train_df = train_df.drop_duplicates(id_col, keep="first")

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

if target_col in test_df.columns:
    print(f"У test.csv є колонка {target_col}. Для навчання і сабміту вона не використовується.")
    test_df = test_df.drop(columns=[target_col])

test_text_col = find_column(test_df.columns, ["Description", "description", "desc", "text", "comments"])
if text_col is None:
    print("Текстової колонки в train не знайдено. Описи будуть порожніми.")
    text_col = "text_clean"
    train_df[text_col] = ""
    test_df[text_col] = ""
elif test_text_col is None:
    print("Текстової колонки в test не знайдено. Описи тесту будуть порожніми.")
    test_df[text_col] = ""
elif test_text_col != text_col:
    test_df[text_col] = test_df[test_text_col]

raw_target = train_df[target_col]
if pd.api.types.is_numeric_dtype(raw_target):
    numeric_target = raw_target.astype(float)
    if np.allclose(numeric_target.to_numpy(), np.rint(numeric_target.to_numpy())):
        train_df[target_col] = numeric_target.round().astype(int)

classes = sorted(train_df[target_col].dropna().unique().tolist())
LABEL_TO_IDX = {label: index for index, label in enumerate(classes)}
IDX_TO_LABEL = {index: label for label, index in LABEL_TO_IDX.items()}
N_CLASSES = len(classes)
train_df = train_df.dropna(subset=[target_col]).reset_index(drop=True)
y_all = train_df[target_col].map(LABEL_TO_IDX).to_numpy(dtype=int)

label_gaps = np.diff(np.asarray(classes, dtype=float))
print("Колонка id:", id_col)
print("Колонка цілі:", target_col)
print("Колонка тексту:", text_col)
print("Відповідність міток індексам:", LABEL_TO_IDX)
if len(label_gaps) and not np.allclose(label_gaps, 1.0):
    print("Крок між мітками не дорівнює 1. Очікування рахується за рангом класу, у файл повертається початкова мітка.")

print("train", train_df.shape, "test", test_df.shape)
print(train_df.dtypes)
display(train_df.head(3))
display(sample_df.head(3))


## 4. Прив'язка фотографій

Імена файлів у каталозі знімків можуть бути `{PetID}.jpg` або `{PetID}-1.jpg` / `{PetID}_1.jpg`. Спочатку перевіряється точний збіг імені без розширення з ідентифікатором. Якщо його немає, від імені відрізається останній числовий суфікс після `-` або `_`.

Для моделі беруться не більше ніж три перші кадри за цим номером: зазвичай це головне фото профілю і кілька додаткових ракурсів. Решта кадрів лишається в статистиці розвідки, але не в ознаках, інакше тварини з десятками майже однакових фото домінують у середньому векторі й у часі прогону.


In [ ]:
def list_image_files(image_dir):
    files = []
    for path in Path(image_dir).rglob("*"):
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS:
            files.append(path)
    return files


def link_images(pet_ids, files):
    pet_set = set(pet_ids)
    grouped = {pet_id: [] for pet_id in pet_set}
    for path in files:
        stem = path.stem
        pet_id = None
        order = 0
        if stem in pet_set:
            pet_id = stem
        else:
            for separator in ("-", "_"):
                if separator not in stem:
                    continue
                base, rest = stem.rsplit(separator, 1)
                if base in pet_set:
                    pet_id = base
                    order = int(rest) if rest.isdigit() else 10**6
                    break
        if pet_id is not None:
            grouped[pet_id].append((order, path))
    linked = {}
    for pet_id, items in grouped.items():
        items.sort(key=lambda item: (item[0], item[1].name))
        linked[pet_id] = [path for _, path in items]
    return linked


image_files = list_image_files(IMAGE_DIR)
extension_counts = Counter(path.suffix.lower() for path in image_files)
print("Файлів зображень:", len(image_files))
print("Розширення:", dict(extension_counts))
print("Приклади імен:", [path.name for path in image_files[:12]])

all_pet_ids = list(dict.fromkeys(list(train_df[id_col]) + list(test_df[test_id_col])))
full_image_paths = link_images(all_pet_ids, image_files)
model_image_paths = {
    pet_id: paths[:MAX_IMAGES] for pet_id, paths in full_image_paths.items()
}

train_ids = train_df[id_col].tolist()
test_ids = test_df[test_id_col].tolist()
train_photo_counts = np.array([len(full_image_paths[pet_id]) for pet_id in train_ids])
train_with_photo = int((train_photo_counts > 0).sum())
test_photo_counts = np.array([len(full_image_paths[pet_id]) for pet_id in test_ids])
print(
    f"Train із хоча б одним фото: {train_with_photo}/{len(train_ids)} "
    f"({train_with_photo / max(len(train_ids), 1):.1%})"
)
print(
    f"Test із хоча б одним фото: {int((test_photo_counts > 0).sum())}/{len(test_ids)} "
    f"({(test_photo_counts > 0).mean():.1%})"
)
print("Фото на тварину в train: min", int(train_photo_counts.min()),
      "median", float(np.median(train_photo_counts)),
      "max", int(train_photo_counts.max()))


## 5. Розвідувальний аналіз

Перед моделями потрібно побачити три речі, від яких залежить архітектура.

- Чи збалансовані класи. Якщо один домінує, класифікатор без ваг класів зсувається до нього, а QWK за такий зсув майже не платить, бо далекі рідкісні класи все одно промахуються.
- Чи є порожні описи і якою мовою вони написані. Очищення нижче розраховане на англійський текст: скорочення, латиниця, англійські стоп-слова.
- Скільки тварин узагалі мають фото. Якщо частка без знімків велика, зображення не може бути єдиним джерелом прогнозу, і в ознаку додається прапорець відсутності фото.


In [ ]:
class_counts = train_df[target_col].value_counts().sort_index()
print("Розподіл цілі:")
print(class_counts)
print("Частка найчастішого класу:", round(float(class_counts.max() / class_counts.sum()), 4))

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(class_counts.index.astype(str), class_counts.values, color="steelblue")
ax.set_xlabel(target_col)
ax.set_ylabel("Кількість")
ax.set_title("Розподіл швидкості адопції")
for index, value in enumerate(class_counts.values):
    ax.text(index, value, str(int(value)), ha="center", va="bottom")
plt.tight_layout()
plt.show()

missing_rate = train_df.isna().mean().sort_values(ascending=False)
missing_rate = missing_rate[missing_rate > 0]
print("Частка пропусків у train (лише колонки, де вони є):")
display(missing_rate.to_frame("missing_rate"))

if text_col in train_df.columns:
    raw_text = train_df[text_col].fillna("").astype(str)
    empty_text = raw_text.str.strip().eq("").mean()
    word_counts = raw_text.map(lambda text: len(text.split()))
    latin_chars = raw_text.map(lambda text: len(re.findall(r"[A-Za-z]", text)))
    all_chars = raw_text.map(lambda text: max(len(re.findall(r"[^\s]", text)), 1))
    nonempty = raw_text.str.strip().ne("")
    if nonempty.any():
        latin_share = float((latin_chars[nonempty] / all_chars[nonempty]).mean())
    else:
        latin_share = 0.0
    print(f"Порожніх описів: {empty_text:.1%}")
    print(f"Середня частка латинських літер серед непробільних символів непорожніх описів: {latin_share:.1%}")
    print(word_counts.describe())
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(word_counts.clip(upper=word_counts.quantile(0.99)), bins=40, color="steelblue")
    ax.set_xlabel("Слів в описі")
    ax.set_ylabel("Тварин")
    ax.set_title("Довжина опису")
    plt.tight_layout()
    plt.show()

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(train_photo_counts, bins=min(30, int(train_photo_counts.max()) + 1), color="steelblue")
ax.set_xlabel("Кількість фото")
ax.set_ylabel("Тварин")
ax.set_title("Скільки фото прив'язано до одного профілю")
plt.tight_layout()
plt.show()

type_col = find_column(train_df.columns, ["Type", "type", "Animal", "Species"])
if type_col is not None:
    share = pd.crosstab(train_df[type_col], train_df[target_col], normalize="index")
    print(f"Частка класів швидкості всередині колонки {type_col}:")
    display(share.round(3))

preview_ids = [pet_id for pet_id in train_ids if full_image_paths[pet_id]]
if preview_ids:
    generator = np.random.default_rng(SEED)
    chosen = generator.choice(preview_ids, size=min(6, len(preview_ids)), replace=False)
    fig, axes = plt.subplots(2, 3, figsize=(12, 8))
    id_to_label = dict(zip(train_ids, train_df[target_col].tolist()))
    for ax, pet_id in zip(axes.ravel(), list(chosen) + [None] * 6):
        ax.axis("off")
        if pet_id is None:
            continue
        try:
            with Image.open(full_image_paths[pet_id][0]) as image:
                ax.imshow(image.convert("RGB"))
            ax.set_title(f"{pet_id}\n{target_col}={id_to_label[pet_id]}")
        except Exception as exc:
            ax.set_title(f"Не вдалося відкрити {pet_id}: {exc}")
    plt.tight_layout()
    plt.show()
else:
    print("Жодному train PetID не знайдено файлу зображення. Перевірте шаблон імен вище.")


### Висновки за розвідувальним аналізом

У таблиці немає окремих колонок віку, породи чи здоров'я. Train містить 6431 рядок і лише PetID, Description, AdoptionSpeed. Test містить 1891 рядок без мітки. Усе, що модель може використати, є в тексті опису і на фото.

Класи 1–4 розподілені так: 1197, 1773, 1328 і 2133. Найповільніша адопція (4) займає 33.2%. Це не крайній дисбаланс, але константний прогноз цього класу дає QWK 0: метрика не платить за вгадування моди.

Описи майже повні: порожніх 0.1%, після очищення 0.26%. Серед непробільних символів 94.6% — латиниця, тож англійське очищення доречне. Типовий опис має 46 слів (медіана), найдовший — 1153. Обрізання до 128 токенів зачіпає довгий хвіст, не типовий профіль.

Фото названі як PetID-номер.jpg, у каталозі 37920 файлів. Фото є в кожного з 6431 навчальних профілів і в 1887 з 1891 тестових. Медіана — 4 кадри, максимум — 30. Обмеження моделі першими трьома кадрами відкидає щонайменше одне фото в половини тварин.


## 6. Очищення тексту і розбиття

Опис профілю чистять так:

- розкриваються англійські скорочення, щоб `don't` і `do not` не стали різними ознаками;
- лишається латиниця в нижньому регістрі;
- прибираються стоп-слова, але **не** заперечення (`not`, `no`, `never`, `without`). Фраза «not vaccinated» без `not` змінює зміст на протилежний;
- короткі уламки з однієї літери відкидаються.

Словник для нейромережі і TF-IDF будуються лише на навчальній частині розбиття. Валідаційні слова, яких не було в навчанні, стають невідомими токенами. Інакше оцінка QWK завищена словами, яких модель у чесному прогнозі ще не бачила.

Розбиття 80/20 стратифіковане за ціллю і зафіксоване зерном 42. Те саме розбиття використовують усі три експерименти.


In [ ]:
NEGATION_TOKENS = {
    "no", "not", "nor", "never", "none", "nobody", "nothing",
    "neither", "nowhere", "without", "cannot",
}
CONTRACTIONS = {
    "ain't": "is not",
    "aren't": "are not",
    "can't": "cannot",
    "couldn't": "could not",
    "didn't": "did not",
    "doesn't": "does not",
    "don't": "do not",
    "hadn't": "had not",
    "hasn't": "has not",
    "haven't": "have not",
    "he's": "he is",
    "here's": "here is",
    "how's": "how is",
    "i'm": "i am",
    "i've": "i have",
    "isn't": "is not",
    "it's": "it is",
    "let's": "let us",
    "mustn't": "must not",
    "she's": "she is",
    "that's": "that is",
    "there's": "there is",
    "they're": "they are",
    "they've": "they have",
    "we're": "we are",
    "we've": "we have",
    "weren't": "were not",
    "what's": "what is",
    "where's": "where is",
    "who's": "who is",
    "won't": "will not",
    "wouldn't": "would not",
    "you're": "you are",
    "you've": "you have",
}
CONTRACTION_RE = re.compile(
    r"\b(" + "|".join(sorted(map(re.escape, CONTRACTIONS), key=len, reverse=True)) + r")\b"
)
FALLBACK_STOPWORDS = set(
    "a about above after again against all am an and any are as at be because been "
    "before being below between both but by can did do does doing down during each "
    "few for from further had has have having he her here hers herself him himself "
    "his how i if in into is it its itself just me more most my myself of off on "
    "once only or other our ours ourselves out over own same she should so some "
    "such than that the their theirs them themselves then there these they this "
    "those through to too under until up very was we were what when where which "
    "while who whom why will with you your yours yourself yourselves".split()
)


def load_stopwords():
    try:
        import nltk
        from nltk.corpus import stopwords

        nltk.download("stopwords", quiet=True)
        words = set(stopwords.words("english"))
        print("Стоп-слова: NLTK, english")
    except Exception as exc:
        print(f"Стоп-слова NLTK недоступні ({type(exc).__name__}: {exc}). Використовую вбудований список.")
        words = set(FALLBACK_STOPWORDS)
    return words - NEGATION_TOKENS


STOPWORDS = load_stopwords()


def clean_text(value):
    if not isinstance(value, str):
        return ""
    text = value.lower().replace("’", "'").replace("`", "'")
    text = CONTRACTION_RE.sub(lambda match: CONTRACTIONS[match.group(0)], text)
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    tokens = [token for token in text.split() if len(token) > 1 and token not in STOPWORDS]
    return " ".join(tokens)


class TextVocabulary:
    def __init__(self, min_freq=MIN_WORD_FREQ, max_len=MAX_TEXT_LEN):
        self.min_freq = min_freq
        self.max_len = max_len
        self.word2idx = {"<pad>": 0, "<unk>": 1}

    def fit(self, texts):
        counter = Counter()
        for text in texts:
            counter.update(str(text).split())
        for word, count in counter.items():
            if count >= self.min_freq and word not in self.word2idx:
                self.word2idx[word] = len(self.word2idx)
        return self

    def encode(self, text):
        indexes = [self.word2idx.get(word, 1) for word in str(text).split()[: self.max_len]]
        if not indexes:
            indexes = [1]
        return torch.tensor(indexes, dtype=torch.long)

    def __len__(self):
        return len(self.word2idx)


class TabularPreprocessor:
    # Категорії з помірною кількістю значень ідуть в ембединги, величини — у стандартизацію.
    # Ідентифікатор рятівника і кличка мають тисячі значень і не узагальнюються на нових
    # тварин, тому в модель не потрапляють. Вік, внесок і кількість фото лишаються числами.

    def __init__(self, id_column, target_column, text_column):
        self.id_column = id_column
        self.target_column = target_column
        self.text_column = text_column
        self.cat_cols = []
        self.num_cols = []
        self.cat_maps = {}
        self.num_median = {}
        self.num_mean = {}
        self.num_std = {}

    def _drop_column(self, column, nunique):
        key = column.lower().replace("_", "")
        if key in {"name", "rescuerid", "petname", "url", "peturl"}:
            return True
        if key.endswith("id") and key not in {"breed1", "breed2"} and nunique > 50:
            return True
        if nunique > TABULAR_MAX_CARDINALITY and not self._forced_numeric(column):
            return True
        return False

    def _forced_numeric(self, column):
        key = column.lower().replace("_", "")
        return any(hint in key for hint in NUMERIC_NAME_HINTS)

    def fit(self, frame):
        self.cat_cols = []
        self.num_cols = []
        self.cat_maps = {}
        self.num_median = {}
        self.num_mean = {}
        self.num_std = {}
        skip = {self.id_column, self.target_column, self.text_column, "text_clean"}
        for column in frame.columns:
            if column in skip:
                continue
            series = frame[column]
            nunique = int(series.nunique(dropna=True))
            if nunique <= 1 or self._drop_column(column, nunique):
                continue
            if pd.api.types.is_float_dtype(series) or self._forced_numeric(column):
                self.num_cols.append(column)
            else:
                self.cat_cols.append(column)
        for column in self.cat_cols:
            values = [value for value in frame[column].dropna().unique().tolist()]
            self.cat_maps[column] = {value: index + 1 for index, value in enumerate(values)}
        for column in self.num_cols:
            numeric = pd.to_numeric(frame[column], errors="coerce")
            median = float(numeric.median()) if numeric.notna().any() else 0.0
            filled = numeric.fillna(median)
            std = float(filled.std(ddof=0))
            self.num_median[column] = median
            self.num_mean[column] = float(filled.mean())
            self.num_std[column] = std if std >= 1e-6 else 1.0
        return self

    def transform(self, frame):
        categorical = np.zeros((len(frame), len(self.cat_cols)), dtype=np.int64)
        for column_index, column in enumerate(self.cat_cols):
            mapping = self.cat_maps[column]
            categorical[:, column_index] = [
                mapping.get(value, 0) if pd.notna(value) else 0
                for value in frame[column].tolist()
            ]
        numeric = np.zeros((len(frame), len(self.num_cols)), dtype=np.float32)
        for column_index, column in enumerate(self.num_cols):
            values = pd.to_numeric(frame[column], errors="coerce").fillna(self.num_median[column])
            numeric[:, column_index] = (
                (values - self.num_mean[column]) / self.num_std[column]
            ).to_numpy(dtype=np.float32)
        return categorical, numeric

    @property
    def cat_cardinalities(self):
        return [len(self.cat_maps[column]) + 1 for column in self.cat_cols]


source_text = train_df[text_col].fillna("").astype(str) if text_col in train_df.columns else pd.Series([""] * len(train_df))
test_source = test_df[text_col].fillna("").astype(str) if text_col in test_df.columns else pd.Series([""] * len(test_df))
train_df["text_clean"] = source_text.map(clean_text)
test_df["text_clean"] = test_source.map(clean_text)
print("Порожніх описів після очищення:", float(train_df["text_clean"].eq("").mean()))
shown = 0
for row_index, raw in source_text.items():
    if not str(raw).strip():
        continue
    print("---")
    print("Було:", str(raw)[:400])
    print("Стало:", train_df.at[row_index, "text_clean"][:400])
    shown += 1
    if shown == 3:
        break

idx_all = np.arange(len(train_df))
try:
    idx_tr, idx_va = train_test_split(
        idx_all,
        test_size=VAL_SIZE,
        random_state=SEED,
        stratify=y_all,
    )
except ValueError as exc:
    print(f"Стратифікація неможлива ({exc}). Ділю без неї, із тим самим зерном.")
    idx_tr, idx_va = train_test_split(
        idx_all,
        test_size=VAL_SIZE,
        random_state=SEED,
        stratify=None,
    )
y_tr = y_all[idx_tr]
y_va = y_all[idx_va]
print("Навчання", len(idx_tr), "валідація", len(idx_va))
print("Класи у валідації:", dict(zip(*np.unique(y_va, return_counts=True))))


## 7. Експеримент 1. Текст: TF-IDF і логістична регресія

Опис часто прямо називає те, що впливає на швидкість усиновлення: вік, стерилізацію, характер, терміновість. Мішок уніграм і біграм це ловить без навчання нейромережі і дає нижню межу, з якою далі порівнюються фото і злиття.

`class_weight="balanced"` збільшує внесок рідкісних класів у функцію втрат. Без цього регресія майже завжди обирає найчастішу швидкість, а для QWK такий константний прогноз близький до нуля.

Поруч записується ще простіший орієнтир: завжди передбачати найчастіший клас навчальної частини. Якщо TF-IDF не обганяє його, текст у такому поданні сигналу не несе.


In [ ]:
majority_class = int(np.bincount(y_tr).argmax())
majority_pred = np.full_like(y_va, majority_class)
majority_qwk = quadratic_weighted_kappa(y_va, majority_pred)
print(
    f"Найчастіший клас індексу {majority_class} "
    f"(мітка {IDX_TO_LABEL[majority_class]}), val QWK {majority_qwk:.4f}, "
    f"accuracy {accuracy_score(y_va, majority_pred):.4f}"
)
log_experiment(
    "Exp0. Найчастіший клас",
    majority_qwk,
    "константа",
    0,
    0,
    "орієнтир без ознак",
)
RUNS["exp0"] = {"decoder": "константа", "val_qwk": majority_qwk, "best_epoch": 0}

started = time.perf_counter()
tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=TFIDF_MIN_DF,
    max_features=TFIDF_MAX_FEATURES,
)
x_text_tr = tfidf.fit_transform(train_df.iloc[idx_tr]["text_clean"])
x_text_va = tfidf.transform(train_df.iloc[idx_va]["text_clean"])
text_clf = LogisticRegression(
    max_iter=500,
    class_weight="balanced",
    random_state=SEED,
)
text_clf.fit(x_text_tr, y_tr)
text_proba_raw = text_clf.predict_proba(x_text_va)
text_proba = np.zeros((len(y_va), N_CLASSES), dtype=np.float64)
for column, class_index in enumerate(text_clf.classes_):
    text_proba[:, int(class_index)] = text_proba_raw[:, column]
text_decoded = decode_probabilities(text_proba, N_CLASSES)
text_scores = {
    name: quadratic_weighted_kappa(y_va, prediction)
    for name, prediction in text_decoded.items()
}
text_decoder = "expectation" if text_scores["expectation"] > text_scores["argmax"] else "argmax"
text_qwk = text_scores[text_decoder]
text_pred = text_decoded[text_decoder]
elapsed = time.perf_counter() - started
print(
    f"TF-IDF ознак: {x_text_tr.shape[1]}. "
    f"QWK argmax {text_scores['argmax']:.4f}, expectation {text_scores['expectation']:.4f}. "
    f"Обрано {text_decoder}. Accuracy {accuracy_score(y_va, text_pred):.4f}. Час {elapsed:.1f} с."
)
plot_confusion(y_va, text_pred, "Exp1. Матриця плутанини на валідації")
log_experiment(
    "Exp1. TF-IDF + логістична регресія",
    text_qwk,
    text_decoder,
    0,
    0,
    f"argmax {text_scores['argmax']:.4f}; expectation {text_scores['expectation']:.4f}; {elapsed:.0f} с",
)
RUNS["exp1"] = {"decoder": text_decoder, "val_qwk": text_qwk, "best_epoch": 0}


### Висновки експерименту 1

Найчастіший клас (мітка 4) має val QWK 0.000 і accuracy 0.332. TF-IDF на уніграмах і біграмах (20 000 ознак) піднімає QWK до 0.273 за argmax і до 0.228 за округленим очікуванням; accuracy 0.424. Опис несе сигнал: текст явно кращий за константу. Для цієї метрики виграв argmax, не очікування. До порогу 0.5 мішок слів не дотягує, тому сам по собі він лишається нижньою межею, а не фінальною моделлю. Навчання зайняло 5 секунд.


## 8. Експеримент 2. Зображення: заморожений ResNet18

На кількох тисячах фото мережа, навчена з нуля, легко запам'ятовує фон притулку і поставу конкретного кадру. ResNet18, уже навчений розрізняти об'єкти на ImageNet, дає вектор сцени: тварина, поза, крупність плану. Ці ваги заморожені. Навчається лише лінійна голова від 512 ознак і прапорця «фото не було».

До трьох кадрів усереднюються. Нульовий вектор без прапорця не відрізнити від дивного, але справжнього фото, тому відсутність знімка кодується окремою одиницею.

Голова теж зупиняється за val QWK. Ваги класів зворотні до частоти: інакше лінійний шар зсувається до найчастішої швидкості, і внесок фото неможливо побачити.


In [ ]:
class PetImageDataset(Dataset):
    def __init__(self, records, transform):
        self.records = records
        self.transform = transform

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        pet_id, path = self.records[index]
        try:
            with Image.open(path) as image:
                tensor = self.transform(image.convert("RGB"))
            ok = 1
        except Exception:
            tensor = torch.zeros(3, 224, 224)
            ok = 0
        return tensor, ok, pet_id


def extract_resnet_features(paths_by_pet, backbone, transform):
    records = [
        (pet_id, path)
        for pet_id, paths in paths_by_pet.items()
        for path in paths
    ]
    sums = {pet_id: np.zeros(512, dtype=np.float32) for pet_id in paths_by_pet}
    counts = {pet_id: 0 for pet_id in paths_by_pet}
    if not records:
        return {pet_id: (sums[pet_id], 0) for pet_id in paths_by_pet}

    loader = DataLoader(
        PetImageDataset(records, transform),
        batch_size=FEATURE_BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
    )
    backbone.eval()
    for tensors, oks, pet_ids_batch in tqdm(loader, desc="ResNet18"):
        features = backbone(tensors.to(device)).detach().cpu().numpy()
        oks = oks.numpy()
        for index, pet_id in enumerate(pet_ids_batch):
            if int(oks[index]) == 1:
                sums[pet_id] += features[index]
                counts[pet_id] += 1
    extracted = {}
    for pet_id in paths_by_pet:
        if counts[pet_id] > 0:
            extracted[pet_id] = (sums[pet_id] / counts[pet_id], counts[pet_id])
        else:
            extracted[pet_id] = (np.zeros(512, dtype=np.float32), 0)
    return extracted


def stack_image_features(pet_ids, feature_map):
    vectors = np.zeros((len(pet_ids), 512), dtype=np.float32)
    missing = np.ones((len(pet_ids), 1), dtype=np.float32)
    for index, pet_id in enumerate(pet_ids):
        vector, count = feature_map[pet_id]
        if count > 0:
            vectors[index] = vector
            missing[index, 0] = 0.0
    combined = np.concatenate([vectors, missing], axis=1)
    return vectors, missing, combined


def load_or_extract_features():
    needed = set(train_ids) | set(test_ids)
    if FEATURE_CACHE.exists():
        cached = np.load(FEATURE_CACHE, allow_pickle=False)
        cached_ids = [normalize_id(pet_id) for pet_id in cached["ids"].tolist()]
        if set(cached_ids) >= needed:
            feature_map = {}
            counts = np.array(cached["counts"], copy=True)
            matrix = np.array(cached["features"], dtype=np.float32, copy=True)
            for index, pet_id in enumerate(cached_ids):
                feature_map[pet_id] = (matrix[index].copy(), int(counts[index]))
            cached.close()
            print(f"Ознаки ResNet завантажено з кешу: {FEATURE_CACHE}")
            return feature_map
        cached.close()
        print("Кеш ознак не покриває поточні ідентифікатори, рахую заново.")

    try:
        weights = ResNet18_Weights.IMAGENET1K_V1
        backbone = resnet18(weights=weights)
        transform = weights.transforms()
    except Exception as exc:
        raise RuntimeError(
            "Не вдалося завантажити ваги ResNet18. Увімкніть Internet у налаштуваннях ноутбука."
        ) from exc
    backbone.fc = nn.Identity()
    backbone.to(device)
    backbone.eval()
    for parameter in backbone.parameters():
        parameter.requires_grad = False

    feature_map = extract_resnet_features(model_image_paths, backbone, transform)
    ids_array = np.array(list(feature_map.keys()))
    np.savez_compressed(
        FEATURE_CACHE,
        ids=ids_array,
        features=np.stack([feature_map[pet_id][0] for pet_id in ids_array]),
        counts=np.array([feature_map[pet_id][1] for pet_id in ids_array], dtype=np.int32),
    )
    print(f"Ознаки збережено: {FEATURE_CACHE}")
    del backbone
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return feature_map


class FeatureDataset(Dataset):
    def __init__(self, features, labels=None):
        self.features = torch.tensor(np.asarray(features), dtype=torch.float32)
        self.labels = None if labels is None else torch.tensor(np.asarray(labels), dtype=torch.long)

    def __len__(self):
        return self.features.size(0)

    def __getitem__(self, index):
        if self.labels is None:
            return self.features[index]
        return self.features[index], self.labels[index]


class ImageHead(nn.Module):
    def __init__(self, in_dim, n_classes, dropout=DROPOUT):
        super().__init__()
        self.net = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(in_dim, n_classes),
        )

    def forward(self, features):
        return self.net(features)


def image_logits(model, batch, compute_loss):
    features, targets = batch
    features = features.to(device)
    targets = targets.to(device)
    logits = model(features)
    loss = image_criterion(logits, targets) if compute_loss else None
    return logits, targets, loss


@torch.no_grad()
def predict_feature_model(model, features):
    model.eval()
    loader = DataLoader(FeatureDataset(features), batch_size=256, shuffle=False)
    chunks = []
    for batch_features in loader:
        logits = model(batch_features.to(device))
        chunks.append(torch.softmax(logits, dim=1).cpu().numpy())
    return np.concatenate(chunks)


feature_map = load_or_extract_features()
_, _, image_combined = stack_image_features(train_ids, feature_map)
_, _, image_combined_test = stack_image_features(test_ids, feature_map)
print("Матриця ознак зображень train:", image_combined.shape, "частка без фото", round(float(image_combined[:, -1].mean()), 4))

set_seed(SEED)
image_criterion = nn.CrossEntropyLoss(weight=class_weights_from_labels(y_tr, N_CLASSES))
image_model = ImageHead(image_combined.shape[1], N_CLASSES).to(device)
image_optimizer = torch.optim.AdamW(image_model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
print("Параметрів голови зображень:", count_parameters(image_model))
train_loader_img = DataLoader(
    FeatureDataset(image_combined[idx_tr], y_tr),
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=torch.Generator().manual_seed(SEED),
)
val_loader_img = DataLoader(
    FeatureDataset(image_combined[idx_va], y_va),
    batch_size=BATCH_SIZE,
    shuffle=False,
)
started = time.perf_counter()
image_fit = train_until_plateau(
    image_model, train_loader_img, val_loader_img, image_optimizer, image_logits, N_CLASSES
)
_, image_probs, image_true = collect_probabilities(image_model, val_loader_img, image_logits)
image_decoded = decode_probabilities(image_probs, N_CLASSES)
image_scores = {
    name: quadratic_weighted_kappa(image_true, prediction)
    for name, prediction in image_decoded.items()
}
image_decoder = image_fit["best_decoder"]
image_qwk = image_scores[image_decoder]
image_pred = image_decoded[image_decoder]
elapsed = time.perf_counter() - started
print(
    f"Exp2 обраний декодер {image_decoder}, QWK {image_qwk:.4f}, "
    f"accuracy {accuracy_score(image_true, image_pred):.4f}, "
    f"епох {image_fit['epochs_ran']}, найкраща {image_fit['best_epoch']}, час {elapsed:.1f} с."
)
plot_confusion(image_true, image_pred, "Exp2. Матриця плутанини на валідації")
log_experiment(
    "Exp2. ResNet18 заморожений + лінійна голова",
    image_qwk,
    image_decoder,
    image_fit["epochs_ran"],
    image_fit["best_epoch"],
    f"argmax {image_scores['argmax']:.4f}; expectation {image_scores['expectation']:.4f}; {elapsed:.0f} с",
)
RUNS["exp2"] = {
    "decoder": image_decoder,
    "val_qwk": image_qwk,
    "best_epoch": image_fit["best_epoch"],
}


### Висновки експерименту 2

Заморожений ResNet18 і лінійна голова (2056 параметрів) дають val QWK 0.425 за argmax і accuracy 0.442. Очікування класу тут слабше: 0.321. По епохах argmax ішов 0.384, 0.370, 0.425, 0.421, 0.408. Рання зупинка спрацювала на 5-й епосі, найкраща — 3-тя, коли val loss ще був 1.260; далі втрати на валідації виросли.

Це на 0.152 QWK вище за текст. Вигляд тварини пояснює швидкість адопції краще, ніж мішок слів з опису. На train фото є в усіх профілів, тож нульовий вектор і прапорець відсутності знімка на навчанні не спрацьовували. Голова коротка, і плато настало швидко: заморожені ознаки ImageNet уже вичерпали те, що лінійний шар уміє з них витягти.


## 9. Експеримент 3. Злиття ResNet18 і BiLSTM

Вектор фото вже пораховано і в цей експеримент входить як готовий масив. Текст кодується двонаправленим LSTM: прямий прохід читає опис від початку історії до висновку, зворотний — навпаки, і для рішення важливі обидва кінці фрази. Береться один шар і розмірність 128, бо текстів кілька тисяч: глибша рекурентна мережа запам'ятає навчальні формулювання.

Один шар LSTM не має проміжного dropout між шарами, тому dropout стоїть на ембедингу слова і перед фінальним лінійним шаром.

Якщо в таблиці є придатні колонки, вони входять третьою гілкою. Категорії з числом значень до 400 отримують власний ембединг: порода чи колір — це код, а не величина. Вік, внесок, кількість фото стандартизуються. Кличка і ідентифікатор рятівника відкидаються: на новій тварині такого значення ще не було, і ембединг його не узагальнить.

Голова — конкатенація гілок, dropout і лінійний шар на число класів. Функція втрат — крос-ентропія з вагами, зворотними до частоти класу. Рання зупинка та сама, що для голови зображень. Графік втрат і QWK будується для цього експерименту, бо тут уперше одночасно навчаються текстові ваги і голова злиття.


In [ ]:
class FusionDataset(Dataset):
    def __init__(self, texts, image_vectors, missing_flags, labels, vocabulary, categorical=None, numeric=None):
        self.texts = list(texts)
        self.image_vectors = np.asarray(image_vectors, dtype=np.float32)
        self.missing_flags = np.asarray(missing_flags, dtype=np.float32)
        self.labels = np.asarray(labels, dtype=np.int64)
        self.vocabulary = vocabulary
        self.categorical = None if categorical is None else np.asarray(categorical)
        self.numeric = None if numeric is None else np.asarray(numeric, dtype=np.float32)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):
        item = {
            "text": self.vocabulary.encode(self.texts[index]),
            "image": torch.tensor(self.image_vectors[index]),
            "missing": torch.tensor(self.missing_flags[index]),
            "label": torch.tensor(self.labels[index]),
        }
        if self.categorical is not None:
            item["cat"] = torch.tensor(self.categorical[index], dtype=torch.long)
            item["num"] = torch.tensor(self.numeric[index], dtype=torch.float32)
        return item


def collate_fusion(batch):
    collated = {
        "text": pad_sequence([item["text"] for item in batch], batch_first=True, padding_value=0),
        "image": torch.stack([item["image"] for item in batch]),
        "missing": torch.stack([item["missing"] for item in batch]),
        "label": torch.stack([item["label"] for item in batch]),
    }
    if "cat" in batch[0]:
        collated["cat"] = torch.stack([item["cat"] for item in batch])
        collated["num"] = torch.stack([item["num"] for item in batch])
    return collated


class FusionNet(nn.Module):
    def __init__(self, vocab_size, n_classes, cat_cardinalities, n_numeric):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, EMBED_DIM, padding_idx=0)
        self.emb_dropout = nn.Dropout(DROPOUT)
        self.lstm = nn.LSTM(
            EMBED_DIM,
            LSTM_HIDDEN,
            num_layers=1,
            batch_first=True,
            bidirectional=True,
        )
        self.cat_embs = nn.ModuleList()
        cat_dim = 0
        for cardinality in cat_cardinalities:
            emb_dim = min(16, max(2, int(round(cardinality ** 0.5))))
            self.cat_embs.append(nn.Embedding(cardinality, emb_dim))
            cat_dim += emb_dim
        self.n_numeric = n_numeric
        self.fusion_dim = 512 + 1 + LSTM_HIDDEN * 2 + cat_dim + n_numeric
        self.head = nn.Sequential(
            nn.Dropout(DROPOUT),
            nn.Linear(self.fusion_dim, n_classes),
        )

    def encode_text(self, text):
        lengths = (text != 0).sum(dim=1).clamp(min=1)
        embedded = self.emb_dropout(self.embedding(text))
        packed = pack_padded_sequence(
            embedded, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        _, (hidden, _) = self.lstm(packed)
        return torch.cat([hidden[0], hidden[1]], dim=1)

    def forward(self, text, image, missing, cat=None, num=None):
        parts = [image, missing, self.encode_text(text)]
        if len(self.cat_embs) and cat is not None:
            embedded_cats = [embedding(cat[:, index]) for index, embedding in enumerate(self.cat_embs)]
            parts.append(torch.cat(embedded_cats, dim=1))
        if self.n_numeric and num is not None:
            parts.append(num)
        return self.head(torch.cat(parts, dim=1))


def fusion_logits(model, batch, compute_loss):
    batch = batch_move(batch)
    logits = model(
        batch["text"],
        batch["image"],
        batch["missing"],
        batch.get("cat"),
        batch.get("num"),
    )
    targets = batch["label"]
    loss = fusion_criterion(logits, targets) if compute_loss else None
    return logits, targets, loss


def make_fusion_loader(texts, image_vectors, missing_flags, labels, vocabulary, categorical, numeric, shuffle):
    dataset = FusionDataset(
        texts, image_vectors, missing_flags, labels, vocabulary, categorical, numeric
    )
    generator = torch.Generator().manual_seed(SEED) if shuffle else None
    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        generator=generator,
        collate_fn=collate_fusion,
    )


@torch.no_grad()
def predict_fusion(model, loader):
    model.eval()
    chunks = []
    for batch in loader:
        logits, _, _ = fusion_logits(model, batch, compute_loss=False)
        chunks.append(torch.softmax(logits, dim=1).cpu().numpy())
    return np.concatenate(chunks)


tabular = TabularPreprocessor(id_col, target_col, text_col).fit(train_df.iloc[idx_tr])
print("Категоріальні колонки:", tabular.cat_cols)
print("Числові колонки:", tabular.num_cols)
cat_all, num_all = tabular.transform(train_df)
cat_test, num_test = tabular.transform(test_df)
use_tabular = bool(tabular.cat_cols or tabular.num_cols)
cat_tr = cat_all[idx_tr] if use_tabular else None
cat_va = cat_all[idx_va] if use_tabular else None
num_tr = num_all[idx_tr] if use_tabular else None
num_va = num_all[idx_va] if use_tabular else None

image_vectors_all = image_combined[:, :512]
missing_all = image_combined[:, 512:]
image_vectors_test = image_combined_test[:, :512]
missing_test = image_combined_test[:, 512:]

vocabulary = TextVocabulary().fit(train_df.iloc[idx_tr]["text_clean"])
print("Розмір словника:", len(vocabulary))

set_seed(SEED)
fusion_criterion = nn.CrossEntropyLoss(weight=class_weights_from_labels(y_tr, N_CLASSES))
fusion_model = FusionNet(
    len(vocabulary),
    N_CLASSES,
    tabular.cat_cardinalities if use_tabular else [],
    len(tabular.num_cols) if use_tabular else 0,
).to(device)
fusion_optimizer = torch.optim.AdamW(
    fusion_model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
)
print("Вхід голови злиття:", fusion_model.fusion_dim)
print("Параметрів, що навчаються:", count_parameters(fusion_model))

train_loader_fusion = make_fusion_loader(
    train_df.iloc[idx_tr]["text_clean"],
    image_vectors_all[idx_tr],
    missing_all[idx_tr],
    y_tr,
    vocabulary,
    cat_tr,
    num_tr,
    shuffle=True,
)
val_loader_fusion = make_fusion_loader(
    train_df.iloc[idx_va]["text_clean"],
    image_vectors_all[idx_va],
    missing_all[idx_va],
    y_va,
    vocabulary,
    cat_va,
    num_va,
    shuffle=False,
)
started = time.perf_counter()
fusion_fit = train_until_plateau(
    fusion_model,
    train_loader_fusion,
    val_loader_fusion,
    fusion_optimizer,
    fusion_logits,
    N_CLASSES,
)
_, fusion_probs, fusion_true = collect_probabilities(fusion_model, val_loader_fusion, fusion_logits)
fusion_decoded = decode_probabilities(fusion_probs, N_CLASSES)
fusion_scores = {
    name: quadratic_weighted_kappa(fusion_true, prediction)
    for name, prediction in fusion_decoded.items()
}
fusion_decoder = fusion_fit["best_decoder"]
fusion_qwk = fusion_scores[fusion_decoder]
fusion_pred = fusion_decoded[fusion_decoder]
elapsed = time.perf_counter() - started
print(
    f"Exp3 обраний декодер {fusion_decoder}, QWK {fusion_qwk:.4f}, "
    f"accuracy {accuracy_score(fusion_true, fusion_pred):.4f}, "
    f"епох {fusion_fit['epochs_ran']}, найкраща {fusion_fit['best_epoch']}, час {elapsed:.1f} с."
)
plot_training_history(fusion_fit["history"])
plot_confusion(fusion_true, fusion_pred, "Exp3. Матриця плутанини на валідації")
exp3_name = "Exp3. ResNet18 + BiLSTM"
if use_tabular:
    exp3_name += " + табличні ознаки"
log_experiment(
    exp3_name,
    fusion_qwk,
    fusion_decoder,
    fusion_fit["epochs_ran"],
    fusion_fit["best_epoch"],
    f"argmax {fusion_scores['argmax']:.4f}; expectation {fusion_scores['expectation']:.4f}; {elapsed:.0f} с",
)
RUNS["exp3"] = {
    "name": exp3_name,
    "decoder": fusion_decoder,
    "val_qwk": fusion_qwk,
    "best_epoch": fusion_fit["best_epoch"],
    "use_tabular": use_tabular,
}


### Висновки експерименту 3

Табличних колонок для третьої гілки немає. Злиття — вектор ResNet і BiLSTM на словнику з 6740 слів, 1 129 992 параметри, що навчаються. Найкращий val QWK — 0.423 на 2-й епосі (argmax), accuracy 0.454. Очікування дало 0.362. На 3-й і 4-й епохах train loss падав до 1.048, а val loss зріс до 1.369, QWK відкотився до 0.388 і 0.398. Рання зупинка на 4-й епосі це обірвала.

Злиття не краще за самі фото: 0.423 проти 0.425. Додатковий мільйон текстових параметрів запам'ятовує формулювання описів і не покращує порядок класів. Для цього набору окрема візуальна голова лишається сильнішою моделлю.


## 10. Порівняння і файл відповіді

Обирається найраніший експеримент, який пізніші не обганяють більше ніж на 0.001 val QWK. Такий приріст уже вважався шумом під час ранньої зупинки, і ускладнювати сабміт без нього немає підстави.

Обрана модель навчається заново на **усьому** розміченому train. Тестові мітки, навіть якщо вони раптом є у файлі, у це навчання не входять. Для нейромереж число епох дорівнює найкращій епосі валідаційного прогону. Декодер (argmax або очікування) теж фіксується за валідацією і на тесті не перебирається.

TF-IDF, словник і табличні перетворення для фінального навчання будуються вже на повному train, бо приховувати від фінальної моделі частину розмічених описів немає причини. Ознаки ResNet від міток не залежать і беруться з кешу.


In [ ]:
comparison = pd.DataFrame(EXPERIMENT_ROWS)
print("Журнал експериментів, відсортований за val QWK:")
display(comparison.sort_values(["Val QWK", "Експеримент"], ascending=[False, True]).reset_index(drop=True))

best_index = 0
best_score = -1.0
for row_index, row in enumerate(EXPERIMENT_ROWS):
    score = row["Val QWK"]
    if score != score:
        score = -1.0
    if row_index == 0 or score > best_score + MIN_DELTA:
        best_index = row_index
        best_score = score
best_row = EXPERIMENT_ROWS[best_index]
best_name = best_row["Експеримент"]
print(
    f"Для сабміту обрано: {best_name}. "
    f"Val QWK {best_row['Val QWK']}. Декодер: {best_row['Декодер']}. "
    f"Найкраща епоха: {best_row['Найкраща епоха']}."
)


def probabilities_to_labels(probs, decoder):
    decoded = decode_probabilities(probs, N_CLASSES)
    if decoder not in decoded:
        decoder = "argmax"
    indexes = decoded[decoder]
    return [IDX_TO_LABEL[int(index)] for index in indexes]


def align_proba(proba, class_indexes):
    full = np.zeros((proba.shape[0], N_CLASSES), dtype=np.float64)
    for column, class_index in enumerate(class_indexes):
        full[:, int(class_index)] = proba[:, column]
    return full


if best_name.startswith("Exp0"):
    mode_label = IDX_TO_LABEL[int(np.bincount(y_all).argmax())]
    pred_labels = [mode_label] * len(test_df)
    print("Фінальний прогноз — найчастіший клас повного train:", mode_label)

elif best_name.startswith("Exp1"):
    final_tfidf = TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=TFIDF_MIN_DF,
        max_features=TFIDF_MAX_FEATURES,
    )
    final_clf = LogisticRegression(max_iter=500, class_weight="balanced", random_state=SEED)
    final_clf.fit(final_tfidf.fit_transform(train_df["text_clean"]), y_all)
    test_proba = align_proba(
        final_clf.predict_proba(final_tfidf.transform(test_df["text_clean"])),
        final_clf.classes_,
    )
    pred_labels = probabilities_to_labels(test_proba, RUNS["exp1"]["decoder"])
    print("Фінальну логістичну регресію навчено на всіх розмічених описах.")

elif best_name.startswith("Exp2"):
    set_seed(SEED)
    final_epochs = max(int(RUNS["exp2"]["best_epoch"]), 1)
    final_image_criterion = nn.CrossEntropyLoss(weight=class_weights_from_labels(y_all, N_CLASSES))

    def final_image_logits(model, batch, compute_loss):
        features, targets = batch
        features = features.to(device)
        targets = targets.to(device)
        logits = model(features)
        loss = final_image_criterion(logits, targets) if compute_loss else None
        return logits, targets, loss

    final_image_model = ImageHead(image_combined.shape[1], N_CLASSES).to(device)
    final_image_optimizer = torch.optim.AdamW(
        final_image_model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )
    final_loader = DataLoader(
        FeatureDataset(image_combined, y_all),
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=torch.Generator().manual_seed(SEED),
    )
    print(f"Повне навчання голови зображень: {final_epochs} епох.")
    train_fixed_epochs(final_image_model, final_loader, final_image_optimizer, final_image_logits, final_epochs)
    test_proba = predict_feature_model(final_image_model, image_combined_test)
    pred_labels = probabilities_to_labels(test_proba, RUNS["exp2"]["decoder"])

elif best_name.startswith("Exp3"):
    set_seed(SEED)
    final_epochs = max(int(RUNS["exp3"]["best_epoch"]), 1)
    final_tabular = TabularPreprocessor(id_col, target_col, text_col).fit(train_df)
    final_cat, final_num = final_tabular.transform(train_df)
    final_cat_test, final_num_test = final_tabular.transform(test_df)
    final_vocab = TextVocabulary().fit(train_df["text_clean"])
    final_use_tabular = bool(final_tabular.cat_cols or final_tabular.num_cols)
    final_fusion_criterion = nn.CrossEntropyLoss(weight=class_weights_from_labels(y_all, N_CLASSES))

    def final_fusion_logits(model, batch, compute_loss):
        batch = batch_move(batch)
        logits = model(
            batch["text"],
            batch["image"],
            batch["missing"],
            batch.get("cat"),
            batch.get("num"),
        )
        targets = batch["label"]
        loss = final_fusion_criterion(logits, targets) if compute_loss else None
        return logits, targets, loss

    final_fusion = FusionNet(
        len(final_vocab),
        N_CLASSES,
        final_tabular.cat_cardinalities if final_use_tabular else [],
        len(final_tabular.num_cols) if final_use_tabular else 0,
    ).to(device)
    final_optimizer = torch.optim.AdamW(
        final_fusion.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )
    final_loader = make_fusion_loader(
        train_df["text_clean"],
        image_vectors_all,
        missing_all,
        y_all,
        final_vocab,
        final_cat if final_use_tabular else None,
        final_num if final_use_tabular else None,
        shuffle=True,
    )
    print(f"Повне навчання злиття: {final_epochs} епох, словник {len(final_vocab)}.")
    # Локальні logits цієї комірки не підміняють функцію валідаційного експерименту.
    saved_fusion_logits = fusion_logits
    fusion_logits = final_fusion_logits
    try:
        train_fixed_epochs(final_fusion, final_loader, final_optimizer, fusion_logits, final_epochs)
        test_loader = make_fusion_loader(
            test_df["text_clean"],
            image_vectors_test,
            missing_test,
            np.zeros(len(test_df), dtype=int),
            final_vocab,
            final_cat_test if final_use_tabular else None,
            final_num_test if final_use_tabular else None,
            shuffle=False,
        )
        test_proba = predict_fusion(final_fusion, test_loader)
    finally:
        fusion_logits = saved_fusion_logits
    pred_labels = probabilities_to_labels(test_proba, RUNS["exp3"]["decoder"])

else:
    raise RuntimeError(f"Невідомий експеримент у журналі: {best_name}")

sample_id_col = find_column(sample_df.columns, ["PetID", "pet_id", "id"])
sample_target_col = find_column(sample_df.columns, ["AdoptionSpeed", "adoption_speed", "label", "target"])
if sample_id_col is None or sample_target_col is None:
    raise RuntimeError(f"sample_submission має несподівані колонки: {list(sample_df.columns)}")
if len(pred_labels) != len(test_ids):
    raise RuntimeError(f"Прогнозів {len(pred_labels)}, а рядків у test.csv — {len(test_ids)}.")

submission = pd.DataFrame({
    "PetID": [normalize_id(pet_id) for pet_id in test_ids],
    "AdoptionSpeed": pred_labels,
})
duplicate_ids = int(submission["PetID"].duplicated().sum())
if duplicate_ids:
    print(f"Повторні PetID у test: {duplicate_ids}. У файл іде перший прогноз.")
    submission = submission.drop_duplicates("PetID", keep="first")

sample_ids = sample_df[sample_id_col].map(normalize_id)
same_ids = set(sample_ids) == set(submission["PetID"]) and len(sample_ids) == len(submission)
if same_ids:
    submission = submission.set_index("PetID").loc[sample_ids.tolist()].reset_index()
else:
    print(
        f"sample_submission має {len(sample_ids)} рядків, test.csv — {len(test_ids)}. "
        "У відповідь записано кожен PetID з test.csv: короткий зразок не замінює тестовий набір."
    )

allowed_labels = set(IDX_TO_LABEL.values())
unknown_labels = sorted(set(submission["AdoptionSpeed"]) - allowed_labels)
if unknown_labels:
    raise RuntimeError(f"У відповіді є мітки поза шкалою train: {unknown_labels}")
submission["AdoptionSpeed"] = submission["AdoptionSpeed"].astype(int)
submission = submission[["PetID", "AdoptionSpeed"]]
if submission["PetID"].eq("").any() or submission["AdoptionSpeed"].isna().any():
    raise RuntimeError("У відповіді є порожній PetID або пропущена мітка.")

submission.to_csv(SUBMISSION_PATH, index=False)
print("Збережено", SUBMISSION_PATH)
print(f"Рядків: {len(submission)}. Колонки: {list(submission.columns)}.")
print(submission["AdoptionSpeed"].value_counts().sort_index())
display(submission.head(3))
display(submission.tail(3))


## Загальні висновки

У сабміт пішла голова ResNet18, декодер argmax, 3 епохи на повному train. Файл містить 1891 рядок, колонки PetID і AdoptionSpeed.

Відповіді на питання прогону:

- Текст кращий за найчастіший клас: так, 0.273 проти 0.000.
- Фото додають щось понад текст: так, 0.425 проти 0.273.
- Злиття краще за кожну модальність окремо: ні, 0.423 проти 0.425 у самих фото.
- Округлене очікування краще за argmax: ні. У переможця 0.321 проти 0.425.

Public score цієї моделі — **0.5839**. Внутрішній QWK 0.425 був песимістичнішим за публічний тест, але кращий результат учасників — **0.8088**. Різниця 0.225. Замороженого середнього з трьох кадрів і лінійної голови для цього недостатньо.

Окремий дефект видно вже в складі прогнозів на тесті. У train частки класів 1–4 такі: 18.6%, 27.6%, 20.6%, 33.2%. У файлі відповіді: 37.8% (715), 8.8% (167), 20.9% (395), 32.5% (614). Модель занадто часто ставить найшвидшу адопцію і майже пропускає клас 2. Ваги втрат найбільші саме в класу 1, бо він найрідший. Для QWK помилка «1 замість 4» дорога, тож такий зсув треба прибрати раніше, ніж ускладнювати мережу.


## Наступні експерименти

Попередні блоки не замінюються. Наступний прогін додається комірками нижче. Порядок такий, бо його диктують числа цього прогону, а не бажання перебрати архітектури.

1. Та сама кешована матриця ResNet, та сама лінійна голова, але без ваг класів або з м'якшими вагами. Мета — повернути класу 2 частку ближчу до train і зменшити частку класу 1 у тесті. Це дешево і перевіряє, чи 0.425 уперся в ознаки, чи в зсув втрати.
2. Більше кадрів на тварину. Медіана — 4 фото, стеля моделі — 3, тож половина профілів втрачає щонайменше один ракурс.
3. Донавчити останній блок ResNet невеликою швидкістю навчання. Ознаки ImageNet відрізняють тварину від фону, але не те, наскільки профіль виглядає готовим до усиновлення. Це головний запас до 0.7 і вище. Повторювати BiLSTM не варто: 1.13 млн параметрів уже перенавчилися.
4. Якщо після пункту 3 фото все ще далеко від 0.81, додати текст не рекурентною мережею, а стисненим TF-IDF поруч із візуальним вектором. Мішок слів уже дав 0.273 сам по собі; його можна домішати лінійною головою, не повертаючи перенавчання LSTM.
5. Пошук гіперпараметрів має сенс після того, як голова і розморожений блок обрані. Перебирати швидкість навчання на замороженій мережі, яка вже вперлася в 0.425, не закриє розрив із 0.8088.


## 11. Продовження без повторного прогону

Цей блок не навчає заново TF-IDF, ResNet і BiLSTM. Попередні комірки лишаються історією прогону.

Як запускати на Kaggle: **не натискайте Run All**. Відкрийте першу код-комірку цього розділу і виконайте її та всі комірки нижче. Якщо ядро вже скинуте, ця комірка сама перечитає `train.csv` і `test.csv` і відновить те саме розбиття 5144/1287 за зерном 42. Це секунди, не навчання фото.

Гіпотеза цього кроку: вік, вакцинація, стерилізація і тон історії записані лише в описі. Мішок слів дав val QWK 0.273. Заморожений DistilBERT читає фразу цілком і перетворює сирий опис (без вирізання стоп-слів) на один вектор. Навчається тільки лінійна голова. Ваг класів немає: у попередньому сабміті вони роздули частку класу 1 до 37.8% за базових 18.6%.

Поріг, з яким порівнюємо: текст 0.273, фото 0.425. Новий `submission.csv` пишеться лише якщо val QWK голови вищий за 0.425 більше ніж на 0.001. Інакше на публічному тесті лишається модель фото зі score 0.5839.


In [ ]:
# Стан для нового експерименту. Якщо комірки вище вже виконані в цьому ядрі, таблиці не перечитуються.
needs_tables = "train_df" not in globals() or "idx_tr" not in globals() or "y_all" not in globals()
if not needs_tables:
    print(f"Таблиці вже в пам'яті: train {len(train_df)}, val {len(idx_va)}. Повторно не читаю.")
else:
    import random
    from pathlib import Path
    import numpy as np
    import pandas as pd
    import torch
    from sklearn.metrics import cohen_kappa_score
    from sklearn.model_selection import train_test_split

    SEED = 42
    VAL_SIZE = 0.2
    DATA_ROOT = Path("/kaggle/input/competitions/deep-learning-for-computer-vision-and-nlp-2026-10")
    TRAIN_CSV = DATA_ROOT / "train.csv"
    TEST_CSV = DATA_ROOT / "test.csv"
    SAMPLE_CSV = DATA_ROOT / "sample_submission.csv"
    WORKING = Path("/kaggle/working")
    WORKING.mkdir(parents=True, exist_ok=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    def find_column(columns, options):
        lookup = {str(column).lower(): column for column in columns}
        for option in options:
            if option.lower() in lookup:
                return lookup[option.lower()]
        return None

    def normalize_id(value):
        if pd.isna(value):
            return ""
        text = str(value).strip()
        if text.endswith(".0"):
            text = text[:-2]
        return text

    def read_table(path):
        frame = pd.read_csv(path)
        frame.columns = [str(column).strip() for column in frame.columns]
        return frame

    def quadratic_weighted_kappa(y_true, y_pred):
        y_true = np.asarray(y_true).astype(int)
        y_pred = np.asarray(y_pred).astype(int)
        if len(y_true) == 0:
            return 0.0
        score = cohen_kappa_score(y_true, y_pred, weights="quadratic")
        if score != score:
            return 0.0
        return float(score)

    train_df = read_table(TRAIN_CSV)
    test_df = read_table(TEST_CSV)
    sample_df = read_table(SAMPLE_CSV)
    id_col = find_column(train_df.columns, ["PetID", "pet_id", "id"])
    target_col = find_column(train_df.columns, ["AdoptionSpeed", "adoption_speed", "label", "target"])
    text_col = find_column(train_df.columns, ["Description", "description", "desc", "text", "comments"])
    test_id_col = find_column(test_df.columns, ["PetID", "pet_id", "id"]) or id_col
    train_df[id_col] = train_df[id_col].map(normalize_id)
    test_df[test_id_col] = test_df[test_id_col].map(normalize_id)
    if int(train_df[id_col].duplicated().sum()):
        train_df = train_df.drop_duplicates(id_col, keep="first")
    train_df = train_df.reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)
    if target_col in test_df.columns:
        test_df = test_df.drop(columns=[target_col])
    test_text_col = find_column(test_df.columns, ["Description", "description", "desc", "text", "comments"])
    if text_col is None:
        text_col = "Description"
        train_df[text_col] = ""
        test_df[text_col] = ""
    elif test_text_col is None:
        test_df[text_col] = ""
    elif test_text_col != text_col:
        test_df[text_col] = test_df[test_text_col]
    raw_target = train_df[target_col]
    if pd.api.types.is_numeric_dtype(raw_target):
        numeric_target = raw_target.astype(float)
        if np.allclose(numeric_target.to_numpy(), np.rint(numeric_target.to_numpy())):
            train_df[target_col] = numeric_target.round().astype(int)
    classes = sorted(train_df[target_col].dropna().unique().tolist())
    LABEL_TO_IDX = {label: index for index, label in enumerate(classes)}
    IDX_TO_LABEL = {index: label for label, index in LABEL_TO_IDX.items()}
    N_CLASSES = len(classes)
    train_df = train_df.dropna(subset=[target_col]).reset_index(drop=True)
    y_all = train_df[target_col].map(LABEL_TO_IDX).to_numpy(dtype=int)
    idx_tr, idx_va = train_test_split(
        np.arange(len(train_df)), test_size=VAL_SIZE, random_state=SEED, stratify=y_all
    )
    y_tr = y_all[idx_tr]
    y_va = y_all[idx_va]
    test_ids = test_df[test_id_col].tolist()
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    print(f"Таблиці прочитано заново: train {len(train_df)}, val {len(idx_va)}, test {len(test_df)}.")

if len(idx_va) != 1287:
    print(f"Увага: у валідації {len(idx_va)} рядків, у минулому прогоні було 1287. QWK буде не на тому самому розбитті.")
else:
    print("Розбиття збігається з минулим прогоном: 1287 валідаційних рядків.")

train_raw_text = train_df[text_col].fillna("").astype(str).tolist()
test_raw_text = test_df[text_col].fillna("").astype(str).tolist()
print("Сирих описів train/test:", len(train_raw_text), len(test_raw_text))
print("Пристрій:", device)


In [ ]:
import numpy as np
import torch
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer

TEXT_ENCODER_NAME = "distilbert-base-uncased"
MAX_TEXT_MODEL_LEN = 256
TEXT_EMBED_BATCH = 32
TEXT_EMBED_CACHE = WORKING / "distilbert_description_embeddings.npz"


def embed_texts(texts, tokenizer, encoder):
    vectors = None
    truncated = 0
    encoder.eval()
    for start in tqdm(range(0, len(texts), TEXT_EMBED_BATCH), desc="DistilBERT"):
        chunk = [text if isinstance(text, str) else "" for text in texts[start:start + TEXT_EMBED_BATCH]]
        uncut = tokenizer(chunk, add_special_tokens=True, truncation=False, padding=False)
        truncated += sum(len(token_ids) > MAX_TEXT_MODEL_LEN for token_ids in uncut["input_ids"])
        encoded = tokenizer(
            chunk,
            padding=True,
            truncation=True,
            max_length=MAX_TEXT_MODEL_LEN,
            return_tensors="pt",
        )
        encoded = {key: value.to(device) for key, value in encoded.items() if key in {"input_ids", "attention_mask"}}
        hidden = encoder(**encoded).last_hidden_state
        mask = encoded["attention_mask"].unsqueeze(-1).float()
        pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1.0)
        pooled = pooled.detach().cpu().numpy().astype(np.float32)
        if vectors is None:
            vectors = np.zeros((len(texts), pooled.shape[1]), dtype=np.float32)
        vectors[start:start + len(chunk)] = pooled
    return vectors, truncated


cache_ok = False
if TEXT_EMBED_CACHE.exists():
    cached = np.load(TEXT_EMBED_CACHE, allow_pickle=False)
    same_shape = cached["train"].shape[0] == len(train_raw_text) and cached["test"].shape[0] == len(test_raw_text)
    same_model = str(cached["model_name"].item()) == TEXT_ENCODER_NAME if cached["model_name"].shape == () else False
    if same_shape and same_model and int(cached["max_len"]) == MAX_TEXT_MODEL_LEN:
        text_train_emb = np.array(cached["train"], dtype=np.float32, copy=True)
        text_test_emb = np.array(cached["test"], dtype=np.float32, copy=True)
        cache_ok = True
        print(f"Текстові ембединги взято з кешу {TEXT_EMBED_CACHE}, розмір {text_train_emb.shape}.")
    cached.close()

if not cache_ok:
    tokenizer = AutoTokenizer.from_pretrained(TEXT_ENCODER_NAME)
    encoder = AutoModel.from_pretrained(TEXT_ENCODER_NAME).to(device)
    encoder.eval()
    for parameter in encoder.parameters():
        parameter.requires_grad = False
    text_train_emb, train_truncated = embed_texts(train_raw_text, tokenizer, encoder)
    text_test_emb, test_truncated = embed_texts(test_raw_text, tokenizer, encoder)
    print(
        f"Обрізано описів до {MAX_TEXT_MODEL_LEN} токенів: "
        f"train {train_truncated}/{len(train_raw_text)} ({train_truncated / max(len(train_raw_text), 1):.1%}), "
        f"test {test_truncated}/{len(test_raw_text)} ({test_truncated / max(len(test_raw_text), 1):.1%})."
    )
    np.savez_compressed(
        TEXT_EMBED_CACHE,
        train=text_train_emb,
        test=text_test_emb,
        model_name=np.array(TEXT_ENCODER_NAME),
        max_len=np.int32(MAX_TEXT_MODEL_LEN),
    )
    print("Збережено", TEXT_EMBED_CACHE, text_train_emb.shape)
    del encoder, tokenizer
    if device.type == "cuda":
        torch.cuda.empty_cache()


In [ ]:
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score
from torch.utils.data import DataLoader, Dataset

HEAD_DROPOUT = 0.3
HEAD_LR = 1e-3
HEAD_WEIGHT_DECAY = 1e-4
HEAD_MAX_EPOCHS = 8
HEAD_PATIENCE = 2
HEAD_MIN_DELTA = 0.001
IMAGE_VAL_QWK = 0.4247
TFIDF_VAL_QWK = 0.2730


class EmbeddingDataset(Dataset):
    def __init__(self, features, labels=None):
        self.features = torch.tensor(np.asarray(features), dtype=torch.float32)
        self.labels = None if labels is None else torch.tensor(np.asarray(labels), dtype=torch.long)

    def __len__(self):
        return self.features.size(0)

    def __getitem__(self, index):
        if self.labels is None:
            return self.features[index]
        return self.features[index], self.labels[index]


class TextEmbeddingHead(nn.Module):
    def __init__(self, in_dim, n_classes, dropout=HEAD_DROPOUT):
        super().__init__()
        self.net = nn.Sequential(nn.Dropout(dropout), nn.Linear(in_dim, n_classes))

    def forward(self, features):
        return self.net(features)


def decode_probabilities(probs, n_classes):
    class_index = np.arange(n_classes, dtype=np.float64)
    expectation = np.clip(np.rint(probs @ class_index), 0, n_classes - 1).astype(int)
    return {"argmax": probs.argmax(axis=1).astype(int), "expectation": expectation}


def fit_embedding_head(features_train, labels_train, features_val, labels_val):
    torch.manual_seed(SEED)
    model = TextEmbeddingHead(features_train.shape[1], N_CLASSES).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=HEAD_LR, weight_decay=HEAD_WEIGHT_DECAY)
    criterion = nn.CrossEntropyLoss()
    train_loader = DataLoader(
        EmbeddingDataset(features_train, labels_train),
        batch_size=64,
        shuffle=True,
        generator=torch.Generator().manual_seed(SEED),
    )
    val_loader = DataLoader(EmbeddingDataset(features_val, labels_val), batch_size=256, shuffle=False)
    best_qwk = -1.0
    best_epoch = 1
    best_decoder = "argmax"
    best_state = copy.deepcopy(model.state_dict())
    wait = 0
    epochs_ran = 0
    for epoch in range(1, HEAD_MAX_EPOCHS + 1):
        epochs_ran = epoch
        model.train()
        total_loss, seen = 0.0, 0
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            optimizer.zero_grad(set_to_none=True)
            logits = model(batch_x)
            loss = criterion(logits, batch_y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            total_loss += loss.item() * batch_y.size(0)
            seen += batch_y.size(0)
        model.eval()
        prob_chunks, true_chunks = [], []
        val_loss, val_seen = 0.0, 0
        with torch.no_grad():
            for batch_x, batch_y in val_loader:
                batch_x, batch_y = batch_x.to(device), batch_y.to(device)
                logits = model(batch_x)
                val_loss += criterion(logits, batch_y).item() * batch_y.size(0)
                val_seen += batch_y.size(0)
                prob_chunks.append(torch.softmax(logits, dim=1).cpu().numpy())
                true_chunks.append(batch_y.cpu().numpy())
        probs = np.concatenate(prob_chunks)
        y_true = np.concatenate(true_chunks)
        decoded = decode_probabilities(probs, N_CLASSES)
        qwk_argmax = quadratic_weighted_kappa(y_true, decoded["argmax"])
        qwk_expectation = quadratic_weighted_kappa(y_true, decoded["expectation"])
        epoch_decoder = "expectation" if qwk_expectation > qwk_argmax else "argmax"
        epoch_qwk = qwk_expectation if epoch_decoder == "expectation" else qwk_argmax
        print(
            f"Епоха {epoch:02d}/{HEAD_MAX_EPOCHS} | train loss {total_loss / max(seen, 1):.4f} | "
            f"val loss {val_loss / max(val_seen, 1):.4f} | "
            f"QWK argmax {qwk_argmax:.4f} | QWK expectation {qwk_expectation:.4f}"
        )
        if epoch_qwk > best_qwk + HEAD_MIN_DELTA:
            best_qwk = epoch_qwk
            best_epoch = epoch
            best_decoder = epoch_decoder
            best_state = copy.deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1
            if wait >= HEAD_PATIENCE:
                print(f"Рання зупинка на епосі {epoch}. Найкраща епоха {best_epoch}, QWK {best_qwk:.4f} ({best_decoder}).")
                break
    model.load_state_dict(best_state)
    return model, best_qwk, best_epoch, best_decoder, epochs_ran


def predict_head(model, features):
    model.eval()
    loader = DataLoader(EmbeddingDataset(features), batch_size=256, shuffle=False)
    chunks = []
    with torch.no_grad():
        for batch_x in loader:
            logits = model(batch_x.to(device))
            chunks.append(torch.softmax(logits, dim=1).cpu().numpy())
    return np.concatenate(chunks)


print("Параметрів голови:", int(text_train_emb.shape[1] * N_CLASSES + N_CLASSES))
text_model, text_bert_qwk, text_best_epoch, text_decoder, text_epochs = fit_embedding_head(
    text_train_emb[idx_tr], y_tr, text_train_emb[idx_va], y_va
)
val_probs = predict_head(text_model, text_train_emb[idx_va])
val_decoded = decode_probabilities(val_probs, N_CLASSES)
val_scores = {name: quadratic_weighted_kappa(y_va, pred) for name, pred in val_decoded.items()}
chosen_pred = val_decoded[text_decoder]
print(
    f"Exp4 QWK argmax {val_scores['argmax']:.4f}, expectation {val_scores['expectation']:.4f}. "
    f"Обрано {text_decoder}, accuracy {accuracy_score(y_va, chosen_pred):.4f}."
)
print("Прогнози на валідації:")
print(pd.Series(chosen_pred).map(IDX_TO_LABEL).value_counts().sort_index())
print(f"Порівняння: TF-IDF {TFIDF_VAL_QWK:.4f}, фото {IMAGE_VAL_QWK:.4f}, DistilBERT {val_scores[text_decoder]:.4f}.")

exp4_name = "Exp4. DistilBERT заморожений + лінійна голова"
exp4_notes = (
    f"без ваг класів; argmax {val_scores['argmax']:.4f}; "
    f"expectation {val_scores['expectation']:.4f}; max_len {MAX_TEXT_MODEL_LEN}"
)
if "log_experiment" in globals():
    log_experiment(exp4_name, val_scores[text_decoder], text_decoder, text_epochs, text_best_epoch, exp4_notes)
else:
    print(exp4_name, round(val_scores[text_decoder], 4), text_decoder, "епох", text_epochs, "найкраща", text_best_epoch)

if val_scores[text_decoder] > IMAGE_VAL_QWK + HEAD_MIN_DELTA:
    print("Текст обігнав фото на валідації. Навчаю голову на повному train і пишу submission.csv.")
    torch.manual_seed(SEED)
    final_text_model = TextEmbeddingHead(text_train_emb.shape[1], N_CLASSES).to(device)
    final_optimizer = torch.optim.AdamW(final_text_model.parameters(), lr=HEAD_LR, weight_decay=HEAD_WEIGHT_DECAY)
    final_criterion = nn.CrossEntropyLoss()
    final_loader = DataLoader(
        EmbeddingDataset(text_train_emb, y_all),
        batch_size=64,
        shuffle=True,
        generator=torch.Generator().manual_seed(SEED),
    )
    final_epochs = max(int(text_best_epoch), 1)
    final_text_model.train()
    for epoch in range(1, final_epochs + 1):
        total_loss, seen = 0.0, 0
        for batch_x, batch_y in final_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            final_optimizer.zero_grad(set_to_none=True)
            logits = final_text_model(batch_x)
            loss = final_criterion(logits, batch_y)
            loss.backward()
            nn.utils.clip_grad_norm_(final_text_model.parameters(), 5.0)
            final_optimizer.step()
            total_loss += loss.item() * batch_y.size(0)
            seen += batch_y.size(0)
        print(f"Повне навчання тексту, епоха {epoch}/{final_epochs}, train loss {total_loss / max(seen, 1):.4f}")
    test_probs = predict_head(final_text_model, text_test_emb)
    test_idx = decode_probabilities(test_probs, N_CLASSES)[text_decoder]
    submission = pd.DataFrame({
        "PetID": [normalize_id(pet_id) for pet_id in test_ids],
        "AdoptionSpeed": [IDX_TO_LABEL[int(index)] for index in test_idx],
    })
    submission = submission.drop_duplicates("PetID", keep="first")
    submission["AdoptionSpeed"] = submission["AdoptionSpeed"].astype(int)
    submission = submission[["PetID", "AdoptionSpeed"]]
    submission.to_csv(WORKING / "submission.csv", index=False)
    print(f"Збережено submission.csv, рядків {len(submission)}.")
    print(submission["AdoptionSpeed"].value_counts().sort_index())
else:
    print(
        "Новий submission.csv не записано: val QWK не вищий за фото більше ніж на 0.001. "
        "Публічний файл із ResNet лишається чинним."
    )


### Висновки експерименту 4

Заморожений DistilBERT (середнє за токенами, не довше 256) і лінійна голова на 3076 параметрів дають val QWK 0.163 за argmax і 0.122 за очікуванням класу. Accuracy 0.379. По епохах argmax ішов 0.018, 0.071, 0.090, 0.163, 0.142, 0.145. Рання зупинка спрацювала на 6-й епосі, найкраща — 4-та: тоді val loss був 1.318. На 6-й епосі val loss трохи нижчий (1.310), але QWK уже 0.145, тож зупинка за каппою зупинилась на правильній епосі. Від першої до шостої епохи train loss знизився з 1.352 до 1.292. Для чотирьох рівноймовірних класів крос-ентропія дорівнює близько 1.386, тож голова майже не відійшла від випадкового прогнозу.

Це на 0.110 QWK нижче за мішок слів (0.273) і на 0.262 нижче за фото (0.425). Новий `submission.csv` не записано. Публічний файл ResNet зі score 0.5839 лишається чинним.

Ваг класів у цій голові не було. Зсув у бік класу 1, який дав публічний сабміт, тут зник: клас 1 отримав 121 прогноз із 1287 (9.4% за частки 18.6% у train). Клас 3 майже відсутній: 30 прогнозів (2.3% за частки 20.6%). Клас 4 забирає 671 прогноз (52.1% за частки 33.2%), клас 2 — 465 (36.1% за 27.6%). Accuracy 0.379 лише трохи вища за частку найчастішого класу 0.332, а QWK лишається низькою, бо далекі помилки в бік класу 4 вона штрафує сильніше.

Попередження про послідовність із 1106 токенів — контроль токенайзера до обрізання. У модель увійшло не більше 256 токенів. Обрізано 364 із 6431 описів train (5.7%) і 111 із 1891 test (5.9%). Рядки `vocab_projector` і `vocab_transform` у звіті завантаження — голова маскованого мовного моделювання; енкодер її не використовує.

Корисний сигнал в описі є: ті самі тексти в TF-IDF дали 0.273. Середній вектор замороженого DistilBERT розмазує кілька змістовних слів (вік, вакцинація, тон історії) по решті абзацу. У лінійного шару на 768 числах немає окремої координати на кшталт «kitten» чи «vaccinated», яку бачить логістична регресія на мішку слів. Прихований шар на цих самих векторах відсутній сигнал не створить, і клеїти їх до ResNet означає додати до фото майже порожній блок.

Наступний блок перевіряє взаємодію тексту й фото на ознаках, які вже працюють окремо: стиснений TF-IDF поруч із кешованим вектором ResNet і одна голова з одним прихованим шаром, без ваг класів. Повний енкодер DistilBERT лишається замороженим: на цих кількох тисячах описів BiLSTM з 1.13 млн параметрів уже перенавчився.


## 12. Експеримент 5. TF-IDF і ResNet в одній голові

Цей блок не навчає заново TF-IDF-класифікатор, ResNet, BiLSTM і DistilBERT. Попередні комірки лишаються історією прогону.

Як запускати на Kaggle: **не натискайте Run All**. Виконайте комірки цього розділу зверху вниз. Якщо ядро вже скинуте, перша комірка перечитає таблиці і відновить розбиття 5144/1287 за зерном 42, друга візьме кеш `resnet18_pet_features.npz` або один раз порахує заморожені ознаки фото. Старі голови при цьому не навчаються. Для перерахунку фото потрібні GPU і Internet.

Гіпотеза кроку: опис і фото корисні разом, якщо текст узяти в тій формі, яка вже дала val QWK 0.273. Середній вектор DistilBERT дав 0.163, тому в склейку він не входить. TF-IDF стискається до 128 компонент, стає поруч із вектором ResNet (512 чисел і прапорець відсутності фото), і обидва блоки зводяться до однієї шкали за середнім і розкидом навчальної частини. Одна голова з прихованим шаром на 128 читає їх разом і видає один прогноз: поєднання на кшталт «молодий опис і сильний візуальний вектор» може підняти клас, навіть коли кожен блок окремо цього не робить.

Ваг класів немає. Рання зупинка та сама: val QWK, мінімальний приріст 0.001, терпіння 2 епохи, не більше 8. Новий `submission.csv` пишеться лише якщо val QWK вищий за 0.425 більше ніж на 0.001. Інакше чинним лишається файл ResNet зі score 0.5839.


In [ ]:
import os
import random
import re
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import cohen_kappa_score
from sklearn.model_selection import train_test_split

SEED = 42
VAL_SIZE = 0.2
MAX_IMAGES = 3
FEATURE_BATCH_SIZE = 64
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
DATA_ROOT = Path("/kaggle/input/competitions/deep-learning-for-computer-vision-and-nlp-2026-10")
IMAGE_DIR = DATA_ROOT / "images"
WORKING = Path("/kaggle/working")
WORKING.mkdir(parents=True, exist_ok=True)
NUM_WORKERS = 2 if os.name != "nt" else 0
if "device" not in globals():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

needs_tables = "train_df" not in globals() or "idx_tr" not in globals() or "y_all" not in globals()
if not needs_tables:
    print(f"Таблиці вже в пам'яті: train {len(train_df)}, val {len(idx_va)}. Повторно не читаю.")
else:
    TRAIN_CSV = DATA_ROOT / "train.csv"
    TEST_CSV = DATA_ROOT / "test.csv"
    SAMPLE_CSV = DATA_ROOT / "sample_submission.csv"

    def find_column(columns, options):
        lookup = {str(column).lower(): column for column in columns}
        for option in options:
            if option.lower() in lookup:
                return lookup[option.lower()]
        return None

    def normalize_id(value):
        if pd.isna(value):
            return ""
        text = str(value).strip()
        if text.endswith(".0"):
            text = text[:-2]
        return text

    def read_table(path):
        frame = pd.read_csv(path)
        frame.columns = [str(column).strip() for column in frame.columns]
        return frame

    def quadratic_weighted_kappa(y_true, y_pred):
        y_true = np.asarray(y_true).astype(int)
        y_pred = np.asarray(y_pred).astype(int)
        if len(y_true) == 0:
            return 0.0
        score = cohen_kappa_score(y_true, y_pred, weights="quadratic")
        if score != score:
            return 0.0
        return float(score)

    train_df = read_table(TRAIN_CSV)
    test_df = read_table(TEST_CSV)
    sample_df = read_table(SAMPLE_CSV)
    id_col = find_column(train_df.columns, ["PetID", "pet_id", "id"])
    target_col = find_column(train_df.columns, ["AdoptionSpeed", "adoption_speed", "label", "target"])
    text_col = find_column(train_df.columns, ["Description", "description", "desc", "text", "comments"])
    test_id_col = find_column(test_df.columns, ["PetID", "pet_id", "id"]) or id_col
    train_df[id_col] = train_df[id_col].map(normalize_id)
    test_df[test_id_col] = test_df[test_id_col].map(normalize_id)
    if int(train_df[id_col].duplicated().sum()):
        train_df = train_df.drop_duplicates(id_col, keep="first")
    train_df = train_df.reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)
    if target_col in test_df.columns:
        test_df = test_df.drop(columns=[target_col])
    test_text_col = find_column(test_df.columns, ["Description", "description", "desc", "text", "comments"])
    if text_col is None:
        text_col = "Description"
        train_df[text_col] = ""
        test_df[text_col] = ""
    elif test_text_col is None:
        test_df[text_col] = ""
    elif test_text_col != text_col:
        test_df[text_col] = test_df[test_text_col]
    raw_target = train_df[target_col]
    if pd.api.types.is_numeric_dtype(raw_target):
        numeric_target = raw_target.astype(float)
        if np.allclose(numeric_target.to_numpy(), np.rint(numeric_target.to_numpy())):
            train_df[target_col] = numeric_target.round().astype(int)
    classes = sorted(train_df[target_col].dropna().unique().tolist())
    LABEL_TO_IDX = {label: index for index, label in enumerate(classes)}
    IDX_TO_LABEL = {index: label for label, index in LABEL_TO_IDX.items()}
    N_CLASSES = len(classes)
    train_df = train_df.dropna(subset=[target_col]).reset_index(drop=True)
    y_all = train_df[target_col].map(LABEL_TO_IDX).to_numpy(dtype=int)
    idx_tr, idx_va = train_test_split(
        np.arange(len(train_df)), test_size=VAL_SIZE, random_state=SEED, stratify=y_all
    )
    y_tr = y_all[idx_tr]
    y_va = y_all[idx_va]
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    print(f"Таблиці прочитано заново: train {len(train_df)}, val {len(idx_va)}, test {len(test_df)}.")

if "train_ids" not in globals():
    train_ids = train_df[id_col].tolist()
if "test_ids" not in globals():
    test_ids = test_df[test_id_col].tolist()

if len(idx_va) != 1287:
    print(f"Увага: у валідації {len(idx_va)} рядків, у минулому прогоні було 1287.")
else:
    print("Розбиття збігається з минулим прогоном: 1287 валідаційних рядків.")

if "text_clean" in train_df.columns and "text_clean" in test_df.columns:
    print("Очищений опис уже є, повторно не чищу.")
else:
    NEGATION_TOKENS = {
        "no", "not", "nor", "never", "none", "nobody", "nothing",
        "neither", "nowhere", "without", "cannot",
    }
    CONTRACTIONS = {
        "ain't": "is not",
        "aren't": "are not",
        "can't": "cannot",
        "couldn't": "could not",
        "didn't": "did not",
        "doesn't": "does not",
        "don't": "do not",
        "hadn't": "had not",
        "hasn't": "has not",
        "haven't": "have not",
        "he's": "he is",
        "here's": "here is",
        "how's": "how is",
        "i'm": "i am",
        "i've": "i have",
        "isn't": "is not",
        "it's": "it is",
        "let's": "let us",
        "mustn't": "must not",
        "she's": "she is",
        "that's": "that is",
        "there's": "there is",
        "they're": "they are",
        "they've": "they have",
        "we're": "we are",
        "we've": "we have",
        "weren't": "were not",
        "what's": "what is",
        "where's": "where is",
        "who's": "who is",
        "won't": "will not",
        "wouldn't": "would not",
        "you're": "you are",
        "you've": "you have",
    }
    CONTRACTION_RE = re.compile(
        r"\b(" + "|".join(sorted(map(re.escape, CONTRACTIONS), key=len, reverse=True)) + r")\b"
    )
    FALLBACK_STOPWORDS = set(
        "a about above after again against all am an and any are as at be because been "
        "before being below between both but by can did do does doing down during each "
        "few for from further had has have having he her here hers herself him himself "
        "his how i if in into is it its itself just me more most my myself of off on "
        "once only or other our ours ourselves out over own same she should so some "
        "such than that the their theirs them themselves then there these they this "
        "those through to too under until up very was we were what when where which "
        "while who whom why will with you your yours yourself yourselves".split()
    )

    def load_stopwords():
        try:
            import nltk
            from nltk.corpus import stopwords

            nltk.download("stopwords", quiet=True)
            words = set(stopwords.words("english"))
            print("Стоп-слова: NLTK, english")
        except Exception as exc:
            print(f"Стоп-слова NLTK недоступні ({type(exc).__name__}: {exc}). Використовую вбудований список.")
            words = set(FALLBACK_STOPWORDS)
        return words - NEGATION_TOKENS

    STOPWORDS = load_stopwords()

    def clean_text(value):
        if not isinstance(value, str):
            return ""
        text = value.lower().replace("’", "'").replace("`", "'")
        text = CONTRACTION_RE.sub(lambda match: CONTRACTIONS[match.group(0)], text)
        text = re.sub(r"https?://\S+|www\.\S+", " ", text)
        text = re.sub(r"[^a-z\s]", " ", text)
        tokens = [token for token in text.split() if len(token) > 1 and token not in STOPWORDS]
        return " ".join(tokens)

    train_df["text_clean"] = train_df[text_col].fillna("").astype(str).map(clean_text)
    test_df["text_clean"] = test_df[text_col].fillna("").astype(str).map(clean_text)
    print("Порожніх описів після очищення:", float(train_df["text_clean"].eq("").mean()))

print("Пристрій:", device)


In [ ]:
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision.models import ResNet18_Weights, resnet18
from tqdm.auto import tqdm

reuse_images = (
    "image_combined" in globals()
    and "image_combined_test" in globals()
    and len(image_combined) == len(train_df)
    and len(image_combined_test) == len(test_ids)
)
if reuse_images:
    print(
        "Вектори ResNet уже в пам'яті:",
        tuple(image_combined.shape),
        tuple(image_combined_test.shape),
        ". Заново не рахую.",
    )
else:
    def list_image_files(image_dir):
        files = []
        for path in Path(image_dir).rglob("*"):
            if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS:
                files.append(path)
        return files

    def link_images(pet_ids, files):
        pet_set = set(pet_ids)
        grouped = {pet_id: [] for pet_id in pet_set}
        for path in files:
            stem = path.stem
            pet_id = None
            order = 0
            if stem in pet_set:
                pet_id = stem
            else:
                for separator in ("-", "_"):
                    if separator not in stem:
                        continue
                    base, rest = stem.rsplit(separator, 1)
                    if base in pet_set:
                        pet_id = base
                        order = int(rest) if rest.isdigit() else 10**6
                        break
            if pet_id is not None:
                grouped[pet_id].append((order, path))
        linked = {}
        for pet_id, items in grouped.items():
            items.sort(key=lambda item: (item[0], item[1].name))
            linked[pet_id] = [path for _, path in items]
        return linked

    class PetImageDataset(Dataset):
        def __init__(self, records, transform):
            self.records = records
            self.transform = transform

        def __len__(self):
            return len(self.records)

        def __getitem__(self, index):
            pet_id, path = self.records[index]
            try:
                with Image.open(path) as image:
                    tensor = self.transform(image.convert("RGB"))
                ok = 1
            except Exception:
                tensor = torch.zeros(3, 224, 224)
                ok = 0
            return tensor, ok, pet_id

    def extract_resnet_features(paths_by_pet, backbone, transform):
        records = [(pet_id, path) for pet_id, paths in paths_by_pet.items() for path in paths]
        sums = {pet_id: np.zeros(512, dtype=np.float32) for pet_id in paths_by_pet}
        counts = {pet_id: 0 for pet_id in paths_by_pet}
        if not records:
            return {pet_id: (sums[pet_id], 0) for pet_id in paths_by_pet}
        loader = DataLoader(
            PetImageDataset(records, transform),
            batch_size=FEATURE_BATCH_SIZE,
            shuffle=False,
            num_workers=NUM_WORKERS,
        )
        backbone.eval()
        for tensors, oks, pet_ids_batch in tqdm(loader, desc="ResNet18"):
            features = backbone(tensors.to(device)).detach().cpu().numpy()
            oks = oks.numpy()
            for index, pet_id in enumerate(pet_ids_batch):
                if int(oks[index]) == 1:
                    sums[pet_id] += features[index]
                    counts[pet_id] += 1
        extracted = {}
        for pet_id in paths_by_pet:
            if counts[pet_id] > 0:
                extracted[pet_id] = (sums[pet_id] / counts[pet_id], counts[pet_id])
            else:
                extracted[pet_id] = (np.zeros(512, dtype=np.float32), 0)
        return extracted

    def stack_image_features(pet_ids, feature_map):
        vectors = np.zeros((len(pet_ids), 512), dtype=np.float32)
        missing = np.ones((len(pet_ids), 1), dtype=np.float32)
        for index, pet_id in enumerate(pet_ids):
            vector, count = feature_map[pet_id]
            if count > 0:
                vectors[index] = vector
                missing[index, 0] = 0.0
        return np.concatenate([vectors, missing], axis=1)

    def locate_resnet_cache():
        direct = WORKING / "resnet18_pet_features.npz"
        if direct.exists():
            return direct
        root = Path("/kaggle/input")
        if root.exists():
            found = sorted(root.rglob("resnet18_pet_features.npz"))
            if found:
                print(f"Кеш знайдено серед вхідних даних: {found[0]}")
                return found[0]
        return None

    def load_feature_map(cache_path):
        cached = np.load(cache_path, allow_pickle=False)
        cached_ids = [normalize_id(pet_id) for pet_id in cached["ids"].tolist()]
        counts = np.array(cached["counts"], copy=True)
        matrix = np.array(cached["features"], dtype=np.float32, copy=True)
        cached.close()
        feature_map = {}
        for index, pet_id in enumerate(cached_ids):
            feature_map[pet_id] = (matrix[index].copy(), int(counts[index]))
        return feature_map

    needed = set(train_ids) | set(test_ids)
    cache_path = locate_resnet_cache()
    feature_map = None
    if cache_path is not None:
        loaded = load_feature_map(cache_path)
        if set(loaded) >= needed:
            feature_map = loaded
            print(f"Ознаки ResNet завантажено з кешу: {cache_path}")
        else:
            print("Кеш ознак не покриває поточні ідентифікатори, рахую заново.")
    if feature_map is None:
        print("Кешу ResNet немає. Рахую заморожені ознаки, голову Exp2 не навчаю.")
        if "model_image_paths" not in globals():
            image_files = list_image_files(IMAGE_DIR)
            all_pet_ids = list(dict.fromkeys(list(train_ids) + list(test_ids)))
            full_image_paths = link_images(all_pet_ids, image_files)
            model_image_paths = {pet_id: paths[:MAX_IMAGES] for pet_id, paths in full_image_paths.items()}
            print("Файлів зображень:", len(image_files))
        try:
            weights = ResNet18_Weights.IMAGENET1K_V1
            backbone = resnet18(weights=weights)
            transform = weights.transforms()
        except Exception as exc:
            raise RuntimeError(
                "Не вдалося завантажити ваги ResNet18. Увімкніть Internet у налаштуваннях ноутбука."
            ) from exc
        backbone.fc = nn.Identity()
        backbone.to(device)
        backbone.eval()
        for parameter in backbone.parameters():
            parameter.requires_grad = False
        feature_map = extract_resnet_features(model_image_paths, backbone, transform)
        ids_array = np.array(list(feature_map.keys()))
        cache_out = WORKING / "resnet18_pet_features.npz"
        np.savez_compressed(
            cache_out,
            ids=ids_array,
            features=np.stack([feature_map[pet_id][0] for pet_id in ids_array]),
            counts=np.array([feature_map[pet_id][1] for pet_id in ids_array], dtype=np.int32),
        )
        print(f"Ознаки збережено: {cache_out}")
        del backbone
        if device.type == "cuda":
            torch.cuda.empty_cache()

    image_combined = stack_image_features(train_ids, feature_map)
    image_combined_test = stack_image_features(test_ids, feature_map)

print("Матриця фото train/test:", tuple(image_combined.shape), tuple(image_combined_test.shape))


In [ ]:
import copy

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, Dataset

SVD_DIM = 128
JOINT_HIDDEN = 128
JOINT_DROPOUT = 0.3
JOINT_LR = 1e-3
JOINT_WEIGHT_DECAY = 1e-4
JOINT_MAX_EPOCHS = 8
JOINT_PATIENCE = 2
JOINT_MIN_DELTA = 0.001
JOINT_BATCH = 64
IMAGE_VAL_QWK = 0.4247
TFIDF_VAL_QWK = 0.2730
DISTILBERT_VAL_QWK = 0.1629
TFIDF_MAX_FEATURES = 20000
TFIDF_MIN_DF = 2


class JointDataset(Dataset):
    def __init__(self, features, labels=None):
        self.features = torch.tensor(np.asarray(features), dtype=torch.float32)
        self.labels = None if labels is None else torch.tensor(np.asarray(labels), dtype=torch.long)

    def __len__(self):
        return self.features.size(0)

    def __getitem__(self, index):
        if self.labels is None:
            return self.features[index]
        return self.features[index], self.labels[index]


class JointHead(nn.Module):
    def __init__(self, in_dim, hidden, n_classes, dropout=JOINT_DROPOUT):
        super().__init__()
        self.net = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, n_classes),
        )

    def forward(self, features):
        return self.net(features)


def decode_probabilities(probs, n_classes):
    class_index = np.arange(n_classes, dtype=np.float64)
    expectation = np.clip(np.rint(probs @ class_index), 0, n_classes - 1).astype(int)
    return {"argmax": probs.argmax(axis=1).astype(int), "expectation": expectation}


def fit_joint_head(features_train, labels_train, features_val, labels_val):
    torch.manual_seed(SEED)
    model = JointHead(features_train.shape[1], JOINT_HIDDEN, N_CLASSES).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=JOINT_LR, weight_decay=JOINT_WEIGHT_DECAY)
    criterion = nn.CrossEntropyLoss()
    train_loader = DataLoader(
        JointDataset(features_train, labels_train),
        batch_size=JOINT_BATCH,
        shuffle=True,
        generator=torch.Generator().manual_seed(SEED),
    )
    val_loader = DataLoader(JointDataset(features_val, labels_val), batch_size=256, shuffle=False)
    best_qwk = -1.0
    best_epoch = 1
    best_decoder = "argmax"
    best_state = copy.deepcopy(model.state_dict())
    wait = 0
    epochs_ran = 0
    for epoch in range(1, JOINT_MAX_EPOCHS + 1):
        epochs_ran = epoch
        model.train()
        total_loss, seen = 0.0, 0
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            optimizer.zero_grad(set_to_none=True)
            logits = model(batch_x)
            loss = criterion(logits, batch_y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            total_loss += loss.item() * batch_y.size(0)
            seen += batch_y.size(0)
        model.eval()
        prob_chunks, true_chunks = [], []
        val_loss, val_seen = 0.0, 0
        with torch.no_grad():
            for batch_x, batch_y in val_loader:
                batch_x, batch_y = batch_x.to(device), batch_y.to(device)
                logits = model(batch_x)
                val_loss += criterion(logits, batch_y).item() * batch_y.size(0)
                val_seen += batch_y.size(0)
                prob_chunks.append(torch.softmax(logits, dim=1).cpu().numpy())
                true_chunks.append(batch_y.cpu().numpy())
        probs = np.concatenate(prob_chunks)
        y_true = np.concatenate(true_chunks)
        decoded = decode_probabilities(probs, N_CLASSES)
        qwk_argmax = quadratic_weighted_kappa(y_true, decoded["argmax"])
        qwk_expectation = quadratic_weighted_kappa(y_true, decoded["expectation"])
        epoch_decoder = "expectation" if qwk_expectation > qwk_argmax else "argmax"
        epoch_qwk = qwk_expectation if epoch_decoder == "expectation" else qwk_argmax
        print(
            f"Епоха {epoch:02d}/{JOINT_MAX_EPOCHS} | train loss {total_loss / max(seen, 1):.4f} | "
            f"val loss {val_loss / max(val_seen, 1):.4f} | "
            f"QWK argmax {qwk_argmax:.4f} | QWK expectation {qwk_expectation:.4f}"
        )
        if epoch_qwk > best_qwk + JOINT_MIN_DELTA:
            best_qwk = epoch_qwk
            best_epoch = epoch
            best_decoder = epoch_decoder
            best_state = copy.deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1
            if wait >= JOINT_PATIENCE:
                break
    model.load_state_dict(best_state)
    print(f"Зупинка після епохи {epochs_ran}. Найкраща епоха {best_epoch}, QWK {best_qwk:.4f} ({best_decoder}).")
    return model, best_qwk, best_epoch, best_decoder, epochs_ran


def predict_joint(model, features):
    model.eval()
    loader = DataLoader(JointDataset(features), batch_size=256, shuffle=False)
    chunks = []
    with torch.no_grad():
        for batch_x in loader:
            logits = model(batch_x.to(device))
            chunks.append(torch.softmax(logits, dim=1).cpu().numpy())
    return np.concatenate(chunks)


def text_svd(frame_fit, frames_transform):
    vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=TFIDF_MIN_DF, max_features=TFIDF_MAX_FEATURES)
    sparse_fit = vectorizer.fit_transform(frame_fit)
    svd_dim = min(SVD_DIM, sparse_fit.shape[1] - 1, sparse_fit.shape[0] - 1)
    svd = TruncatedSVD(n_components=svd_dim, random_state=SEED)
    dense_fit = svd.fit_transform(sparse_fit)
    transformed = [svd.transform(vectorizer.transform(frame)) for frame in frames_transform]
    explained = float(svd.explained_variance_ratio_.sum())
    return dense_fit, transformed, explained, sparse_fit.shape[1], svd_dim


text_tr, (text_va,), explained, n_tfidf, svd_dim = text_svd(
    train_df.iloc[idx_tr]["text_clean"],
    [train_df.iloc[idx_va]["text_clean"]],
)
print(
    f"TF-IDF ознак {n_tfidf}, компонент SVD {svd_dim}, "
    f"пояснена дисперсія {explained:.3f}. Ваг класів немає."
)
scaler = StandardScaler()
fused_tr = scaler.fit_transform(np.concatenate([image_combined[idx_tr], text_tr], axis=1)).astype(np.float32)
fused_va = scaler.transform(np.concatenate([image_combined[idx_va], text_va], axis=1)).astype(np.float32)
print("Спільна матриця train/val:", fused_tr.shape, fused_va.shape)

joint_model, joint_qwk, joint_best_epoch, joint_decoder, joint_epochs = fit_joint_head(fused_tr, y_tr, fused_va, y_va)
val_probs = predict_joint(joint_model, fused_va)
val_decoded = decode_probabilities(val_probs, N_CLASSES)
val_scores = {name: quadratic_weighted_kappa(y_va, pred) for name, pred in val_decoded.items()}
chosen_pred = val_decoded[joint_decoder]
print(
    f"Exp5 QWK argmax {val_scores['argmax']:.4f}, expectation {val_scores['expectation']:.4f}. "
    f"Обрано {joint_decoder}, accuracy {accuracy_score(y_va, chosen_pred):.4f}."
)
print("Прогнози на валідації:")
print(pd.Series(chosen_pred).map(IDX_TO_LABEL).value_counts().sort_index())
print(
    f"Порівняння: TF-IDF {TFIDF_VAL_QWK:.4f}, фото {IMAGE_VAL_QWK:.4f}, "
    f"DistilBERT {DISTILBERT_VAL_QWK:.4f}, спільна голова {val_scores[joint_decoder]:.4f}."
)

exp5_name = "Exp5. ResNet + SVD(TF-IDF), прихований шар"
exp5_notes = (
    f"без ваг класів; svd {svd_dim}; пояснена дисперсія {explained:.3f}; "
    f"argmax {val_scores['argmax']:.4f}; expectation {val_scores['expectation']:.4f}"
)
if "log_experiment" in globals():
    log_experiment(exp5_name, val_scores[joint_decoder], joint_decoder, joint_epochs, joint_best_epoch, exp5_notes)
else:
    print(exp5_name, round(val_scores[joint_decoder], 4), joint_decoder, "епох", joint_epochs, "найкраща", joint_best_epoch)

if val_scores[joint_decoder] > IMAGE_VAL_QWK + JOINT_MIN_DELTA:
    print("Спільна голова обігнала фото на валідації. Навчаю її на повному train і пишу submission.csv.")
    text_all, (text_test,), _, _, _ = text_svd(
        train_df["text_clean"],
        [test_df["text_clean"]],
    )
    final_scaler = StandardScaler()
    fused_all = final_scaler.fit_transform(np.concatenate([image_combined, text_all], axis=1)).astype(np.float32)
    fused_test = final_scaler.transform(np.concatenate([image_combined_test, text_test], axis=1)).astype(np.float32)
    torch.manual_seed(SEED)
    final_joint = JointHead(fused_all.shape[1], JOINT_HIDDEN, N_CLASSES).to(device)
    final_optimizer = torch.optim.AdamW(final_joint.parameters(), lr=JOINT_LR, weight_decay=JOINT_WEIGHT_DECAY)
    final_criterion = nn.CrossEntropyLoss()
    final_loader = DataLoader(
        JointDataset(fused_all, y_all),
        batch_size=JOINT_BATCH,
        shuffle=True,
        generator=torch.Generator().manual_seed(SEED),
    )
    final_epochs = max(int(joint_best_epoch), 1)
    final_joint.train()
    for epoch in range(1, final_epochs + 1):
        total_loss, seen = 0.0, 0
        for batch_x, batch_y in final_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            final_optimizer.zero_grad(set_to_none=True)
            logits = final_joint(batch_x)
            loss = final_criterion(logits, batch_y)
            loss.backward()
            nn.utils.clip_grad_norm_(final_joint.parameters(), 5.0)
            final_optimizer.step()
            total_loss += loss.item() * batch_y.size(0)
            seen += batch_y.size(0)
        print(f"Повне навчання спільної голови, епоха {epoch}/{final_epochs}, train loss {total_loss / max(seen, 1):.4f}")
    test_probs = predict_joint(final_joint, fused_test)
    test_idx = decode_probabilities(test_probs, N_CLASSES)[joint_decoder]
    submission = pd.DataFrame({
        "PetID": [normalize_id(pet_id) for pet_id in test_ids],
        "AdoptionSpeed": [IDX_TO_LABEL[int(index)] for index in test_idx],
    })
    submission = submission.drop_duplicates("PetID", keep="first")
    submission["AdoptionSpeed"] = submission["AdoptionSpeed"].astype(int)
    submission = submission[["PetID", "AdoptionSpeed"]]
    submission.to_csv(WORKING / "submission.csv", index=False)
    print(f"Збережено submission.csv, рядків {len(submission)}.")
    print(submission["AdoptionSpeed"].value_counts().sort_index())
else:
    print(
        "Новий submission.csv не записано: val QWK не вищий за фото більше ніж на 0.001. "
        "Публічний файл із ResNet лишається чинним."
    )


### Висновки експерименту 5

Спільна голова (вектор ResNet і 128 компонент SVD, прихований шар 128, без ваг класів) дає val QWK 0.464 за argmax і 0.394 за очікуванням класу. Accuracy 0.495. По епохах argmax ішов 0.436, 0.464, 0.460, 0.455. Зупинка після 4-ї епохи, найкраща — 2-га: val loss тоді 1.182. На 4-й епосі train loss упав до 1.053, а val loss підріс до 1.188, тож зупинка за каппою лишила епоху до цього підйому.

Це на 0.039 QWK вище за саме фото (0.425), на 0.191 вище за TF-IDF (0.273) і на 0.301 вище за DistilBERT (0.163). Один прогноз після спільного шару кращий за кожен блок окремо. Файл записано на всі 1891 тестових рядків і відправлено: публічний score 0.6676 проти 0.5839 у попереднього ResNet. До кращого публічного результату учасників, 0.8088, лишається 0.141.

128 компонент пояснюють 0.227 дисперсії TF-IDF. У голову потрапила менша частина словника, яким логістична регресія набрала 0.273.

Ваг класів не було. На валідації клас 1 отримав 263 прогнози з 1287 (20.4% за частки 18.6% у train), клас 2 — 324 (25.2% за 27.6%), клас 3 — 159 (12.4% за 20.6%), клас 4 — 541 (42.0% за 33.2%). На тесті: клас 1 — 316 (16.7%), клас 2 — 526 (27.8%), клас 3 — 217 (11.5%), клас 4 — 832 (44.0%). Класи 1 і 2 лягли близько до train. Клас 3 лишається рідкісним, клас 4 забирає 44% відповідей за частки 33.2% у train.

Повне навчання йшло 2 епохи. Train loss 1.251 і 1.138 повторює перші дві епохи на розбитті (1.257 і 1.140).

Наступний блок лишає той самий вектор фото і ту саму голову, а піднімає SVD з 128 до 512 компонент. Новий сабміт має сенс лише коли val QWK вищий за 0.4639 більше ніж на 0.001. Інакше чинним лишається файл зі score 0.6676.


## 13. Експеримент 6. Той самий ResNet, SVD на 512 компонент

Exp5 дав val QWK 0.464 і публічний score 0.6676. 128 компонент пояснили лише 0.227 дисперсії TF-IDF, тож більша частина словника до голови не дійшла. Цей блок лишає вектор фото, шкалювання і голову з прихованим шаром на 128 без змін. Змінюється одне число: текст стискається до 512 компонент.

Як запускати на Kaggle: **не натискайте Run All**. Не перезапускайте останню код-комірку розділу 12: вона знову запише `submission.csv` моделі Exp5. Виконайте лише комірки цього розділу зверху вниз. Якщо ядро вже скинуте, перші дві комірки відновлять таблиці, очищений опис і кеш фото. Голову Exp5 вони не навчають. Якщо кешу `resnet18_pet_features.npz` немає, фото рахуються заново замороженим ResNet; для цього потрібні GPU і Internet.

Поріг сабміту: val QWK вищий за 0.4639 більше ніж на 0.001. Інакше чинним лишається файл зі score 0.6676.


In [ ]:
import os
import random
import re
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import cohen_kappa_score
from sklearn.model_selection import train_test_split

SEED = 42
VAL_SIZE = 0.2
MAX_IMAGES = 3
FEATURE_BATCH_SIZE = 64
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
DATA_ROOT = Path("/kaggle/input/competitions/deep-learning-for-computer-vision-and-nlp-2026-10")
IMAGE_DIR = DATA_ROOT / "images"
WORKING = Path("/kaggle/working")
WORKING.mkdir(parents=True, exist_ok=True)
NUM_WORKERS = 2 if os.name != "nt" else 0
if "device" not in globals():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

needs_tables = "train_df" not in globals() or "idx_tr" not in globals() or "y_all" not in globals()
if not needs_tables:
    print(f"Таблиці вже в пам'яті: train {len(train_df)}, val {len(idx_va)}. Повторно не читаю.")
else:
    TRAIN_CSV = DATA_ROOT / "train.csv"
    TEST_CSV = DATA_ROOT / "test.csv"
    SAMPLE_CSV = DATA_ROOT / "sample_submission.csv"

    def find_column(columns, options):
        lookup = {str(column).lower(): column for column in columns}
        for option in options:
            if option.lower() in lookup:
                return lookup[option.lower()]
        return None

    def normalize_id(value):
        if pd.isna(value):
            return ""
        text = str(value).strip()
        if text.endswith(".0"):
            text = text[:-2]
        return text

    def read_table(path):
        frame = pd.read_csv(path)
        frame.columns = [str(column).strip() for column in frame.columns]
        return frame

    def quadratic_weighted_kappa(y_true, y_pred):
        y_true = np.asarray(y_true).astype(int)
        y_pred = np.asarray(y_pred).astype(int)
        if len(y_true) == 0:
            return 0.0
        score = cohen_kappa_score(y_true, y_pred, weights="quadratic")
        if score != score:
            return 0.0
        return float(score)

    train_df = read_table(TRAIN_CSV)
    test_df = read_table(TEST_CSV)
    sample_df = read_table(SAMPLE_CSV)
    id_col = find_column(train_df.columns, ["PetID", "pet_id", "id"])
    target_col = find_column(train_df.columns, ["AdoptionSpeed", "adoption_speed", "label", "target"])
    text_col = find_column(train_df.columns, ["Description", "description", "desc", "text", "comments"])
    test_id_col = find_column(test_df.columns, ["PetID", "pet_id", "id"]) or id_col
    train_df[id_col] = train_df[id_col].map(normalize_id)
    test_df[test_id_col] = test_df[test_id_col].map(normalize_id)
    if int(train_df[id_col].duplicated().sum()):
        train_df = train_df.drop_duplicates(id_col, keep="first")
    train_df = train_df.reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)
    if target_col in test_df.columns:
        test_df = test_df.drop(columns=[target_col])
    test_text_col = find_column(test_df.columns, ["Description", "description", "desc", "text", "comments"])
    if text_col is None:
        text_col = "Description"
        train_df[text_col] = ""
        test_df[text_col] = ""
    elif test_text_col is None:
        test_df[text_col] = ""
    elif test_text_col != text_col:
        test_df[text_col] = test_df[test_text_col]
    raw_target = train_df[target_col]
    if pd.api.types.is_numeric_dtype(raw_target):
        numeric_target = raw_target.astype(float)
        if np.allclose(numeric_target.to_numpy(), np.rint(numeric_target.to_numpy())):
            train_df[target_col] = numeric_target.round().astype(int)
    classes = sorted(train_df[target_col].dropna().unique().tolist())
    LABEL_TO_IDX = {label: index for index, label in enumerate(classes)}
    IDX_TO_LABEL = {index: label for label, index in LABEL_TO_IDX.items()}
    N_CLASSES = len(classes)
    train_df = train_df.dropna(subset=[target_col]).reset_index(drop=True)
    y_all = train_df[target_col].map(LABEL_TO_IDX).to_numpy(dtype=int)
    idx_tr, idx_va = train_test_split(
        np.arange(len(train_df)), test_size=VAL_SIZE, random_state=SEED, stratify=y_all
    )
    y_tr = y_all[idx_tr]
    y_va = y_all[idx_va]
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    print(f"Таблиці прочитано заново: train {len(train_df)}, val {len(idx_va)}, test {len(test_df)}.")

if "train_ids" not in globals():
    train_ids = train_df[id_col].tolist()
if "test_ids" not in globals():
    test_ids = test_df[test_id_col].tolist()

if len(idx_va) != 1287:
    print(f"Увага: у валідації {len(idx_va)} рядків, у минулому прогоні було 1287.")
else:
    print("Розбиття збігається з минулим прогоном: 1287 валідаційних рядків.")

if "text_clean" in train_df.columns and "text_clean" in test_df.columns:
    print("Очищений опис уже є, повторно не чищу.")
else:
    NEGATION_TOKENS = {
        "no", "not", "nor", "never", "none", "nobody", "nothing",
        "neither", "nowhere", "without", "cannot",
    }
    CONTRACTIONS = {
        "ain't": "is not",
        "aren't": "are not",
        "can't": "cannot",
        "couldn't": "could not",
        "didn't": "did not",
        "doesn't": "does not",
        "don't": "do not",
        "hadn't": "had not",
        "hasn't": "has not",
        "haven't": "have not",
        "he's": "he is",
        "here's": "here is",
        "how's": "how is",
        "i'm": "i am",
        "i've": "i have",
        "isn't": "is not",
        "it's": "it is",
        "let's": "let us",
        "mustn't": "must not",
        "she's": "she is",
        "that's": "that is",
        "there's": "there is",
        "they're": "they are",
        "they've": "they have",
        "we're": "we are",
        "we've": "we have",
        "weren't": "were not",
        "what's": "what is",
        "where's": "where is",
        "who's": "who is",
        "won't": "will not",
        "wouldn't": "would not",
        "you're": "you are",
        "you've": "you have",
    }
    CONTRACTION_RE = re.compile(
        r"\b(" + "|".join(sorted(map(re.escape, CONTRACTIONS), key=len, reverse=True)) + r")\b"
    )
    FALLBACK_STOPWORDS = set(
        "a about above after again against all am an and any are as at be because been "
        "before being below between both but by can did do does doing down during each "
        "few for from further had has have having he her here hers herself him himself "
        "his how i if in into is it its itself just me more most my myself of off on "
        "once only or other our ours ourselves out over own same she should so some "
        "such than that the their theirs them themselves then there these they this "
        "those through to too under until up very was we were what when where which "
        "while who whom why will with you your yours yourself yourselves".split()
    )

    def load_stopwords():
        try:
            import nltk
            from nltk.corpus import stopwords

            nltk.download("stopwords", quiet=True)
            words = set(stopwords.words("english"))
            print("Стоп-слова: NLTK, english")
        except Exception as exc:
            print(f"Стоп-слова NLTK недоступні ({type(exc).__name__}: {exc}). Використовую вбудований список.")
            words = set(FALLBACK_STOPWORDS)
        return words - NEGATION_TOKENS

    STOPWORDS = load_stopwords()

    def clean_text(value):
        if not isinstance(value, str):
            return ""
        text = value.lower().replace("’", "'").replace("`", "'")
        text = CONTRACTION_RE.sub(lambda match: CONTRACTIONS[match.group(0)], text)
        text = re.sub(r"https?://\S+|www\.\S+", " ", text)
        text = re.sub(r"[^a-z\s]", " ", text)
        tokens = [token for token in text.split() if len(token) > 1 and token not in STOPWORDS]
        return " ".join(tokens)

    train_df["text_clean"] = train_df[text_col].fillna("").astype(str).map(clean_text)
    test_df["text_clean"] = test_df[text_col].fillna("").astype(str).map(clean_text)
    print("Порожніх описів після очищення:", float(train_df["text_clean"].eq("").mean()))

print("Пристрій:", device)


In [ ]:
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision.models import ResNet18_Weights, resnet18
from tqdm.auto import tqdm

reuse_images = (
    "image_combined" in globals()
    and "image_combined_test" in globals()
    and len(image_combined) == len(train_df)
    and len(image_combined_test) == len(test_ids)
)
if reuse_images:
    print(
        "Вектори ResNet уже в пам'яті:",
        tuple(image_combined.shape),
        tuple(image_combined_test.shape),
        ". Заново не рахую.",
    )
else:
    def list_image_files(image_dir):
        files = []
        for path in Path(image_dir).rglob("*"):
            if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS:
                files.append(path)
        return files

    def link_images(pet_ids, files):
        pet_set = set(pet_ids)
        grouped = {pet_id: [] for pet_id in pet_set}
        for path in files:
            stem = path.stem
            pet_id = None
            order = 0
            if stem in pet_set:
                pet_id = stem
            else:
                for separator in ("-", "_"):
                    if separator not in stem:
                        continue
                    base, rest = stem.rsplit(separator, 1)
                    if base in pet_set:
                        pet_id = base
                        order = int(rest) if rest.isdigit() else 10**6
                        break
            if pet_id is not None:
                grouped[pet_id].append((order, path))
        linked = {}
        for pet_id, items in grouped.items():
            items.sort(key=lambda item: (item[0], item[1].name))
            linked[pet_id] = [path for _, path in items]
        return linked

    class PetImageDataset(Dataset):
        def __init__(self, records, transform):
            self.records = records
            self.transform = transform

        def __len__(self):
            return len(self.records)

        def __getitem__(self, index):
            pet_id, path = self.records[index]
            try:
                with Image.open(path) as image:
                    tensor = self.transform(image.convert("RGB"))
                ok = 1
            except Exception:
                tensor = torch.zeros(3, 224, 224)
                ok = 0
            return tensor, ok, pet_id

    def extract_resnet_features(paths_by_pet, backbone, transform):
        records = [(pet_id, path) for pet_id, paths in paths_by_pet.items() for path in paths]
        sums = {pet_id: np.zeros(512, dtype=np.float32) for pet_id in paths_by_pet}
        counts = {pet_id: 0 for pet_id in paths_by_pet}
        if not records:
            return {pet_id: (sums[pet_id], 0) for pet_id in paths_by_pet}
        loader = DataLoader(
            PetImageDataset(records, transform),
            batch_size=FEATURE_BATCH_SIZE,
            shuffle=False,
            num_workers=NUM_WORKERS,
        )
        backbone.eval()
        for tensors, oks, pet_ids_batch in tqdm(loader, desc="ResNet18"):
            features = backbone(tensors.to(device)).detach().cpu().numpy()
            oks = oks.numpy()
            for index, pet_id in enumerate(pet_ids_batch):
                if int(oks[index]) == 1:
                    sums[pet_id] += features[index]
                    counts[pet_id] += 1
        extracted = {}
        for pet_id in paths_by_pet:
            if counts[pet_id] > 0:
                extracted[pet_id] = (sums[pet_id] / counts[pet_id], counts[pet_id])
            else:
                extracted[pet_id] = (np.zeros(512, dtype=np.float32), 0)
        return extracted

    def stack_image_features(pet_ids, feature_map):
        vectors = np.zeros((len(pet_ids), 512), dtype=np.float32)
        missing = np.ones((len(pet_ids), 1), dtype=np.float32)
        for index, pet_id in enumerate(pet_ids):
            vector, count = feature_map[pet_id]
            if count > 0:
                vectors[index] = vector
                missing[index, 0] = 0.0
        return np.concatenate([vectors, missing], axis=1)

    def locate_resnet_cache():
        direct = WORKING / "resnet18_pet_features.npz"
        if direct.exists():
            return direct
        root = Path("/kaggle/input")
        if root.exists():
            found = sorted(root.rglob("resnet18_pet_features.npz"))
            if found:
                print(f"Кеш знайдено серед вхідних даних: {found[0]}")
                return found[0]
        return None

    def load_feature_map(cache_path):
        cached = np.load(cache_path, allow_pickle=False)
        cached_ids = [normalize_id(pet_id) for pet_id in cached["ids"].tolist()]
        counts = np.array(cached["counts"], copy=True)
        matrix = np.array(cached["features"], dtype=np.float32, copy=True)
        cached.close()
        feature_map = {}
        for index, pet_id in enumerate(cached_ids):
            feature_map[pet_id] = (matrix[index].copy(), int(counts[index]))
        return feature_map

    needed = set(train_ids) | set(test_ids)
    cache_path = locate_resnet_cache()
    feature_map = None
    if cache_path is not None:
        loaded = load_feature_map(cache_path)
        if set(loaded) >= needed:
            feature_map = loaded
            print(f"Ознаки ResNet завантажено з кешу: {cache_path}")
        else:
            print("Кеш ознак не покриває поточні ідентифікатори, рахую заново.")
    if feature_map is None:
        print("Кешу ResNet немає. Рахую заморожені ознаки, голову Exp2 не навчаю.")
        if "model_image_paths" not in globals():
            image_files = list_image_files(IMAGE_DIR)
            all_pet_ids = list(dict.fromkeys(list(train_ids) + list(test_ids)))
            full_image_paths = link_images(all_pet_ids, image_files)
            model_image_paths = {pet_id: paths[:MAX_IMAGES] for pet_id, paths in full_image_paths.items()}
            print("Файлів зображень:", len(image_files))
        try:
            weights = ResNet18_Weights.IMAGENET1K_V1
            backbone = resnet18(weights=weights)
            transform = weights.transforms()
        except Exception as exc:
            raise RuntimeError(
                "Не вдалося завантажити ваги ResNet18. Увімкніть Internet у налаштуваннях ноутбука."
            ) from exc
        backbone.fc = nn.Identity()
        backbone.to(device)
        backbone.eval()
        for parameter in backbone.parameters():
            parameter.requires_grad = False
        feature_map = extract_resnet_features(model_image_paths, backbone, transform)
        ids_array = np.array(list(feature_map.keys()))
        cache_out = WORKING / "resnet18_pet_features.npz"
        np.savez_compressed(
            cache_out,
            ids=ids_array,
            features=np.stack([feature_map[pet_id][0] for pet_id in ids_array]),
            counts=np.array([feature_map[pet_id][1] for pet_id in ids_array], dtype=np.int32),
        )
        print(f"Ознаки збережено: {cache_out}")
        del backbone
        if device.type == "cuda":
            torch.cuda.empty_cache()

    image_combined = stack_image_features(train_ids, feature_map)
    image_combined_test = stack_image_features(test_ids, feature_map)

print("Матриця фото train/test:", tuple(image_combined.shape), tuple(image_combined_test.shape))


In [ ]:
import copy

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, Dataset

SVD_DIM = 512
JOINT_HIDDEN = 128
JOINT_DROPOUT = 0.3
JOINT_LR = 1e-3
JOINT_WEIGHT_DECAY = 1e-4
JOINT_MAX_EPOCHS = 8
JOINT_PATIENCE = 2
JOINT_MIN_DELTA = 0.001
JOINT_BATCH = 64
IMAGE_VAL_QWK = 0.4247
TFIDF_VAL_QWK = 0.2730
DISTILBERT_VAL_QWK = 0.1629
EXP5_VAL_QWK = 0.4639
TFIDF_MAX_FEATURES = 20000
TFIDF_MIN_DF = 2


class JointDataset(Dataset):
    def __init__(self, features, labels=None):
        self.features = torch.tensor(np.asarray(features), dtype=torch.float32)
        self.labels = None if labels is None else torch.tensor(np.asarray(labels), dtype=torch.long)

    def __len__(self):
        return self.features.size(0)

    def __getitem__(self, index):
        if self.labels is None:
            return self.features[index]
        return self.features[index], self.labels[index]


class JointHead(nn.Module):
    def __init__(self, in_dim, hidden, n_classes, dropout=JOINT_DROPOUT):
        super().__init__()
        self.net = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, n_classes),
        )

    def forward(self, features):
        return self.net(features)


def decode_probabilities(probs, n_classes):
    class_index = np.arange(n_classes, dtype=np.float64)
    expectation = np.clip(np.rint(probs @ class_index), 0, n_classes - 1).astype(int)
    return {"argmax": probs.argmax(axis=1).astype(int), "expectation": expectation}


def fit_joint_head(features_train, labels_train, features_val, labels_val):
    torch.manual_seed(SEED)
    model = JointHead(features_train.shape[1], JOINT_HIDDEN, N_CLASSES).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=JOINT_LR, weight_decay=JOINT_WEIGHT_DECAY)
    criterion = nn.CrossEntropyLoss()
    train_loader = DataLoader(
        JointDataset(features_train, labels_train),
        batch_size=JOINT_BATCH,
        shuffle=True,
        generator=torch.Generator().manual_seed(SEED),
    )
    val_loader = DataLoader(JointDataset(features_val, labels_val), batch_size=256, shuffle=False)
    best_qwk = -1.0
    best_epoch = 1
    best_decoder = "argmax"
    best_state = copy.deepcopy(model.state_dict())
    wait = 0
    epochs_ran = 0
    for epoch in range(1, JOINT_MAX_EPOCHS + 1):
        epochs_ran = epoch
        model.train()
        total_loss, seen = 0.0, 0
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            optimizer.zero_grad(set_to_none=True)
            logits = model(batch_x)
            loss = criterion(logits, batch_y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            total_loss += loss.item() * batch_y.size(0)
            seen += batch_y.size(0)
        model.eval()
        prob_chunks, true_chunks = [], []
        val_loss, val_seen = 0.0, 0
        with torch.no_grad():
            for batch_x, batch_y in val_loader:
                batch_x, batch_y = batch_x.to(device), batch_y.to(device)
                logits = model(batch_x)
                val_loss += criterion(logits, batch_y).item() * batch_y.size(0)
                val_seen += batch_y.size(0)
                prob_chunks.append(torch.softmax(logits, dim=1).cpu().numpy())
                true_chunks.append(batch_y.cpu().numpy())
        probs = np.concatenate(prob_chunks)
        y_true = np.concatenate(true_chunks)
        decoded = decode_probabilities(probs, N_CLASSES)
        qwk_argmax = quadratic_weighted_kappa(y_true, decoded["argmax"])
        qwk_expectation = quadratic_weighted_kappa(y_true, decoded["expectation"])
        epoch_decoder = "expectation" if qwk_expectation > qwk_argmax else "argmax"
        epoch_qwk = qwk_expectation if epoch_decoder == "expectation" else qwk_argmax
        print(
            f"Епоха {epoch:02d}/{JOINT_MAX_EPOCHS} | train loss {total_loss / max(seen, 1):.4f} | "
            f"val loss {val_loss / max(val_seen, 1):.4f} | "
            f"QWK argmax {qwk_argmax:.4f} | QWK expectation {qwk_expectation:.4f}"
        )
        if epoch_qwk > best_qwk + JOINT_MIN_DELTA:
            best_qwk = epoch_qwk
            best_epoch = epoch
            best_decoder = epoch_decoder
            best_state = copy.deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1
            if wait >= JOINT_PATIENCE:
                break
    model.load_state_dict(best_state)
    print(f"Зупинка після епохи {epochs_ran}. Найкраща епоха {best_epoch}, QWK {best_qwk:.4f} ({best_decoder}).")
    return model, best_qwk, best_epoch, best_decoder, epochs_ran


def predict_joint(model, features):
    model.eval()
    loader = DataLoader(JointDataset(features), batch_size=256, shuffle=False)
    chunks = []
    with torch.no_grad():
        for batch_x in loader:
            logits = model(batch_x.to(device))
            chunks.append(torch.softmax(logits, dim=1).cpu().numpy())
    return np.concatenate(chunks)


def text_svd(frame_fit, frames_transform):
    vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=TFIDF_MIN_DF, max_features=TFIDF_MAX_FEATURES)
    sparse_fit = vectorizer.fit_transform(frame_fit)
    svd_dim = min(SVD_DIM, sparse_fit.shape[1] - 1, sparse_fit.shape[0] - 1)
    svd = TruncatedSVD(n_components=svd_dim, random_state=SEED)
    dense_fit = svd.fit_transform(sparse_fit)
    transformed = [svd.transform(vectorizer.transform(frame)) for frame in frames_transform]
    explained = float(svd.explained_variance_ratio_.sum())
    return dense_fit, transformed, explained, sparse_fit.shape[1], svd_dim


text_tr, (text_va,), explained, n_tfidf, svd_dim = text_svd(
    train_df.iloc[idx_tr]["text_clean"],
    [train_df.iloc[idx_va]["text_clean"]],
)
print(
    f"TF-IDF ознак {n_tfidf}, компонент SVD {svd_dim}, "
    f"пояснена дисперсія {explained:.3f}. У Exp5 при 128 компонентах було 0.227. Ваг класів немає."
)
scaler = StandardScaler()
fused_tr = scaler.fit_transform(np.concatenate([image_combined[idx_tr], text_tr], axis=1)).astype(np.float32)
fused_va = scaler.transform(np.concatenate([image_combined[idx_va], text_va], axis=1)).astype(np.float32)
print("Спільна матриця train/val:", fused_tr.shape, fused_va.shape)

joint_model, joint_qwk, joint_best_epoch, joint_decoder, joint_epochs = fit_joint_head(fused_tr, y_tr, fused_va, y_va)
val_probs = predict_joint(joint_model, fused_va)
val_decoded = decode_probabilities(val_probs, N_CLASSES)
val_scores = {name: quadratic_weighted_kappa(y_va, pred) for name, pred in val_decoded.items()}
chosen_pred = val_decoded[joint_decoder]
print(
    f"Exp6 QWK argmax {val_scores['argmax']:.4f}, expectation {val_scores['expectation']:.4f}. "
    f"Обрано {joint_decoder}, accuracy {accuracy_score(y_va, chosen_pred):.4f}."
)
print("Прогнози на валідації:")
print(pd.Series(chosen_pred).map(IDX_TO_LABEL).value_counts().sort_index())
print(
    f"Порівняння: TF-IDF {TFIDF_VAL_QWK:.4f}, фото {IMAGE_VAL_QWK:.4f}, "
    f"DistilBERT {DISTILBERT_VAL_QWK:.4f}, Exp5 {EXP5_VAL_QWK:.4f}, "
f"SVD-512 {val_scores[joint_decoder]:.4f}."
)

exp6_name = "Exp6. ResNet + SVD 512(TF-IDF), прихований шар"
exp6_notes = (
    f"без ваг класів; svd {svd_dim}; пояснена дисперсія {explained:.3f}; "
    f"argmax {val_scores['argmax']:.4f}; expectation {val_scores['expectation']:.4f}"
)
if "log_experiment" in globals():
    log_experiment(exp6_name, val_scores[joint_decoder], joint_decoder, joint_epochs, joint_best_epoch, exp6_notes)
else:
    print(exp6_name, round(val_scores[joint_decoder], 4), joint_decoder, "епох", joint_epochs, "найкраща", joint_best_epoch)

if val_scores[joint_decoder] > EXP5_VAL_QWK + JOINT_MIN_DELTA:
    print("SVD-512 обігнав Exp5 на валідації. Навчаю голову на повному train і пишу submission.csv.")
    text_all, (text_test,), _, _, _ = text_svd(
        train_df["text_clean"],
        [test_df["text_clean"]],
    )
    final_scaler = StandardScaler()
    fused_all = final_scaler.fit_transform(np.concatenate([image_combined, text_all], axis=1)).astype(np.float32)
    fused_test = final_scaler.transform(np.concatenate([image_combined_test, text_test], axis=1)).astype(np.float32)
    torch.manual_seed(SEED)
    final_joint = JointHead(fused_all.shape[1], JOINT_HIDDEN, N_CLASSES).to(device)
    final_optimizer = torch.optim.AdamW(final_joint.parameters(), lr=JOINT_LR, weight_decay=JOINT_WEIGHT_DECAY)
    final_criterion = nn.CrossEntropyLoss()
    final_loader = DataLoader(
        JointDataset(fused_all, y_all),
        batch_size=JOINT_BATCH,
        shuffle=True,
        generator=torch.Generator().manual_seed(SEED),
    )
    final_epochs = max(int(joint_best_epoch), 1)
    final_joint.train()
    for epoch in range(1, final_epochs + 1):
        total_loss, seen = 0.0, 0
        for batch_x, batch_y in final_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            final_optimizer.zero_grad(set_to_none=True)
            logits = final_joint(batch_x)
            loss = final_criterion(logits, batch_y)
            loss.backward()
            nn.utils.clip_grad_norm_(final_joint.parameters(), 5.0)
            final_optimizer.step()
            total_loss += loss.item() * batch_y.size(0)
            seen += batch_y.size(0)
        print(f"Повне навчання спільної голови, епоха {epoch}/{final_epochs}, train loss {total_loss / max(seen, 1):.4f}")
    test_probs = predict_joint(final_joint, fused_test)
    test_idx = decode_probabilities(test_probs, N_CLASSES)[joint_decoder]
    submission = pd.DataFrame({
        "PetID": [normalize_id(pet_id) for pet_id in test_ids],
        "AdoptionSpeed": [IDX_TO_LABEL[int(index)] for index in test_idx],
    })
    submission = submission.drop_duplicates("PetID", keep="first")
    submission["AdoptionSpeed"] = submission["AdoptionSpeed"].astype(int)
    submission = submission[["PetID", "AdoptionSpeed"]]
    submission.to_csv(WORKING / "submission.csv", index=False)
    print(f"Збережено submission.csv, рядків {len(submission)}.")
    print(submission["AdoptionSpeed"].value_counts().sort_index())
else:
    print(
        "Новий submission.csv не записано: val QWK не вищий за Exp5 більше ніж на 0.001. "
        "Публічний файл зі score 0.6676 лишається чинним."
    )


### Висновки експерименту 6

512 компонент SVD пояснюють 0.449 дисперсії TF-IDF проти 0.227 у 128 компонент. Спільна матриця має 1025 ознак. На збережених вагах 5-ї епохи val QWK 0.473 за argmax і 0.424 за очікуванням класу, accuracy 0.470. По епохах argmax ішов 0.421, 0.443, 0.430, 0.452, 0.473, 0.463, 0.446. Зупинка після 7-ї епохи, найкраща — 5-та. Train loss за цей час упав з 1.259 до 0.815, а val loss був найнижчий на 2–3 епохах (1.184) і до 7-ї виріс до 1.260.

На валідації це на 0.009 вище за Exp5 (0.464). Публічний score цього файлу — 0.6524, нижче за 0.6676 в Exp5. Приріст 0.009 на 1287 рядках на публічний тест не перенісся. Accuracy теж нижча, ніж в Exp5: 0.470 проти 0.495.

На валідації клас 1 отримав 188 прогнозів із 1287 (14.6% за частки 18.6% у train), клас 2 — 396 (30.8% за 27.6%), клас 3 — 211 (16.4% за 20.6%), клас 4 — 492 (38.2% за 33.2%). На тесті: клас 1 — 256 (13.5%), клас 2 — 605 (32.0%), клас 3 — 245 (13.0%), клас 4 — 785 (41.5%).

Повне навчання йшло 5 епох, train loss на п’ятій — 0.937. Файл на 1891 рядок записано і відправлено.

Додаткові компоненти тексту вичерпали себе: дисперсія зросла вдвічі, публічний score упав. Кращий публічний файл лишається Exp5 зі score 0.6676. Наступний блок змінює фото, не ширину SVD: до 8 знімків на тварину замість 3, текст знову 128 компонент, як у Exp5. Новий `submission.csv` має сенс лише коли val QWK вищий за 0.4639 більше ніж на 0.02.


## 14. Експеримент 7. До 8 фото і SVD на 128 компонент

Exp6 підняв val QWK із 0.464 до 0.473, а публічний score опустив із 0.6676 до 0.6524. Ширший текст далі не ростимо. Кращий публічний файл — Exp5: заморожений ResNet по 3 фото і TF-IDF, стиснений до 128 компонент. У списку сабмітів на Kaggle обраним лишається саме він.

Цей блок змінює лише фото. Медіана в train — 4 знімки, стеля попередніх моделей — 3, тож частина ракурсів у середнє не входить. Тут середнє бере до 8 файлів на тварину. Текст, шкалювання і голова ті самі, що в Exp5: SVD на 128, прихований шар на 128, без ваг класів. Кеш пишеться в `resnet18_pet_features_8.npz` і не затирає кеш на 3 фото.

Як запускати на Kaggle: **не натискайте Run All**. Не перезапускайте код-комірки навчання в розділах 12 і 13: вони знову перезапишуть `submission.csv`. Виконайте лише цей розділ зверху вниз. Якщо ядро скинуте, перша комірка відновить таблиці й опис. Друга порахує ознаки восьми фото замороженим ResNet; для цього потрібні GPU і Internet. Голови попередніх експериментів вона не навчає.

Поріг сабміту вищий, ніж раніше: val QWK має перевищити 0.4639 більше ніж на 0.02. Приріст 0.009 у Exp6 на публічному тесті не підтвердився.


In [ ]:
import os
import random
import re
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import cohen_kappa_score
from sklearn.model_selection import train_test_split

SEED = 42
VAL_SIZE = 0.2
MAX_IMAGES = 3
FEATURE_BATCH_SIZE = 64
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
DATA_ROOT = Path("/kaggle/input/competitions/deep-learning-for-computer-vision-and-nlp-2026-10")
IMAGE_DIR = DATA_ROOT / "images"
WORKING = Path("/kaggle/working")
WORKING.mkdir(parents=True, exist_ok=True)
NUM_WORKERS = 2 if os.name != "nt" else 0
if "device" not in globals():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

needs_tables = "train_df" not in globals() or "idx_tr" not in globals() or "y_all" not in globals()
if not needs_tables:
    print(f"Таблиці вже в пам'яті: train {len(train_df)}, val {len(idx_va)}. Повторно не читаю.")
else:
    TRAIN_CSV = DATA_ROOT / "train.csv"
    TEST_CSV = DATA_ROOT / "test.csv"
    SAMPLE_CSV = DATA_ROOT / "sample_submission.csv"

    def find_column(columns, options):
        lookup = {str(column).lower(): column for column in columns}
        for option in options:
            if option.lower() in lookup:
                return lookup[option.lower()]
        return None

    def normalize_id(value):
        if pd.isna(value):
            return ""
        text = str(value).strip()
        if text.endswith(".0"):
            text = text[:-2]
        return text

    def read_table(path):
        frame = pd.read_csv(path)
        frame.columns = [str(column).strip() for column in frame.columns]
        return frame

    def quadratic_weighted_kappa(y_true, y_pred):
        y_true = np.asarray(y_true).astype(int)
        y_pred = np.asarray(y_pred).astype(int)
        if len(y_true) == 0:
            return 0.0
        score = cohen_kappa_score(y_true, y_pred, weights="quadratic")
        if score != score:
            return 0.0
        return float(score)

    train_df = read_table(TRAIN_CSV)
    test_df = read_table(TEST_CSV)
    sample_df = read_table(SAMPLE_CSV)
    id_col = find_column(train_df.columns, ["PetID", "pet_id", "id"])
    target_col = find_column(train_df.columns, ["AdoptionSpeed", "adoption_speed", "label", "target"])
    text_col = find_column(train_df.columns, ["Description", "description", "desc", "text", "comments"])
    test_id_col = find_column(test_df.columns, ["PetID", "pet_id", "id"]) or id_col
    train_df[id_col] = train_df[id_col].map(normalize_id)
    test_df[test_id_col] = test_df[test_id_col].map(normalize_id)
    if int(train_df[id_col].duplicated().sum()):
        train_df = train_df.drop_duplicates(id_col, keep="first")
    train_df = train_df.reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)
    if target_col in test_df.columns:
        test_df = test_df.drop(columns=[target_col])
    test_text_col = find_column(test_df.columns, ["Description", "description", "desc", "text", "comments"])
    if text_col is None:
        text_col = "Description"
        train_df[text_col] = ""
        test_df[text_col] = ""
    elif test_text_col is None:
        test_df[text_col] = ""
    elif test_text_col != text_col:
        test_df[text_col] = test_df[test_text_col]
    raw_target = train_df[target_col]
    if pd.api.types.is_numeric_dtype(raw_target):
        numeric_target = raw_target.astype(float)
        if np.allclose(numeric_target.to_numpy(), np.rint(numeric_target.to_numpy())):
            train_df[target_col] = numeric_target.round().astype(int)
    classes = sorted(train_df[target_col].dropna().unique().tolist())
    LABEL_TO_IDX = {label: index for index, label in enumerate(classes)}
    IDX_TO_LABEL = {index: label for label, index in LABEL_TO_IDX.items()}
    N_CLASSES = len(classes)
    train_df = train_df.dropna(subset=[target_col]).reset_index(drop=True)
    y_all = train_df[target_col].map(LABEL_TO_IDX).to_numpy(dtype=int)
    idx_tr, idx_va = train_test_split(
        np.arange(len(train_df)), test_size=VAL_SIZE, random_state=SEED, stratify=y_all
    )
    y_tr = y_all[idx_tr]
    y_va = y_all[idx_va]
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    print(f"Таблиці прочитано заново: train {len(train_df)}, val {len(idx_va)}, test {len(test_df)}.")

if "train_ids" not in globals():
    train_ids = train_df[id_col].tolist()
if "test_ids" not in globals():
    test_ids = test_df[test_id_col].tolist()

if len(idx_va) != 1287:
    print(f"Увага: у валідації {len(idx_va)} рядків, у минулому прогоні було 1287.")
else:
    print("Розбиття збігається з минулим прогоном: 1287 валідаційних рядків.")

if "text_clean" in train_df.columns and "text_clean" in test_df.columns:
    print("Очищений опис уже є, повторно не чищу.")
else:
    NEGATION_TOKENS = {
        "no", "not", "nor", "never", "none", "nobody", "nothing",
        "neither", "nowhere", "without", "cannot",
    }
    CONTRACTIONS = {
        "ain't": "is not",
        "aren't": "are not",
        "can't": "cannot",
        "couldn't": "could not",
        "didn't": "did not",
        "doesn't": "does not",
        "don't": "do not",
        "hadn't": "had not",
        "hasn't": "has not",
        "haven't": "have not",
        "he's": "he is",
        "here's": "here is",
        "how's": "how is",
        "i'm": "i am",
        "i've": "i have",
        "isn't": "is not",
        "it's": "it is",
        "let's": "let us",
        "mustn't": "must not",
        "she's": "she is",
        "that's": "that is",
        "there's": "there is",
        "they're": "they are",
        "they've": "they have",
        "we're": "we are",
        "we've": "we have",
        "weren't": "were not",
        "what's": "what is",
        "where's": "where is",
        "who's": "who is",
        "won't": "will not",
        "wouldn't": "would not",
        "you're": "you are",
        "you've": "you have",
    }
    CONTRACTION_RE = re.compile(
        r"\b(" + "|".join(sorted(map(re.escape, CONTRACTIONS), key=len, reverse=True)) + r")\b"
    )
    FALLBACK_STOPWORDS = set(
        "a about above after again against all am an and any are as at be because been "
        "before being below between both but by can did do does doing down during each "
        "few for from further had has have having he her here hers herself him himself "
        "his how i if in into is it its itself just me more most my myself of off on "
        "once only or other our ours ourselves out over own same she should so some "
        "such than that the their theirs them themselves then there these they this "
        "those through to too under until up very was we were what when where which "
        "while who whom why will with you your yours yourself yourselves".split()
    )

    def load_stopwords():
        try:
            import nltk
            from nltk.corpus import stopwords

            nltk.download("stopwords", quiet=True)
            words = set(stopwords.words("english"))
            print("Стоп-слова: NLTK, english")
        except Exception as exc:
            print(f"Стоп-слова NLTK недоступні ({type(exc).__name__}: {exc}). Використовую вбудований список.")
            words = set(FALLBACK_STOPWORDS)
        return words - NEGATION_TOKENS

    STOPWORDS = load_stopwords()

    def clean_text(value):
        if not isinstance(value, str):
            return ""
        text = value.lower().replace("’", "'").replace("`", "'")
        text = CONTRACTION_RE.sub(lambda match: CONTRACTIONS[match.group(0)], text)
        text = re.sub(r"https?://\S+|www\.\S+", " ", text)
        text = re.sub(r"[^a-z\s]", " ", text)
        tokens = [token for token in text.split() if len(token) > 1 and token not in STOPWORDS]
        return " ".join(tokens)

    train_df["text_clean"] = train_df[text_col].fillna("").astype(str).map(clean_text)
    test_df["text_clean"] = test_df[text_col].fillna("").astype(str).map(clean_text)
    print("Порожніх описів після очищення:", float(train_df["text_clean"].eq("").mean()))

print("Пристрій:", device)


In [ ]:
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision.models import ResNet18_Weights, resnet18
from tqdm.auto import tqdm

PHOTO_LIMIT = 8
reuse_images = (
    globals().get("IMAGE_FEATURE_LIMIT") == PHOTO_LIMIT
    and "image_combined" in globals()
    and "image_combined_test" in globals()
    and len(image_combined) == len(train_df)
    and len(image_combined_test) == len(test_ids)
)
if reuse_images:
    print(
        "Вектори ResNet по 8 фото вже в пам'яті:",
        tuple(image_combined.shape),
        tuple(image_combined_test.shape),
        ". Заново не рахую.",
    )
else:
    def list_image_files(image_dir):
        files = []
        for path in Path(image_dir).rglob("*"):
            if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS:
                files.append(path)
        return files

    def link_images(pet_ids, files):
        pet_set = set(pet_ids)
        grouped = {pet_id: [] for pet_id in pet_set}
        for path in files:
            stem = path.stem
            pet_id = None
            order = 0
            if stem in pet_set:
                pet_id = stem
            else:
                for separator in ("-", "_"):
                    if separator not in stem:
                        continue
                    base, rest = stem.rsplit(separator, 1)
                    if base in pet_set:
                        pet_id = base
                        order = int(rest) if rest.isdigit() else 10**6
                        break
            if pet_id is not None:
                grouped[pet_id].append((order, path))
        linked = {}
        for pet_id, items in grouped.items():
            items.sort(key=lambda item: (item[0], item[1].name))
            linked[pet_id] = [path for _, path in items]
        return linked

    class PetImageDataset(Dataset):
        def __init__(self, records, transform):
            self.records = records
            self.transform = transform

        def __len__(self):
            return len(self.records)

        def __getitem__(self, index):
            pet_id, path = self.records[index]
            try:
                with Image.open(path) as image:
                    tensor = self.transform(image.convert("RGB"))
                ok = 1
            except Exception:
                tensor = torch.zeros(3, 224, 224)
                ok = 0
            return tensor, ok, pet_id

    def extract_resnet_features(paths_by_pet, backbone, transform):
        records = [(pet_id, path) for pet_id, paths in paths_by_pet.items() for path in paths]
        sums = {pet_id: np.zeros(512, dtype=np.float32) for pet_id in paths_by_pet}
        counts = {pet_id: 0 for pet_id in paths_by_pet}
        if not records:
            return {pet_id: (sums[pet_id], 0) for pet_id in paths_by_pet}
        loader = DataLoader(
            PetImageDataset(records, transform),
            batch_size=FEATURE_BATCH_SIZE,
            shuffle=False,
            num_workers=NUM_WORKERS,
        )
        backbone.eval()
        for tensors, oks, pet_ids_batch in tqdm(loader, desc="ResNet18"):
            features = backbone(tensors.to(device)).detach().cpu().numpy()
            oks = oks.numpy()
            for index, pet_id in enumerate(pet_ids_batch):
                if int(oks[index]) == 1:
                    sums[pet_id] += features[index]
                    counts[pet_id] += 1
        extracted = {}
        for pet_id in paths_by_pet:
            if counts[pet_id] > 0:
                extracted[pet_id] = (sums[pet_id] / counts[pet_id], counts[pet_id])
            else:
                extracted[pet_id] = (np.zeros(512, dtype=np.float32), 0)
        return extracted

    def stack_image_features(pet_ids, feature_map):
        vectors = np.zeros((len(pet_ids), 512), dtype=np.float32)
        missing = np.ones((len(pet_ids), 1), dtype=np.float32)
        for index, pet_id in enumerate(pet_ids):
            vector, count = feature_map[pet_id]
            if count > 0:
                vectors[index] = vector
                missing[index, 0] = 0.0
        return np.concatenate([vectors, missing], axis=1)

    def locate_resnet_cache():
        direct = WORKING / "resnet18_pet_features_8.npz"
        if direct.exists():
            return direct
        root = Path("/kaggle/input")
        if root.exists():
            found = sorted(root.rglob("resnet18_pet_features_8.npz"))
            if found:
                print(f"Кеш знайдено серед вхідних даних: {found[0]}")
                return found[0]
        return None

    def load_feature_map(cache_path):
        cached = np.load(cache_path, allow_pickle=False)
        cached_ids = [normalize_id(pet_id) for pet_id in cached["ids"].tolist()]
        counts = np.array(cached["counts"], copy=True)
        matrix = np.array(cached["features"], dtype=np.float32, copy=True)
        cached.close()
        feature_map = {}
        for index, pet_id in enumerate(cached_ids):
            feature_map[pet_id] = (matrix[index].copy(), int(counts[index]))
        return feature_map

    needed = set(train_ids) | set(test_ids)
    cache_path = locate_resnet_cache()
    feature_map = None
    if cache_path is not None:
        loaded = load_feature_map(cache_path)
        if set(loaded) >= needed:
            feature_map = loaded
            print(f"Ознаки ResNet завантажено з кешу: {cache_path}")
        else:
            print("Кеш ознак не покриває поточні ідентифікатори, рахую заново.")
    if feature_map is None:
        print("Кешу на 8 фото немає. Рахую заморожені ознаки, попередні голови не навчаю.")
        image_files = list_image_files(IMAGE_DIR)
        all_pet_ids = list(dict.fromkeys(list(train_ids) + list(test_ids)))
        full_image_paths = link_images(all_pet_ids, image_files)
        model_image_paths = {pet_id: paths[:PHOTO_LIMIT] for pet_id, paths in full_image_paths.items()}
        photo_counts = np.array([len(full_image_paths[pet_id]) for pet_id in train_ids])
        used_counts = np.array([len(model_image_paths[pet_id]) for pet_id in train_ids])
        print("Файлів зображень:", len(image_files))
        print(
            f"Фото на тварину в train: median {float(np.median(photo_counts)):.0f}, "
            f"max {int(photo_counts.max())}. У середнє входить до {PHOTO_LIMIT}: "
            f"median {float(np.median(used_counts)):.0f}, max {int(used_counts.max())}."
        )
        try:
            weights = ResNet18_Weights.IMAGENET1K_V1
            backbone = resnet18(weights=weights)
            transform = weights.transforms()
        except Exception as exc:
            raise RuntimeError(
                "Не вдалося завантажити ваги ResNet18. Увімкніть Internet у налаштуваннях ноутбука."
            ) from exc
        backbone.fc = nn.Identity()
        backbone.to(device)
        backbone.eval()
        for parameter in backbone.parameters():
            parameter.requires_grad = False
        feature_map = extract_resnet_features(model_image_paths, backbone, transform)
        ids_array = np.array(list(feature_map.keys()))
        cache_out = WORKING / "resnet18_pet_features_8.npz"
        np.savez_compressed(
            cache_out,
            ids=ids_array,
            features=np.stack([feature_map[pet_id][0] for pet_id in ids_array]),
            counts=np.array([feature_map[pet_id][1] for pet_id in ids_array], dtype=np.int32),
        )
        print(f"Ознаки збережено: {cache_out}")
        del backbone
        if device.type == "cuda":
            torch.cuda.empty_cache()

    image_combined = stack_image_features(train_ids, feature_map)
    image_combined_test = stack_image_features(test_ids, feature_map)

IMAGE_FEATURE_LIMIT = PHOTO_LIMIT
print("Матриця фото train/test:", tuple(image_combined.shape), tuple(image_combined_test.shape), f"стеля {PHOTO_LIMIT}")


In [ ]:
import copy

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, Dataset

SVD_DIM = 128
JOINT_HIDDEN = 128
JOINT_DROPOUT = 0.3
JOINT_LR = 1e-3
JOINT_WEIGHT_DECAY = 1e-4
JOINT_MAX_EPOCHS = 8
JOINT_PATIENCE = 2
JOINT_MIN_DELTA = 0.001
JOINT_BATCH = 64
IMAGE_VAL_QWK = 0.4247
TFIDF_VAL_QWK = 0.2730
DISTILBERT_VAL_QWK = 0.1629
EXP5_VAL_QWK = 0.4639
EXP6_VAL_QWK = 0.4727
SUBMISSION_MARGIN = 0.02
TFIDF_MAX_FEATURES = 20000
TFIDF_MIN_DF = 2


class JointDataset(Dataset):
    def __init__(self, features, labels=None):
        self.features = torch.tensor(np.asarray(features), dtype=torch.float32)
        self.labels = None if labels is None else torch.tensor(np.asarray(labels), dtype=torch.long)

    def __len__(self):
        return self.features.size(0)

    def __getitem__(self, index):
        if self.labels is None:
            return self.features[index]
        return self.features[index], self.labels[index]


class JointHead(nn.Module):
    def __init__(self, in_dim, hidden, n_classes, dropout=JOINT_DROPOUT):
        super().__init__()
        self.net = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, n_classes),
        )

    def forward(self, features):
        return self.net(features)


def decode_probabilities(probs, n_classes):
    class_index = np.arange(n_classes, dtype=np.float64)
    expectation = np.clip(np.rint(probs @ class_index), 0, n_classes - 1).astype(int)
    return {"argmax": probs.argmax(axis=1).astype(int), "expectation": expectation}


def fit_joint_head(features_train, labels_train, features_val, labels_val):
    torch.manual_seed(SEED)
    model = JointHead(features_train.shape[1], JOINT_HIDDEN, N_CLASSES).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=JOINT_LR, weight_decay=JOINT_WEIGHT_DECAY)
    criterion = nn.CrossEntropyLoss()
    train_loader = DataLoader(
        JointDataset(features_train, labels_train),
        batch_size=JOINT_BATCH,
        shuffle=True,
        generator=torch.Generator().manual_seed(SEED),
    )
    val_loader = DataLoader(JointDataset(features_val, labels_val), batch_size=256, shuffle=False)
    best_qwk = -1.0
    best_epoch = 1
    best_decoder = "argmax"
    best_state = copy.deepcopy(model.state_dict())
    wait = 0
    epochs_ran = 0
    for epoch in range(1, JOINT_MAX_EPOCHS + 1):
        epochs_ran = epoch
        model.train()
        total_loss, seen = 0.0, 0
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            optimizer.zero_grad(set_to_none=True)
            logits = model(batch_x)
            loss = criterion(logits, batch_y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            total_loss += loss.item() * batch_y.size(0)
            seen += batch_y.size(0)
        model.eval()
        prob_chunks, true_chunks = [], []
        val_loss, val_seen = 0.0, 0
        with torch.no_grad():
            for batch_x, batch_y in val_loader:
                batch_x, batch_y = batch_x.to(device), batch_y.to(device)
                logits = model(batch_x)
                val_loss += criterion(logits, batch_y).item() * batch_y.size(0)
                val_seen += batch_y.size(0)
                prob_chunks.append(torch.softmax(logits, dim=1).cpu().numpy())
                true_chunks.append(batch_y.cpu().numpy())
        probs = np.concatenate(prob_chunks)
        y_true = np.concatenate(true_chunks)
        decoded = decode_probabilities(probs, N_CLASSES)
        qwk_argmax = quadratic_weighted_kappa(y_true, decoded["argmax"])
        qwk_expectation = quadratic_weighted_kappa(y_true, decoded["expectation"])
        epoch_decoder = "expectation" if qwk_expectation > qwk_argmax else "argmax"
        epoch_qwk = qwk_expectation if epoch_decoder == "expectation" else qwk_argmax
        print(
            f"Епоха {epoch:02d}/{JOINT_MAX_EPOCHS} | train loss {total_loss / max(seen, 1):.4f} | "
            f"val loss {val_loss / max(val_seen, 1):.4f} | "
            f"QWK argmax {qwk_argmax:.4f} | QWK expectation {qwk_expectation:.4f}"
        )
        if epoch_qwk > best_qwk + JOINT_MIN_DELTA:
            best_qwk = epoch_qwk
            best_epoch = epoch
            best_decoder = epoch_decoder
            best_state = copy.deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1
            if wait >= JOINT_PATIENCE:
                break
    model.load_state_dict(best_state)
    print(f"Зупинка після епохи {epochs_ran}. Найкраща епоха {best_epoch}, QWK {best_qwk:.4f} ({best_decoder}).")
    return model, best_qwk, best_epoch, best_decoder, epochs_ran


def predict_joint(model, features):
    model.eval()
    loader = DataLoader(JointDataset(features), batch_size=256, shuffle=False)
    chunks = []
    with torch.no_grad():
        for batch_x in loader:
            logits = model(batch_x.to(device))
            chunks.append(torch.softmax(logits, dim=1).cpu().numpy())
    return np.concatenate(chunks)


def text_svd(frame_fit, frames_transform):
    vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=TFIDF_MIN_DF, max_features=TFIDF_MAX_FEATURES)
    sparse_fit = vectorizer.fit_transform(frame_fit)
    svd_dim = min(SVD_DIM, sparse_fit.shape[1] - 1, sparse_fit.shape[0] - 1)
    svd = TruncatedSVD(n_components=svd_dim, random_state=SEED)
    dense_fit = svd.fit_transform(sparse_fit)
    transformed = [svd.transform(vectorizer.transform(frame)) for frame in frames_transform]
    explained = float(svd.explained_variance_ratio_.sum())
    return dense_fit, transformed, explained, sparse_fit.shape[1], svd_dim


text_tr, (text_va,), explained, n_tfidf, svd_dim = text_svd(
    train_df.iloc[idx_tr]["text_clean"],
    [train_df.iloc[idx_va]["text_clean"]],
)
print(
    f"TF-IDF ознак {n_tfidf}, компонент SVD {svd_dim}, "
    f"пояснена дисперсія {explained:.3f}. Ваг класів немає."
)
scaler = StandardScaler()
fused_tr = scaler.fit_transform(np.concatenate([image_combined[idx_tr], text_tr], axis=1)).astype(np.float32)
fused_va = scaler.transform(np.concatenate([image_combined[idx_va], text_va], axis=1)).astype(np.float32)
print("Спільна матриця train/val:", fused_tr.shape, fused_va.shape)

joint_model, joint_qwk, joint_best_epoch, joint_decoder, joint_epochs = fit_joint_head(fused_tr, y_tr, fused_va, y_va)
val_probs = predict_joint(joint_model, fused_va)
val_decoded = decode_probabilities(val_probs, N_CLASSES)
val_scores = {name: quadratic_weighted_kappa(y_va, pred) for name, pred in val_decoded.items()}
chosen_pred = val_decoded[joint_decoder]
print(
    f"Exp7 QWK argmax {val_scores['argmax']:.4f}, expectation {val_scores['expectation']:.4f}. "
    f"Обрано {joint_decoder}, accuracy {accuracy_score(y_va, chosen_pred):.4f}."
)
print("Прогнози на валідації:")
print(pd.Series(chosen_pred).map(IDX_TO_LABEL).value_counts().sort_index())
print(
    f"Порівняння: TF-IDF {TFIDF_VAL_QWK:.4f}, фото {IMAGE_VAL_QWK:.4f}, "
    f"DistilBERT {DISTILBERT_VAL_QWK:.4f}, Exp5 {EXP5_VAL_QWK:.4f}, Exp6 {EXP6_VAL_QWK:.4f}, "
f"до 8 фото {val_scores[joint_decoder]:.4f}."
)

exp7_name = "Exp7. ResNet до 8 фото + SVD 128, прихований шар"
exp7_notes = (
    f"без ваг класів; svd {svd_dim}; пояснена дисперсія {explained:.3f}; "
    f"argmax {val_scores['argmax']:.4f}; expectation {val_scores['expectation']:.4f}"
)
if "log_experiment" in globals():
    log_experiment(exp7_name, val_scores[joint_decoder], joint_decoder, joint_epochs, joint_best_epoch, exp7_notes)
else:
    print(exp7_name, round(val_scores[joint_decoder], 4), joint_decoder, "епох", joint_epochs, "найкраща", joint_best_epoch)

if val_scores[joint_decoder] > EXP5_VAL_QWK + SUBMISSION_MARGIN:
    print("Вісім фото обігнали Exp5 на валідації більше ніж на 0.02. Навчаю голову на повному train і пишу submission.csv.")
    text_all, (text_test,), _, _, _ = text_svd(
        train_df["text_clean"],
        [test_df["text_clean"]],
    )
    final_scaler = StandardScaler()
    fused_all = final_scaler.fit_transform(np.concatenate([image_combined, text_all], axis=1)).astype(np.float32)
    fused_test = final_scaler.transform(np.concatenate([image_combined_test, text_test], axis=1)).astype(np.float32)
    torch.manual_seed(SEED)
    final_joint = JointHead(fused_all.shape[1], JOINT_HIDDEN, N_CLASSES).to(device)
    final_optimizer = torch.optim.AdamW(final_joint.parameters(), lr=JOINT_LR, weight_decay=JOINT_WEIGHT_DECAY)
    final_criterion = nn.CrossEntropyLoss()
    final_loader = DataLoader(
        JointDataset(fused_all, y_all),
        batch_size=JOINT_BATCH,
        shuffle=True,
        generator=torch.Generator().manual_seed(SEED),
    )
    final_epochs = max(int(joint_best_epoch), 1)
    final_joint.train()
    for epoch in range(1, final_epochs + 1):
        total_loss, seen = 0.0, 0
        for batch_x, batch_y in final_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            final_optimizer.zero_grad(set_to_none=True)
            logits = final_joint(batch_x)
            loss = final_criterion(logits, batch_y)
            loss.backward()
            nn.utils.clip_grad_norm_(final_joint.parameters(), 5.0)
            final_optimizer.step()
            total_loss += loss.item() * batch_y.size(0)
            seen += batch_y.size(0)
        print(f"Повне навчання спільної голови, епоха {epoch}/{final_epochs}, train loss {total_loss / max(seen, 1):.4f}")
    test_probs = predict_joint(final_joint, fused_test)
    test_idx = decode_probabilities(test_probs, N_CLASSES)[joint_decoder]
    submission = pd.DataFrame({
        "PetID": [normalize_id(pet_id) for pet_id in test_ids],
        "AdoptionSpeed": [IDX_TO_LABEL[int(index)] for index in test_idx],
    })
    submission = submission.drop_duplicates("PetID", keep="first")
    submission["AdoptionSpeed"] = submission["AdoptionSpeed"].astype(int)
    submission = submission[["PetID", "AdoptionSpeed"]]
    submission.to_csv(WORKING / "submission.csv", index=False)
    print(f"Збережено submission.csv, рядків {len(submission)}.")
    print(submission["AdoptionSpeed"].value_counts().sort_index())
else:
    print(
        "Новий submission.csv не записано: val QWK не вищий за Exp5 більше ніж на 0.02. "
        "Публічний файл зі score 0.6676 лишається чинним."
    )


### Висновки експерименту 7

До 8 фото і SVD на 128 дають val QWK 0.471 за argmax і 0.441 за очікуванням класу. Accuracy 0.479. По епохах argmax ішов 0.426, 0.463, 0.441, 0.471, 0.462, 0.467. Зупинка після 6-ї епохи, найкраща — 4-та. Val loss найнижчий на 3-й епосі (1.171), на 4-й уже 1.184 при train loss 1.047.

Це на 0.007 вище за Exp5 із трьома фото (0.464) і на 0.002 нижче за Exp6 (0.473). Поріг 0.02 не пройдено, новий `submission.csv` не записано. Публічний файл Exp5 зі score 0.6676 лишається чинним. Матриця лишилась 641 на 1287: змінився склад середнього вектора, ширина та сама. Пояснена дисперсія тексту теж та сама, 0.227.

На валідації клас 1 отримав 274 прогнози з 1287 (21.3% за частки 18.6% у train), клас 2 — 330 (25.6% за 27.6%), клас 3 — 181 (14.1% за 20.6%), клас 4 — 502 (39.0% за 33.2%).

Середнє по більшій кількості кадрів каппу майже не зсунуло. Число фото в цьому векторі не записане: три схожі знімки і двадцять схожих дають близьке середнє. Наступний блок лишає візуальні вектори Exp5, до 3 фото, і SVD на 128, і додає одну ознаку — логарифм кількості всіх фото профілю. Середнє по всіх кадрах у цей крок не входить.


## 15. Експеримент 8. Кількість фото поруч із вектором Exp5

Середнє до 8 фото дало val QWK 0.471 проти 0.464 у трьох фото. Приріст 0.007 менший за поріг сабміту. Усереднення всіх кадрів у цей крок не беремо: ширше середнє вже перевірене, а число знімків у середньому векторі не записане. Три схожі фото і двадцять схожих дають близький вектор.

Цей блок лишає візуальні ознаки Exp5: заморожений ResNet, не більше 3 фото, SVD тексту на 128, та сама голова з прихованим шаром, без ваг класів. Поруч додається одна ознака — натуральний логарифм від кількості всіх фото профілю, включно з тими, що в середнє не ввійшли. Кеш `resnet18_pet_features_8.npz` не використовується і не затирається.

Як запускати на Kaggle: **не натискайте Run All**. Не перезапускайте комірки навчання в розділах 12–14. Виконайте лише цей розділ зверху вниз. Підрахунок файлів фото швидкий. Якщо в сесії немає `resnet18_pet_features.npz`, друга комірка один раз порахує заморожені ознаки трьох фото; для цього потрібні GPU і Internet.

Поріг сабміту той самий: val QWK вищий за 0.4639 більше ніж на 0.02. Інакше чинним лишається файл зі score 0.6676.


In [ ]:
import os
import random
import re
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import cohen_kappa_score
from sklearn.model_selection import train_test_split

SEED = 42
VAL_SIZE = 0.2
MAX_IMAGES = 3
FEATURE_BATCH_SIZE = 64
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
DATA_ROOT = Path("/kaggle/input/competitions/deep-learning-for-computer-vision-and-nlp-2026-10")
IMAGE_DIR = DATA_ROOT / "images"
WORKING = Path("/kaggle/working")
WORKING.mkdir(parents=True, exist_ok=True)
NUM_WORKERS = 2 if os.name != "nt" else 0
if "device" not in globals():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

needs_tables = "train_df" not in globals() or "idx_tr" not in globals() or "y_all" not in globals()
if not needs_tables:
    print(f"Таблиці вже в пам'яті: train {len(train_df)}, val {len(idx_va)}. Повторно не читаю.")
else:
    TRAIN_CSV = DATA_ROOT / "train.csv"
    TEST_CSV = DATA_ROOT / "test.csv"
    SAMPLE_CSV = DATA_ROOT / "sample_submission.csv"

    def find_column(columns, options):
        lookup = {str(column).lower(): column for column in columns}
        for option in options:
            if option.lower() in lookup:
                return lookup[option.lower()]
        return None

    def normalize_id(value):
        if pd.isna(value):
            return ""
        text = str(value).strip()
        if text.endswith(".0"):
            text = text[:-2]
        return text

    def read_table(path):
        frame = pd.read_csv(path)
        frame.columns = [str(column).strip() for column in frame.columns]
        return frame

    def quadratic_weighted_kappa(y_true, y_pred):
        y_true = np.asarray(y_true).astype(int)
        y_pred = np.asarray(y_pred).astype(int)
        if len(y_true) == 0:
            return 0.0
        score = cohen_kappa_score(y_true, y_pred, weights="quadratic")
        if score != score:
            return 0.0
        return float(score)

    train_df = read_table(TRAIN_CSV)
    test_df = read_table(TEST_CSV)
    sample_df = read_table(SAMPLE_CSV)
    id_col = find_column(train_df.columns, ["PetID", "pet_id", "id"])
    target_col = find_column(train_df.columns, ["AdoptionSpeed", "adoption_speed", "label", "target"])
    text_col = find_column(train_df.columns, ["Description", "description", "desc", "text", "comments"])
    test_id_col = find_column(test_df.columns, ["PetID", "pet_id", "id"]) or id_col
    train_df[id_col] = train_df[id_col].map(normalize_id)
    test_df[test_id_col] = test_df[test_id_col].map(normalize_id)
    if int(train_df[id_col].duplicated().sum()):
        train_df = train_df.drop_duplicates(id_col, keep="first")
    train_df = train_df.reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)
    if target_col in test_df.columns:
        test_df = test_df.drop(columns=[target_col])
    test_text_col = find_column(test_df.columns, ["Description", "description", "desc", "text", "comments"])
    if text_col is None:
        text_col = "Description"
        train_df[text_col] = ""
        test_df[text_col] = ""
    elif test_text_col is None:
        test_df[text_col] = ""
    elif test_text_col != text_col:
        test_df[text_col] = test_df[test_text_col]
    raw_target = train_df[target_col]
    if pd.api.types.is_numeric_dtype(raw_target):
        numeric_target = raw_target.astype(float)
        if np.allclose(numeric_target.to_numpy(), np.rint(numeric_target.to_numpy())):
            train_df[target_col] = numeric_target.round().astype(int)
    classes = sorted(train_df[target_col].dropna().unique().tolist())
    LABEL_TO_IDX = {label: index for index, label in enumerate(classes)}
    IDX_TO_LABEL = {index: label for label, index in LABEL_TO_IDX.items()}
    N_CLASSES = len(classes)
    train_df = train_df.dropna(subset=[target_col]).reset_index(drop=True)
    y_all = train_df[target_col].map(LABEL_TO_IDX).to_numpy(dtype=int)
    idx_tr, idx_va = train_test_split(
        np.arange(len(train_df)), test_size=VAL_SIZE, random_state=SEED, stratify=y_all
    )
    y_tr = y_all[idx_tr]
    y_va = y_all[idx_va]
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    print(f"Таблиці прочитано заново: train {len(train_df)}, val {len(idx_va)}, test {len(test_df)}.")

if "train_ids" not in globals():
    train_ids = train_df[id_col].tolist()
if "test_ids" not in globals():
    test_ids = test_df[test_id_col].tolist()

if len(idx_va) != 1287:
    print(f"Увага: у валідації {len(idx_va)} рядків, у минулому прогоні було 1287.")
else:
    print("Розбиття збігається з минулим прогоном: 1287 валідаційних рядків.")

if "text_clean" in train_df.columns and "text_clean" in test_df.columns:
    print("Очищений опис уже є, повторно не чищу.")
else:
    NEGATION_TOKENS = {
        "no", "not", "nor", "never", "none", "nobody", "nothing",
        "neither", "nowhere", "without", "cannot",
    }
    CONTRACTIONS = {
        "ain't": "is not",
        "aren't": "are not",
        "can't": "cannot",
        "couldn't": "could not",
        "didn't": "did not",
        "doesn't": "does not",
        "don't": "do not",
        "hadn't": "had not",
        "hasn't": "has not",
        "haven't": "have not",
        "he's": "he is",
        "here's": "here is",
        "how's": "how is",
        "i'm": "i am",
        "i've": "i have",
        "isn't": "is not",
        "it's": "it is",
        "let's": "let us",
        "mustn't": "must not",
        "she's": "she is",
        "that's": "that is",
        "there's": "there is",
        "they're": "they are",
        "they've": "they have",
        "we're": "we are",
        "we've": "we have",
        "weren't": "were not",
        "what's": "what is",
        "where's": "where is",
        "who's": "who is",
        "won't": "will not",
        "wouldn't": "would not",
        "you're": "you are",
        "you've": "you have",
    }
    CONTRACTION_RE = re.compile(
        r"\b(" + "|".join(sorted(map(re.escape, CONTRACTIONS), key=len, reverse=True)) + r")\b"
    )
    FALLBACK_STOPWORDS = set(
        "a about above after again against all am an and any are as at be because been "
        "before being below between both but by can did do does doing down during each "
        "few for from further had has have having he her here hers herself him himself "
        "his how i if in into is it its itself just me more most my myself of off on "
        "once only or other our ours ourselves out over own same she should so some "
        "such than that the their theirs them themselves then there these they this "
        "those through to too under until up very was we were what when where which "
        "while who whom why will with you your yours yourself yourselves".split()
    )

    def load_stopwords():
        try:
            import nltk
            from nltk.corpus import stopwords

            nltk.download("stopwords", quiet=True)
            words = set(stopwords.words("english"))
            print("Стоп-слова: NLTK, english")
        except Exception as exc:
            print(f"Стоп-слова NLTK недоступні ({type(exc).__name__}: {exc}). Використовую вбудований список.")
            words = set(FALLBACK_STOPWORDS)
        return words - NEGATION_TOKENS

    STOPWORDS = load_stopwords()

    def clean_text(value):
        if not isinstance(value, str):
            return ""
        text = value.lower().replace("’", "'").replace("`", "'")
        text = CONTRACTION_RE.sub(lambda match: CONTRACTIONS[match.group(0)], text)
        text = re.sub(r"https?://\S+|www\.\S+", " ", text)
        text = re.sub(r"[^a-z\s]", " ", text)
        tokens = [token for token in text.split() if len(token) > 1 and token not in STOPWORDS]
        return " ".join(tokens)

    train_df["text_clean"] = train_df[text_col].fillna("").astype(str).map(clean_text)
    test_df["text_clean"] = test_df[text_col].fillna("").astype(str).map(clean_text)
    print("Порожніх описів після очищення:", float(train_df["text_clean"].eq("").mean()))

print("Пристрій:", device)


In [ ]:
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision.models import ResNet18_Weights, resnet18
from tqdm.auto import tqdm

reuse_images = (
    globals().get("IMAGE_FEATURE_LIMIT") == 3
    and "image_combined" in globals()
    and "image_combined_test" in globals()
    and len(image_combined) == len(train_df)
    and len(image_combined_test) == len(test_ids)
)
if reuse_images:
    print(
        "Вектори ResNet по 3 фото вже в пам'яті:",
        tuple(image_combined.shape),
        tuple(image_combined_test.shape),
        ". Заново не рахую.",
    )
else:
    def list_image_files(image_dir):
        files = []
        for path in Path(image_dir).rglob("*"):
            if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS:
                files.append(path)
        return files

    def link_images(pet_ids, files):
        pet_set = set(pet_ids)
        grouped = {pet_id: [] for pet_id in pet_set}
        for path in files:
            stem = path.stem
            pet_id = None
            order = 0
            if stem in pet_set:
                pet_id = stem
            else:
                for separator in ("-", "_"):
                    if separator not in stem:
                        continue
                    base, rest = stem.rsplit(separator, 1)
                    if base in pet_set:
                        pet_id = base
                        order = int(rest) if rest.isdigit() else 10**6
                        break
            if pet_id is not None:
                grouped[pet_id].append((order, path))
        linked = {}
        for pet_id, items in grouped.items():
            items.sort(key=lambda item: (item[0], item[1].name))
            linked[pet_id] = [path for _, path in items]
        return linked

    class PetImageDataset(Dataset):
        def __init__(self, records, transform):
            self.records = records
            self.transform = transform

        def __len__(self):
            return len(self.records)

        def __getitem__(self, index):
            pet_id, path = self.records[index]
            try:
                with Image.open(path) as image:
                    tensor = self.transform(image.convert("RGB"))
                ok = 1
            except Exception:
                tensor = torch.zeros(3, 224, 224)
                ok = 0
            return tensor, ok, pet_id

    def extract_resnet_features(paths_by_pet, backbone, transform):
        records = [(pet_id, path) for pet_id, paths in paths_by_pet.items() for path in paths]
        sums = {pet_id: np.zeros(512, dtype=np.float32) for pet_id in paths_by_pet}
        counts = {pet_id: 0 for pet_id in paths_by_pet}
        if not records:
            return {pet_id: (sums[pet_id], 0) for pet_id in paths_by_pet}
        loader = DataLoader(
            PetImageDataset(records, transform),
            batch_size=FEATURE_BATCH_SIZE,
            shuffle=False,
            num_workers=NUM_WORKERS,
        )
        backbone.eval()
        for tensors, oks, pet_ids_batch in tqdm(loader, desc="ResNet18"):
            features = backbone(tensors.to(device)).detach().cpu().numpy()
            oks = oks.numpy()
            for index, pet_id in enumerate(pet_ids_batch):
                if int(oks[index]) == 1:
                    sums[pet_id] += features[index]
                    counts[pet_id] += 1
        extracted = {}
        for pet_id in paths_by_pet:
            if counts[pet_id] > 0:
                extracted[pet_id] = (sums[pet_id] / counts[pet_id], counts[pet_id])
            else:
                extracted[pet_id] = (np.zeros(512, dtype=np.float32), 0)
        return extracted

    def stack_image_features(pet_ids, feature_map):
        vectors = np.zeros((len(pet_ids), 512), dtype=np.float32)
        missing = np.ones((len(pet_ids), 1), dtype=np.float32)
        for index, pet_id in enumerate(pet_ids):
            vector, count = feature_map[pet_id]
            if count > 0:
                vectors[index] = vector
                missing[index, 0] = 0.0
        return np.concatenate([vectors, missing], axis=1)

    def locate_resnet_cache():
        direct = WORKING / "resnet18_pet_features.npz"
        if direct.exists():
            return direct
        root = Path("/kaggle/input")
        if root.exists():
            found = sorted(root.rglob("resnet18_pet_features.npz"))
            if found:
                print(f"Кеш знайдено серед вхідних даних: {found[0]}")
                return found[0]
        return None

    def load_feature_map(cache_path):
        cached = np.load(cache_path, allow_pickle=False)
        cached_ids = [normalize_id(pet_id) for pet_id in cached["ids"].tolist()]
        counts = np.array(cached["counts"], copy=True)
        matrix = np.array(cached["features"], dtype=np.float32, copy=True)
        cached.close()
        feature_map = {}
        for index, pet_id in enumerate(cached_ids):
            feature_map[pet_id] = (matrix[index].copy(), int(counts[index]))
        return feature_map

    needed = set(train_ids) | set(test_ids)
    cache_path = locate_resnet_cache()
    feature_map = None
    if cache_path is not None:
        loaded = load_feature_map(cache_path)
        if set(loaded) >= needed:
            feature_map = loaded
            print(f"Ознаки ResNet завантажено з кешу: {cache_path}")
        else:
            print("Кеш ознак не покриває поточні ідентифікатори, рахую заново.")
    if feature_map is None:
        print("Кешу ResNet немає. Рахую заморожені ознаки, голову Exp2 не навчаю.")
        image_files = list_image_files(IMAGE_DIR)
        all_pet_ids = list(dict.fromkeys(list(train_ids) + list(test_ids)))
        full_image_paths = link_images(all_pet_ids, image_files)
        model_image_paths = {pet_id: paths[:MAX_IMAGES] for pet_id, paths in full_image_paths.items()}
        print("Файлів зображень:", len(image_files), f", у вектор до {MAX_IMAGES} фото.")
        try:
            weights = ResNet18_Weights.IMAGENET1K_V1
            backbone = resnet18(weights=weights)
            transform = weights.transforms()
        except Exception as exc:
            raise RuntimeError(
                "Не вдалося завантажити ваги ResNet18. Увімкніть Internet у налаштуваннях ноутбука."
            ) from exc
        backbone.fc = nn.Identity()
        backbone.to(device)
        backbone.eval()
        for parameter in backbone.parameters():
            parameter.requires_grad = False
        feature_map = extract_resnet_features(model_image_paths, backbone, transform)
        ids_array = np.array(list(feature_map.keys()))
        cache_out = WORKING / "resnet18_pet_features.npz"
        np.savez_compressed(
            cache_out,
            ids=ids_array,
            features=np.stack([feature_map[pet_id][0] for pet_id in ids_array]),
            counts=np.array([feature_map[pet_id][1] for pet_id in ids_array], dtype=np.int32),
        )
        print(f"Ознаки збережено: {cache_out}")
        del backbone
        if device.type == "cuda":
            torch.cuda.empty_cache()

    image_combined = stack_image_features(train_ids, feature_map)
    image_combined_test = stack_image_features(test_ids, feature_map)

IMAGE_FEATURE_LIMIT = 3
print("Матриця фото train/test:", tuple(image_combined.shape), tuple(image_combined_test.shape), "стеля 3")


In [ ]:
from pathlib import Path

import numpy as np

def list_image_files(image_dir):
    files = []
    for path in Path(image_dir).rglob("*"):
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS:
            files.append(path)
    return files


def link_all_images(pet_ids, files):
    pet_set = set(pet_ids)
    grouped = {pet_id: [] for pet_id in pet_set}
    for path in files:
        stem = path.stem
        pet_id = None
        order = 0
        if stem in pet_set:
            pet_id = stem
        else:
            for separator in ("-", "_"):
                if separator not in stem:
                    continue
                base, rest = stem.rsplit(separator, 1)
                if base in pet_set:
                    pet_id = base
                    order = int(rest) if rest.isdigit() else 10**6
                    break
        if pet_id is not None:
            grouped[pet_id].append((order, path))
    linked = {}
    for pet_id, items in grouped.items():
        items.sort(key=lambda item: (item[0], item[1].name))
        linked[pet_id] = [path for _, path in items]
    return linked


counted_files = list_image_files(IMAGE_DIR)
counted_ids = list(dict.fromkeys(list(train_ids) + list(test_ids)))
all_photo_paths = link_all_images(counted_ids, counted_files)
photo_count_train = np.array([len(all_photo_paths[pet_id]) for pet_id in train_ids], dtype=np.float32)
photo_count_test = np.array([len(all_photo_paths[pet_id]) for pet_id in test_ids], dtype=np.float32)
photo_count_log_train = np.log1p(photo_count_train).reshape(-1, 1)
photo_count_log_test = np.log1p(photo_count_test).reshape(-1, 1)
print(
    f"Усіх фото на тварину в train: median {float(np.median(photo_count_train)):.0f}, "
    f"max {int(photo_count_train.max())}, "
    f"більше 3: {float((photo_count_train > 3).mean()):.1%}, "
    f"більше 8: {float((photo_count_train > 8).mean()):.1%}."
)
print(
    f"Усіх фото на тварину в test: median {float(np.median(photo_count_test)):.0f}, "
    f"max {int(photo_count_test.max())}."
)


In [ ]:
import copy

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, Dataset

SVD_DIM = 128
JOINT_HIDDEN = 128
JOINT_DROPOUT = 0.3
JOINT_LR = 1e-3
JOINT_WEIGHT_DECAY = 1e-4
JOINT_MAX_EPOCHS = 8
JOINT_PATIENCE = 2
JOINT_MIN_DELTA = 0.001
JOINT_BATCH = 64
IMAGE_VAL_QWK = 0.4247
TFIDF_VAL_QWK = 0.2730
DISTILBERT_VAL_QWK = 0.1629
EXP5_VAL_QWK = 0.4639
EXP6_VAL_QWK = 0.4727
EXP7_VAL_QWK = 0.4709
SUBMISSION_MARGIN = 0.02
TFIDF_MAX_FEATURES = 20000
TFIDF_MIN_DF = 2


class JointDataset(Dataset):
    def __init__(self, features, labels=None):
        self.features = torch.tensor(np.asarray(features), dtype=torch.float32)
        self.labels = None if labels is None else torch.tensor(np.asarray(labels), dtype=torch.long)

    def __len__(self):
        return self.features.size(0)

    def __getitem__(self, index):
        if self.labels is None:
            return self.features[index]
        return self.features[index], self.labels[index]


class JointHead(nn.Module):
    def __init__(self, in_dim, hidden, n_classes, dropout=JOINT_DROPOUT):
        super().__init__()
        self.net = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, n_classes),
        )

    def forward(self, features):
        return self.net(features)


def decode_probabilities(probs, n_classes):
    class_index = np.arange(n_classes, dtype=np.float64)
    expectation = np.clip(np.rint(probs @ class_index), 0, n_classes - 1).astype(int)
    return {"argmax": probs.argmax(axis=1).astype(int), "expectation": expectation}


def fit_joint_head(features_train, labels_train, features_val, labels_val):
    torch.manual_seed(SEED)
    model = JointHead(features_train.shape[1], JOINT_HIDDEN, N_CLASSES).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=JOINT_LR, weight_decay=JOINT_WEIGHT_DECAY)
    criterion = nn.CrossEntropyLoss()
    train_loader = DataLoader(
        JointDataset(features_train, labels_train),
        batch_size=JOINT_BATCH,
        shuffle=True,
        generator=torch.Generator().manual_seed(SEED),
    )
    val_loader = DataLoader(JointDataset(features_val, labels_val), batch_size=256, shuffle=False)
    best_qwk = -1.0
    best_epoch = 1
    best_decoder = "argmax"
    best_state = copy.deepcopy(model.state_dict())
    wait = 0
    epochs_ran = 0
    for epoch in range(1, JOINT_MAX_EPOCHS + 1):
        epochs_ran = epoch
        model.train()
        total_loss, seen = 0.0, 0
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            optimizer.zero_grad(set_to_none=True)
            logits = model(batch_x)
            loss = criterion(logits, batch_y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            total_loss += loss.item() * batch_y.size(0)
            seen += batch_y.size(0)
        model.eval()
        prob_chunks, true_chunks = [], []
        val_loss, val_seen = 0.0, 0
        with torch.no_grad():
            for batch_x, batch_y in val_loader:
                batch_x, batch_y = batch_x.to(device), batch_y.to(device)
                logits = model(batch_x)
                val_loss += criterion(logits, batch_y).item() * batch_y.size(0)
                val_seen += batch_y.size(0)
                prob_chunks.append(torch.softmax(logits, dim=1).cpu().numpy())
                true_chunks.append(batch_y.cpu().numpy())
        probs = np.concatenate(prob_chunks)
        y_true = np.concatenate(true_chunks)
        decoded = decode_probabilities(probs, N_CLASSES)
        qwk_argmax = quadratic_weighted_kappa(y_true, decoded["argmax"])
        qwk_expectation = quadratic_weighted_kappa(y_true, decoded["expectation"])
        epoch_decoder = "expectation" if qwk_expectation > qwk_argmax else "argmax"
        epoch_qwk = qwk_expectation if epoch_decoder == "expectation" else qwk_argmax
        print(
            f"Епоха {epoch:02d}/{JOINT_MAX_EPOCHS} | train loss {total_loss / max(seen, 1):.4f} | "
            f"val loss {val_loss / max(val_seen, 1):.4f} | "
            f"QWK argmax {qwk_argmax:.4f} | QWK expectation {qwk_expectation:.4f}"
        )
        if epoch_qwk > best_qwk + JOINT_MIN_DELTA:
            best_qwk = epoch_qwk
            best_epoch = epoch
            best_decoder = epoch_decoder
            best_state = copy.deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1
            if wait >= JOINT_PATIENCE:
                break
    model.load_state_dict(best_state)
    print(f"Зупинка після епохи {epochs_ran}. Найкраща епоха {best_epoch}, QWK {best_qwk:.4f} ({best_decoder}).")
    return model, best_qwk, best_epoch, best_decoder, epochs_ran


def predict_joint(model, features):
    model.eval()
    loader = DataLoader(JointDataset(features), batch_size=256, shuffle=False)
    chunks = []
    with torch.no_grad():
        for batch_x in loader:
            logits = model(batch_x.to(device))
            chunks.append(torch.softmax(logits, dim=1).cpu().numpy())
    return np.concatenate(chunks)


def text_svd(frame_fit, frames_transform):
    vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=TFIDF_MIN_DF, max_features=TFIDF_MAX_FEATURES)
    sparse_fit = vectorizer.fit_transform(frame_fit)
    svd_dim = min(SVD_DIM, sparse_fit.shape[1] - 1, sparse_fit.shape[0] - 1)
    svd = TruncatedSVD(n_components=svd_dim, random_state=SEED)
    dense_fit = svd.fit_transform(sparse_fit)
    transformed = [svd.transform(vectorizer.transform(frame)) for frame in frames_transform]
    explained = float(svd.explained_variance_ratio_.sum())
    return dense_fit, transformed, explained, sparse_fit.shape[1], svd_dim


text_tr, (text_va,), explained, n_tfidf, svd_dim = text_svd(
    train_df.iloc[idx_tr]["text_clean"],
    [train_df.iloc[idx_va]["text_clean"]],
)
print(
    f"TF-IDF ознак {n_tfidf}, компонент SVD {svd_dim}, "
    f"пояснена дисперсія {explained:.3f}. Ваг класів немає."
)
scaler = StandardScaler()
fused_tr = scaler.fit_transform(np.concatenate([image_combined[idx_tr], text_tr, photo_count_log_train[idx_tr]], axis=1)).astype(np.float32)
fused_va = scaler.transform(np.concatenate([image_combined[idx_va], text_va, photo_count_log_train[idx_va]], axis=1)).astype(np.float32)
print("Спільна матриця train/val:", fused_tr.shape, fused_va.shape)

joint_model, joint_qwk, joint_best_epoch, joint_decoder, joint_epochs = fit_joint_head(fused_tr, y_tr, fused_va, y_va)
val_probs = predict_joint(joint_model, fused_va)
val_decoded = decode_probabilities(val_probs, N_CLASSES)
val_scores = {name: quadratic_weighted_kappa(y_va, pred) for name, pred in val_decoded.items()}
chosen_pred = val_decoded[joint_decoder]
print(
    f"Exp8 QWK argmax {val_scores['argmax']:.4f}, expectation {val_scores['expectation']:.4f}. "
    f"Обрано {joint_decoder}, accuracy {accuracy_score(y_va, chosen_pred):.4f}."
)
print("Прогнози на валідації:")
print(pd.Series(chosen_pred).map(IDX_TO_LABEL).value_counts().sort_index())
print(
    f"Порівняння: TF-IDF {TFIDF_VAL_QWK:.4f}, фото {IMAGE_VAL_QWK:.4f}, "
    f"DistilBERT {DISTILBERT_VAL_QWK:.4f}, Exp5 {EXP5_VAL_QWK:.4f}, Exp6 {EXP6_VAL_QWK:.4f}, "
f"Exp7 {EXP7_VAL_QWK:.4f}, кількість фото {val_scores[joint_decoder]:.4f}."
)

exp8_name = "Exp8. ResNet 3 фото + кількість фото + SVD 128"
exp8_notes = (
    f"без ваг класів; log1p кількості фото; svd {svd_dim}; пояснена дисперсія {explained:.3f}; "
    f"argmax {val_scores['argmax']:.4f}; expectation {val_scores['expectation']:.4f}"
)
if "log_experiment" in globals():
    log_experiment(exp8_name, val_scores[joint_decoder], joint_decoder, joint_epochs, joint_best_epoch, exp8_notes)
else:
    print(exp8_name, round(val_scores[joint_decoder], 4), joint_decoder, "епох", joint_epochs, "найкраща", joint_best_epoch)

if val_scores[joint_decoder] > EXP5_VAL_QWK + SUBMISSION_MARGIN:
    print("Кількість фото підняла val QWK вище за Exp5 більше ніж на 0.02. Навчаю голову на повному train і пишу submission.csv.")
    text_all, (text_test,), _, _, _ = text_svd(
        train_df["text_clean"],
        [test_df["text_clean"]],
    )
    final_scaler = StandardScaler()
    fused_all = final_scaler.fit_transform(np.concatenate([image_combined, text_all, photo_count_log_train], axis=1)).astype(np.float32)
    fused_test = final_scaler.transform(np.concatenate([image_combined_test, text_test, photo_count_log_test], axis=1)).astype(np.float32)
    torch.manual_seed(SEED)
    final_joint = JointHead(fused_all.shape[1], JOINT_HIDDEN, N_CLASSES).to(device)
    final_optimizer = torch.optim.AdamW(final_joint.parameters(), lr=JOINT_LR, weight_decay=JOINT_WEIGHT_DECAY)
    final_criterion = nn.CrossEntropyLoss()
    final_loader = DataLoader(
        JointDataset(fused_all, y_all),
        batch_size=JOINT_BATCH,
        shuffle=True,
        generator=torch.Generator().manual_seed(SEED),
    )
    final_epochs = max(int(joint_best_epoch), 1)
    final_joint.train()
    for epoch in range(1, final_epochs + 1):
        total_loss, seen = 0.0, 0
        for batch_x, batch_y in final_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            final_optimizer.zero_grad(set_to_none=True)
            logits = final_joint(batch_x)
            loss = final_criterion(logits, batch_y)
            loss.backward()
            nn.utils.clip_grad_norm_(final_joint.parameters(), 5.0)
            final_optimizer.step()
            total_loss += loss.item() * batch_y.size(0)
            seen += batch_y.size(0)
        print(f"Повне навчання спільної голови, епоха {epoch}/{final_epochs}, train loss {total_loss / max(seen, 1):.4f}")
    test_probs = predict_joint(final_joint, fused_test)
    test_idx = decode_probabilities(test_probs, N_CLASSES)[joint_decoder]
    submission = pd.DataFrame({
        "PetID": [normalize_id(pet_id) for pet_id in test_ids],
        "AdoptionSpeed": [IDX_TO_LABEL[int(index)] for index in test_idx],
    })
    submission = submission.drop_duplicates("PetID", keep="first")
    submission["AdoptionSpeed"] = submission["AdoptionSpeed"].astype(int)
    submission = submission[["PetID", "AdoptionSpeed"]]
    submission.to_csv(WORKING / "submission.csv", index=False)
    print(f"Збережено submission.csv, рядків {len(submission)}.")
    print(submission["AdoptionSpeed"].value_counts().sort_index())
else:
    print(
        "Новий submission.csv не записано: val QWK не вищий за Exp5 більше ніж на 0.02. "
        "Публічний файл зі score 0.6676 лишається чинним."
    )


### Висновки експерименту 8

Логарифм кількості всіх фото додано до матриці Exp5: спільний вектор став 642 замість 641. Val QWK 0.446 за argmax і 0.391 за очікуванням класу. Accuracy 0.473. По епохах argmax ішов 0.413, 0.446, 0.440, 0.436. Зупинка після 4-ї епохи, найкраща — 2-га. На ній же val loss найнижчий, 1.179; далі train loss падав до 1.051, а val loss ріс до 1.187.

Це на 0.018 нижче за Exp5 (0.464). Поріг 0.02 не пройдено, новий `submission.csv` не записано. Публічний файл зі score 0.6676 лишається чинним. Колонка не лишилась непоміченою: на першій епосі каппа 0.413, тоді як та сама голова без неї в Exp5 мала 0.436, і розрив далі не закрився.

На валідації клас 1 отримав 248 прогнозів із 1287 (19.3% за частки 18.6% у train), клас 2 — 376 (29.2% за 27.6%), клас 3 — 156 (12.1% за 20.6%), клас 4 — 507 (39.4% за 33.2%).

Кількість фото в цій голові каппу знизила. Середнє по всіх кадрах наступним кроком не ставимо: стеля 8 уже дала лише +0.007 до Exp5. Далі донавчається останній залишковий блок ResNet на зв'язці, яка виграла публічно: до 3 фото і SVD на 128, без колонки кількості. Решта мережі лишається замороженою.


## 16. Експеримент 9. Останній блок ResNet на зв'язці Exp5

Кількість фото знизила val QWK до 0.446 проти 0.464 у Exp5. Середнє до 8 фото дало лише 0.471. Обидва кроки лишаються в історії, у цю модель не входять. Публічно кращий файл — Exp5, score 0.6676: заморожене середнє до 3 фото і SVD тексту на 128.

Тут донавчається лише останній залишковий блок ResNet18 (`layer4`, другий блок). Решта мережі, включно з першим блоком `layer4`, заморожена і лишається в режимі оцінки, щоб статистика її нормалізації не поїхала. Голова та сама: LayerNorm, прихований шар на 128, без ваг класів. LayerNorm стоїть замість зовнішнього шкалювання Exp5, бо вектор фото змінюється під час навчання. Текст — SVD на 128, навчений лише на навчальній частині розбиття. Швидкість навчання блока 1e-4, голови 1e-3.

Як запускати на Kaggle: **не натискайте Run All**. Не перезапускайте комірки навчання попередніх розділів. Виконайте лише цей розділ зверху вниз. Друга комірка читає фото щоепохи, це довше за навчання самої голови. Потрібні GPU і Internet.

Поріг сабміту: val QWK вищий за 0.4639 більше ніж на 0.02. Інакше чинним лишається файл зі score 0.6676.


In [ ]:
import os
import random
import re
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import cohen_kappa_score
from sklearn.model_selection import train_test_split

SEED = 42
VAL_SIZE = 0.2
MAX_IMAGES = 3
FEATURE_BATCH_SIZE = 64
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
DATA_ROOT = Path("/kaggle/input/competitions/deep-learning-for-computer-vision-and-nlp-2026-10")
IMAGE_DIR = DATA_ROOT / "images"
WORKING = Path("/kaggle/working")
WORKING.mkdir(parents=True, exist_ok=True)
NUM_WORKERS = 2 if os.name != "nt" else 0
if "device" not in globals():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

needs_tables = "train_df" not in globals() or "idx_tr" not in globals() or "y_all" not in globals()
if not needs_tables:
    print(f"Таблиці вже в пам'яті: train {len(train_df)}, val {len(idx_va)}. Повторно не читаю.")
else:
    TRAIN_CSV = DATA_ROOT / "train.csv"
    TEST_CSV = DATA_ROOT / "test.csv"
    SAMPLE_CSV = DATA_ROOT / "sample_submission.csv"

    def find_column(columns, options):
        lookup = {str(column).lower(): column for column in columns}
        for option in options:
            if option.lower() in lookup:
                return lookup[option.lower()]
        return None

    def normalize_id(value):
        if pd.isna(value):
            return ""
        text = str(value).strip()
        if text.endswith(".0"):
            text = text[:-2]
        return text

    def read_table(path):
        frame = pd.read_csv(path)
        frame.columns = [str(column).strip() for column in frame.columns]
        return frame

    def quadratic_weighted_kappa(y_true, y_pred):
        y_true = np.asarray(y_true).astype(int)
        y_pred = np.asarray(y_pred).astype(int)
        if len(y_true) == 0:
            return 0.0
        score = cohen_kappa_score(y_true, y_pred, weights="quadratic")
        if score != score:
            return 0.0
        return float(score)

    train_df = read_table(TRAIN_CSV)
    test_df = read_table(TEST_CSV)
    sample_df = read_table(SAMPLE_CSV)
    id_col = find_column(train_df.columns, ["PetID", "pet_id", "id"])
    target_col = find_column(train_df.columns, ["AdoptionSpeed", "adoption_speed", "label", "target"])
    text_col = find_column(train_df.columns, ["Description", "description", "desc", "text", "comments"])
    test_id_col = find_column(test_df.columns, ["PetID", "pet_id", "id"]) or id_col
    train_df[id_col] = train_df[id_col].map(normalize_id)
    test_df[test_id_col] = test_df[test_id_col].map(normalize_id)
    if int(train_df[id_col].duplicated().sum()):
        train_df = train_df.drop_duplicates(id_col, keep="first")
    train_df = train_df.reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)
    if target_col in test_df.columns:
        test_df = test_df.drop(columns=[target_col])
    test_text_col = find_column(test_df.columns, ["Description", "description", "desc", "text", "comments"])
    if text_col is None:
        text_col = "Description"
        train_df[text_col] = ""
        test_df[text_col] = ""
    elif test_text_col is None:
        test_df[text_col] = ""
    elif test_text_col != text_col:
        test_df[text_col] = test_df[test_text_col]
    raw_target = train_df[target_col]
    if pd.api.types.is_numeric_dtype(raw_target):
        numeric_target = raw_target.astype(float)
        if np.allclose(numeric_target.to_numpy(), np.rint(numeric_target.to_numpy())):
            train_df[target_col] = numeric_target.round().astype(int)
    classes = sorted(train_df[target_col].dropna().unique().tolist())
    LABEL_TO_IDX = {label: index for index, label in enumerate(classes)}
    IDX_TO_LABEL = {index: label for label, index in LABEL_TO_IDX.items()}
    N_CLASSES = len(classes)
    train_df = train_df.dropna(subset=[target_col]).reset_index(drop=True)
    y_all = train_df[target_col].map(LABEL_TO_IDX).to_numpy(dtype=int)
    idx_tr, idx_va = train_test_split(
        np.arange(len(train_df)), test_size=VAL_SIZE, random_state=SEED, stratify=y_all
    )
    y_tr = y_all[idx_tr]
    y_va = y_all[idx_va]
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    print(f"Таблиці прочитано заново: train {len(train_df)}, val {len(idx_va)}, test {len(test_df)}.")

if "train_ids" not in globals():
    train_ids = train_df[id_col].tolist()
if "test_ids" not in globals():
    test_ids = test_df[test_id_col].tolist()

if len(idx_va) != 1287:
    print(f"Увага: у валідації {len(idx_va)} рядків, у минулому прогоні було 1287.")
else:
    print("Розбиття збігається з минулим прогоном: 1287 валідаційних рядків.")

if "text_clean" in train_df.columns and "text_clean" in test_df.columns:
    print("Очищений опис уже є, повторно не чищу.")
else:
    NEGATION_TOKENS = {
        "no", "not", "nor", "never", "none", "nobody", "nothing",
        "neither", "nowhere", "without", "cannot",
    }
    CONTRACTIONS = {
        "ain't": "is not",
        "aren't": "are not",
        "can't": "cannot",
        "couldn't": "could not",
        "didn't": "did not",
        "doesn't": "does not",
        "don't": "do not",
        "hadn't": "had not",
        "hasn't": "has not",
        "haven't": "have not",
        "he's": "he is",
        "here's": "here is",
        "how's": "how is",
        "i'm": "i am",
        "i've": "i have",
        "isn't": "is not",
        "it's": "it is",
        "let's": "let us",
        "mustn't": "must not",
        "she's": "she is",
        "that's": "that is",
        "there's": "there is",
        "they're": "they are",
        "they've": "they have",
        "we're": "we are",
        "we've": "we have",
        "weren't": "were not",
        "what's": "what is",
        "where's": "where is",
        "who's": "who is",
        "won't": "will not",
        "wouldn't": "would not",
        "you're": "you are",
        "you've": "you have",
    }
    CONTRACTION_RE = re.compile(
        r"\b(" + "|".join(sorted(map(re.escape, CONTRACTIONS), key=len, reverse=True)) + r")\b"
    )
    FALLBACK_STOPWORDS = set(
        "a about above after again against all am an and any are as at be because been "
        "before being below between both but by can did do does doing down during each "
        "few for from further had has have having he her here hers herself him himself "
        "his how i if in into is it its itself just me more most my myself of off on "
        "once only or other our ours ourselves out over own same she should so some "
        "such than that the their theirs them themselves then there these they this "
        "those through to too under until up very was we were what when where which "
        "while who whom why will with you your yours yourself yourselves".split()
    )

    def load_stopwords():
        try:
            import nltk
            from nltk.corpus import stopwords

            nltk.download("stopwords", quiet=True)
            words = set(stopwords.words("english"))
            print("Стоп-слова: NLTK, english")
        except Exception as exc:
            print(f"Стоп-слова NLTK недоступні ({type(exc).__name__}: {exc}). Використовую вбудований список.")
            words = set(FALLBACK_STOPWORDS)
        return words - NEGATION_TOKENS

    STOPWORDS = load_stopwords()

    def clean_text(value):
        if not isinstance(value, str):
            return ""
        text = value.lower().replace("’", "'").replace("`", "'")
        text = CONTRACTION_RE.sub(lambda match: CONTRACTIONS[match.group(0)], text)
        text = re.sub(r"https?://\S+|www\.\S+", " ", text)
        text = re.sub(r"[^a-z\s]", " ", text)
        tokens = [token for token in text.split() if len(token) > 1 and token not in STOPWORDS]
        return " ".join(tokens)

    train_df["text_clean"] = train_df[text_col].fillna("").astype(str).map(clean_text)
    test_df["text_clean"] = test_df[text_col].fillna("").astype(str).map(clean_text)
    print("Порожніх описів після очищення:", float(train_df["text_clean"].eq("").mean()))

print("Пристрій:", device)


In [ ]:
import copy
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score
from torch.utils.data import DataLoader, Dataset
from torchvision.models import ResNet18_Weights, resnet18
PHOTO_CAP = 3
SVD_DIM = 128
TUNE_HIDDEN = 128
TUNE_DROPOUT = 0.3
HEAD_LR = 1e-3
BLOCK_LR = 1e-4
TUNE_WEIGHT_DECAY = 1e-4
TUNE_MAX_EPOCHS = 8
TUNE_PATIENCE = 2
TUNE_MIN_DELTA = 0.001
TUNE_BATCH = 16
EXP5_VAL_QWK = 0.4639
EXP8_VAL_QWK = 0.4457
SUBMISSION_MARGIN = 0.02
TFIDF_MAX_FEATURES = 20000
TFIDF_MIN_DF = 2

if device.type == "cuda":
    torch.cuda.empty_cache()


def list_image_files(image_dir):
    files = []
    for path in Path(image_dir).rglob("*"):
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS:
            files.append(path)
    return files


def link_all_images(pet_ids, files):
    pet_set = set(pet_ids)
    grouped = {pet_id: [] for pet_id in pet_set}
    for path in files:
        stem = path.stem
        pet_id = None
        order = 0
        if stem in pet_set:
            pet_id = stem
        else:
            for separator in ("-", "_"):
                if separator not in stem:
                    continue
                base, rest = stem.rsplit(separator, 1)
                if base in pet_set:
                    pet_id = base
                    order = int(rest) if rest.isdigit() else 10**6
                    break
        if pet_id is not None:
            grouped[pet_id].append((order, path))
    linked = {}
    for pet_id, items in grouped.items():
        items.sort(key=lambda item: (item[0], item[1].name))
        linked[pet_id] = [path for _, path in items]
    return linked


def capped_paths(pet_ids):
    files = list_image_files(IMAGE_DIR)
    linked = link_all_images(pet_ids, files)
    rows = [linked[pet_id][:PHOTO_CAP] for pet_id in pet_ids]
    print(f"Файлів зображень: {len(files)}. У мережу входить до {PHOTO_CAP} фото на тварину.")
    return rows


class PetFusionDataset(Dataset):
    def __init__(self, row_index, text_matrix, path_rows, transform, labels=None):
        self.row_index = np.asarray(row_index, dtype=int)
        self.text_matrix = np.asarray(text_matrix, dtype=np.float32)
        self.path_rows = path_rows
        self.transform = transform
        self.labels = None if labels is None else np.asarray(labels, dtype=np.int64)

    def __len__(self):
        return len(self.row_index)

    def __getitem__(self, index):
        row = int(self.row_index[index])
        frames = []
        for path in self.path_rows[row]:
            try:
                with Image.open(path) as image:
                    frames.append(self.transform(image.convert("RGB")))
            except Exception:
                continue
        valid = len(frames)
        while len(frames) < PHOTO_CAP:
            frames.append(torch.zeros(3, 224, 224))
        item = {
            "images": torch.stack(frames[:PHOTO_CAP]),
            "valid": valid,
            "text": torch.tensor(self.text_matrix[row]),
        }
        if self.labels is not None:
            item["label"] = int(self.labels[row])
        return item


def make_backbone():
    try:
        weights = ResNet18_Weights.IMAGENET1K_V1
        backbone = resnet18(weights=weights)
        transform = weights.transforms()
    except Exception as exc:
        raise RuntimeError(
            "Не вдалося завантажити ваги ResNet18. Увімкніть Internet у налаштуваннях ноутбука."
        ) from exc
    backbone.fc = nn.Identity()
    for parameter in backbone.parameters():
        parameter.requires_grad = False
    for parameter in backbone.layer4[-1].parameters():
        parameter.requires_grad = True
    return backbone.to(device), transform


def trainable_count(module):
    return sum(parameter.numel() for parameter in module.parameters() if parameter.requires_grad)


class TuneHead(nn.Module):
    def __init__(self, in_dim, hidden, n_classes, dropout=TUNE_DROPOUT):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(in_dim),
            nn.Dropout(dropout),
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, n_classes),
        )

    def forward(self, features):
        return self.net(features)


def pool_images(backbone, images, valid):
    batch, slots, channels, height, width = images.shape
    flat = images.reshape(batch * slots, channels, height, width)
    features = backbone(flat).reshape(batch, slots, -1)
    positions = torch.arange(slots, device=images.device).view(1, slots)
    mask = positions < valid.view(batch, 1)
    features = features * mask.unsqueeze(-1)
    pooled = features.sum(dim=1) / valid.clamp(min=1).view(batch, 1).float()
    missing = (valid == 0).float().unsqueeze(1)
    pooled = pooled * (1.0 - missing)
    return torch.cat([pooled, missing], dim=1)


def text_matrices(fit_text, groups):
    vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=TFIDF_MIN_DF, max_features=TFIDF_MAX_FEATURES)
    sparse_fit = vectorizer.fit_transform(fit_text)
    svd_dim = min(SVD_DIM, sparse_fit.shape[1] - 1, sparse_fit.shape[0] - 1)
    svd = TruncatedSVD(n_components=svd_dim, random_state=SEED)
    svd.fit(sparse_fit)
    matrices = [svd.transform(vectorizer.transform(rows)).astype(np.float32) for rows in groups]
    explained = float(svd.explained_variance_ratio_.sum())
    return matrices, explained, sparse_fit.shape[1], svd_dim


def decode_probabilities(probs, n_classes):
    class_index = np.arange(n_classes, dtype=np.float64)
    expectation = np.clip(np.rint(probs @ class_index), 0, n_classes - 1).astype(int)
    return {"argmax": probs.argmax(axis=1).astype(int), "expectation": expectation}


def set_train_mode(backbone, head):
    backbone.eval()
    backbone.layer4[-1].train()
    head.train()


def batch_logits(backbone, head, batch):
    images = batch["images"].to(device)
    valid = batch["valid"].to(device)
    text = batch["text"].to(device)
    visual = pool_images(backbone, images, valid)
    return head(torch.cat([visual, text], dim=1))


def fit_tuned(backbone, head, train_loader, val_loader):
    optimizer = torch.optim.AdamW(
        [
            {"params": backbone.layer4[-1].parameters(), "lr": BLOCK_LR},
            {"params": head.parameters(), "lr": HEAD_LR},
        ],
        weight_decay=TUNE_WEIGHT_DECAY,
    )
    criterion = nn.CrossEntropyLoss()
    best_qwk = -1.0
    best_epoch = 1
    best_decoder = "argmax"
    best_state = {
        "block": copy.deepcopy(backbone.layer4[-1].state_dict()),
        "head": copy.deepcopy(head.state_dict()),
    }
    wait = 0
    epochs_ran = 0
    for epoch in range(1, TUNE_MAX_EPOCHS + 1):
        epochs_ran = epoch
        set_train_mode(backbone, head)
        total_loss, seen = 0.0, 0
        for batch in train_loader:
            labels = batch["label"].to(device)
            optimizer.zero_grad(set_to_none=True)
            logits = batch_logits(backbone, head, batch)
            loss = criterion(logits, labels)
            loss.backward()
            nn.utils.clip_grad_norm_(
                [parameter for parameter in list(backbone.layer4[-1].parameters()) + list(head.parameters()) if parameter.requires_grad],
                5.0,
            )
            optimizer.step()
            total_loss += loss.item() * labels.size(0)
            seen += labels.size(0)
        qwk_argmax, qwk_expectation, val_loss = evaluate_qwk(backbone, head, val_loader, criterion)
        epoch_decoder = "expectation" if qwk_expectation > qwk_argmax else "argmax"
        epoch_qwk = qwk_expectation if epoch_decoder == "expectation" else qwk_argmax
        print(
            f"Епоха {epoch:02d}/{TUNE_MAX_EPOCHS} | train loss {total_loss / max(seen, 1):.4f} | "
            f"val loss {val_loss:.4f} | QWK argmax {qwk_argmax:.4f} | QWK expectation {qwk_expectation:.4f}"
        )
        if epoch_qwk > best_qwk + TUNE_MIN_DELTA:
            best_qwk = epoch_qwk
            best_epoch = epoch
            best_decoder = epoch_decoder
            best_state = {
                "block": copy.deepcopy(backbone.layer4[-1].state_dict()),
                "head": copy.deepcopy(head.state_dict()),
            }
            wait = 0
        else:
            wait += 1
            if wait >= TUNE_PATIENCE:
                break
    backbone.layer4[-1].load_state_dict(best_state["block"])
    head.load_state_dict(best_state["head"])
    print(f"Зупинка після епохи {epochs_ran}. Найкраща епоха {best_epoch}, QWK {best_qwk:.4f} ({best_decoder}).")
    return best_qwk, best_epoch, best_decoder, epochs_ran


def evaluate_qwk(backbone, head, loader, criterion):
    backbone.eval()
    head.eval()
    prob_chunks, true_chunks = [], []
    total_loss, seen = 0.0, 0
    with torch.no_grad():
        for batch in loader:
            labels = batch["label"].to(device)
            logits = batch_logits(backbone, head, batch)
            total_loss += criterion(logits, labels).item() * labels.size(0)
            seen += labels.size(0)
            prob_chunks.append(torch.softmax(logits, dim=1).cpu().numpy())
            true_chunks.append(labels.cpu().numpy())
    probs = np.concatenate(prob_chunks)
    y_true = np.concatenate(true_chunks)
    decoded = decode_probabilities(probs, N_CLASSES)
    return (
        quadratic_weighted_kappa(y_true, decoded["argmax"]),
        quadratic_weighted_kappa(y_true, decoded["expectation"]),
        total_loss / max(seen, 1),
    )


def predict_probs(backbone, head, loader):
    backbone.eval()
    head.eval()
    chunks = []
    with torch.no_grad():
        for batch in loader:
            logits = batch_logits(backbone, head, batch)
            chunks.append(torch.softmax(logits, dim=1).cpu().numpy())
    return np.concatenate(chunks)


def make_loader(row_index, text_matrix, path_rows, transform, labels, shuffle):
    dataset = PetFusionDataset(row_index, text_matrix, path_rows, transform, labels)
    return DataLoader(
        dataset,
        batch_size=TUNE_BATCH,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        generator=torch.Generator().manual_seed(SEED) if shuffle else None,
    )


train_photo_rows = capped_paths(train_ids)
test_photo_rows = capped_paths(test_ids)
(text_train_matrix,), explained, n_tfidf, svd_dim = text_matrices(
    train_df.iloc[idx_tr]["text_clean"],
    [train_df["text_clean"]],
)
print(
    f"TF-IDF ознак {n_tfidf}, компонент SVD {svd_dim}, пояснена дисперсія {explained:.3f}. "
    "Ваг класів немає. Кількість фото в модель не входить."
)

torch.manual_seed(SEED)
backbone, image_transform = make_backbone()
head = TuneHead(513 + svd_dim, TUNE_HIDDEN, N_CLASSES).to(device)
print(
    "Параметрів блока, що навчається:",
    trainable_count(backbone),
    ". Параметрів голови:",
    trainable_count(head),
)
train_loader = make_loader(idx_tr, text_train_matrix, train_photo_rows, image_transform, y_all, True)
val_loader = make_loader(idx_va, text_train_matrix, train_photo_rows, image_transform, y_all, False)
_, tune_best_epoch, tune_decoder, tune_epochs = fit_tuned(backbone, head, train_loader, val_loader)
val_probs = predict_probs(backbone, head, val_loader)
val_decoded = decode_probabilities(val_probs, N_CLASSES)
val_scores = {name: quadratic_weighted_kappa(y_va, pred) for name, pred in val_decoded.items()}
chosen_pred = val_decoded[tune_decoder]
print(
    f"Exp9 QWK argmax {val_scores['argmax']:.4f}, expectation {val_scores['expectation']:.4f}. "
    f"Обрано {tune_decoder}, accuracy {accuracy_score(y_va, chosen_pred):.4f}."
)
print("Прогнози на валідації:")
print(pd.Series(chosen_pred).map(IDX_TO_LABEL).value_counts().sort_index())
print(
    f"Порівняння: Exp5 {EXP5_VAL_QWK:.4f}, Exp8 {EXP8_VAL_QWK:.4f}, "
    f"останній блок {val_scores[tune_decoder]:.4f}."
)

exp9_name = "Exp9. Останній блок ResNet + SVD 128"
exp9_notes = (
    f"layer4[-1]; lr блока {BLOCK_LR}; без ваг класів; без кількості фото; svd {svd_dim}; "
    f"argmax {val_scores['argmax']:.4f}; expectation {val_scores['expectation']:.4f}"
)
if "log_experiment" in globals():
    log_experiment(exp9_name, val_scores[tune_decoder], tune_decoder, tune_epochs, tune_best_epoch, exp9_notes)
else:
    print(exp9_name, round(val_scores[tune_decoder], 4), tune_decoder, "епох", tune_epochs, "найкраща", tune_best_epoch)

if val_scores[tune_decoder] > EXP5_VAL_QWK + SUBMISSION_MARGIN:
    print("Останній блок обігнав Exp5 на валідації більше ніж на 0.02. Навчаю на повному train і пишу submission.csv.")
    (text_full, text_test), _, _, _ = text_matrices(
        train_df["text_clean"],
        [train_df["text_clean"], test_df["text_clean"]],
    )
    torch.manual_seed(SEED)
    final_backbone, final_transform = make_backbone()
    final_head = TuneHead(513 + text_full.shape[1], TUNE_HIDDEN, N_CLASSES).to(device)
    final_optimizer = torch.optim.AdamW(
        [
            {"params": final_backbone.layer4[-1].parameters(), "lr": BLOCK_LR},
            {"params": final_head.parameters(), "lr": HEAD_LR},
        ],
        weight_decay=TUNE_WEIGHT_DECAY,
    )
    final_criterion = nn.CrossEntropyLoss()
    final_loader = make_loader(np.arange(len(train_df)), text_full, train_photo_rows, final_transform, y_all, True)
    final_epochs = max(int(tune_best_epoch), 1)
    for epoch in range(1, final_epochs + 1):
        set_train_mode(final_backbone, final_head)
        total_loss, seen = 0.0, 0
        for batch in final_loader:
            labels = batch["label"].to(device)
            final_optimizer.zero_grad(set_to_none=True)
            logits = batch_logits(final_backbone, final_head, batch)
            loss = final_criterion(logits, labels)
            loss.backward()
            nn.utils.clip_grad_norm_(
                [parameter for parameter in list(final_backbone.layer4[-1].parameters()) + list(final_head.parameters()) if parameter.requires_grad],
                5.0,
            )
            final_optimizer.step()
            total_loss += loss.item() * labels.size(0)
            seen += labels.size(0)
        print(f"Повне навчання, епоха {epoch}/{final_epochs}, train loss {total_loss / max(seen, 1):.4f}")
    test_loader = make_loader(np.arange(len(test_ids)), text_test, test_photo_rows, final_transform, None, False)
    test_probs = predict_probs(final_backbone, final_head, test_loader)
    test_idx = decode_probabilities(test_probs, N_CLASSES)[tune_decoder]
    submission = pd.DataFrame({
        "PetID": [normalize_id(pet_id) for pet_id in test_ids],
        "AdoptionSpeed": [IDX_TO_LABEL[int(index)] for index in test_idx],
    })
    submission = submission.drop_duplicates("PetID", keep="first")
    submission["AdoptionSpeed"] = submission["AdoptionSpeed"].astype(int)
    submission = submission[["PetID", "AdoptionSpeed"]]
    submission.to_csv(WORKING / "submission.csv", index=False)
    print(f"Збережено submission.csv, рядків {len(submission)}.")
    print(submission["AdoptionSpeed"].value_counts().sort_index())
else:
    print(
        "Новий submission.csv не записано: val QWK не вищий за Exp5 більше ніж на 0.02. "
        "Публічний файл зі score 0.6676 лишається чинним."
    )


### Висновки експерименту 9

Останній залишковий блок ResNet, 4 720 640 параметрів, і голова на 83 974 параметри дають val QWK 0.402 за argmax і 0.377 за очікуванням класу. Accuracy 0.442. По епохах argmax ішов 0.402, 0.396, 0.392. Зупинка після 3-ї епохи, найкраща — 1-ша. Train loss упав з 1.261 до 1.047. Val loss на 2-й епосі знизився до 1.227, але каппа вже була нижча за першу епоху; на 3-й val loss виріс до 1.259.

Це на 0.062 нижче за заморожений Exp5 (0.464). Поріг 0.02 не пройдено, новий `submission.csv` не записано. Публічний файл зі score 0.6676 лишається чинним.

На валідації клас 1 отримав 300 прогнозів із 1287 (23.3% за частки 18.6% у train), клас 2 — 335 (26.0% за 27.6%), клас 3 — 68 (5.3% за 20.6%), клас 4 — 584 (45.4% за 33.2%). Клас 3 стиснувся сильніше, ніж у замороженій моделі.

4.7 млн ваг блоку оновлювались разом із ще ненавченою головою. Уже перша епоха відсунула ознаки ImageNet від робочої точки Exp5, і наступні епохи цей розрив не закрили. Ширше розмороження тим самим способом не продовжуємо. Наступний блок повертає заморожений енкодер і голову Exp5, а ResNet18 замінює на EfficientNet-B0: середнє до 3 фото, SVD на 128, без кількості фото і без ваг класів.


## 17. Експеримент 10. Заморожений EfficientNet-B0 і SVD на 128

Останній блок ResNet дав val QWK 0.402 проти 0.464 у замороженого Exp5. 4.7 млн ваг зрушили ознаки ImageNet ще до того, як голова навчилась їх читати. Розморожувати більший шматок тим самим способом не будемо. Публічний файл лишається Exp5, score 0.6676.

Цей блок зберігає решту рецепту Exp5: середнє до 3 фото, SVD тексту на 128, голова з одним прихованим шаром, без ваг класів і без кількості фото. Змінюється енкодер. Заморожений EfficientNet-B0 дає довший візуальний вектор, ніж ResNet18, і під час навчання голови не змінюється. Ознаки пишуться в `efficientnet_b0_pet_features.npz` і не затирають кеші ResNet.

Як запускати на Kaggle: **не натискайте Run All**. Не перезапускайте комірки навчання попередніх розділів. Виконайте лише цей розділ зверху вниз. Друга комірка один раз читає фото; якщо кеш уже є, бере його. Далі навчається лише голова. Потрібні GPU і Internet, коли кешу ще немає.

Поріг сабміту: val QWK вищий за 0.4639 більше ніж на 0.02. Інакше чинним лишається файл зі score 0.6676.


In [ ]:
import os
import random
import re
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import cohen_kappa_score
from sklearn.model_selection import train_test_split

SEED = 42
VAL_SIZE = 0.2
MAX_IMAGES = 3
FEATURE_BATCH_SIZE = 64
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
DATA_ROOT = Path("/kaggle/input/competitions/deep-learning-for-computer-vision-and-nlp-2026-10")
IMAGE_DIR = DATA_ROOT / "images"
WORKING = Path("/kaggle/working")
WORKING.mkdir(parents=True, exist_ok=True)
NUM_WORKERS = 2 if os.name != "nt" else 0
if "device" not in globals():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

needs_tables = "train_df" not in globals() or "idx_tr" not in globals() or "y_all" not in globals()
if not needs_tables:
    print(f"Таблиці вже в пам'яті: train {len(train_df)}, val {len(idx_va)}. Повторно не читаю.")
else:
    TRAIN_CSV = DATA_ROOT / "train.csv"
    TEST_CSV = DATA_ROOT / "test.csv"
    SAMPLE_CSV = DATA_ROOT / "sample_submission.csv"

    def find_column(columns, options):
        lookup = {str(column).lower(): column for column in columns}
        for option in options:
            if option.lower() in lookup:
                return lookup[option.lower()]
        return None

    def normalize_id(value):
        if pd.isna(value):
            return ""
        text = str(value).strip()
        if text.endswith(".0"):
            text = text[:-2]
        return text

    def read_table(path):
        frame = pd.read_csv(path)
        frame.columns = [str(column).strip() for column in frame.columns]
        return frame

    def quadratic_weighted_kappa(y_true, y_pred):
        y_true = np.asarray(y_true).astype(int)
        y_pred = np.asarray(y_pred).astype(int)
        if len(y_true) == 0:
            return 0.0
        score = cohen_kappa_score(y_true, y_pred, weights="quadratic")
        if score != score:
            return 0.0
        return float(score)

    train_df = read_table(TRAIN_CSV)
    test_df = read_table(TEST_CSV)
    sample_df = read_table(SAMPLE_CSV)
    id_col = find_column(train_df.columns, ["PetID", "pet_id", "id"])
    target_col = find_column(train_df.columns, ["AdoptionSpeed", "adoption_speed", "label", "target"])
    text_col = find_column(train_df.columns, ["Description", "description", "desc", "text", "comments"])
    test_id_col = find_column(test_df.columns, ["PetID", "pet_id", "id"]) or id_col
    train_df[id_col] = train_df[id_col].map(normalize_id)
    test_df[test_id_col] = test_df[test_id_col].map(normalize_id)
    if int(train_df[id_col].duplicated().sum()):
        train_df = train_df.drop_duplicates(id_col, keep="first")
    train_df = train_df.reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)
    if target_col in test_df.columns:
        test_df = test_df.drop(columns=[target_col])
    test_text_col = find_column(test_df.columns, ["Description", "description", "desc", "text", "comments"])
    if text_col is None:
        text_col = "Description"
        train_df[text_col] = ""
        test_df[text_col] = ""
    elif test_text_col is None:
        test_df[text_col] = ""
    elif test_text_col != text_col:
        test_df[text_col] = test_df[test_text_col]
    raw_target = train_df[target_col]
    if pd.api.types.is_numeric_dtype(raw_target):
        numeric_target = raw_target.astype(float)
        if np.allclose(numeric_target.to_numpy(), np.rint(numeric_target.to_numpy())):
            train_df[target_col] = numeric_target.round().astype(int)
    classes = sorted(train_df[target_col].dropna().unique().tolist())
    LABEL_TO_IDX = {label: index for index, label in enumerate(classes)}
    IDX_TO_LABEL = {index: label for label, index in LABEL_TO_IDX.items()}
    N_CLASSES = len(classes)
    train_df = train_df.dropna(subset=[target_col]).reset_index(drop=True)
    y_all = train_df[target_col].map(LABEL_TO_IDX).to_numpy(dtype=int)
    idx_tr, idx_va = train_test_split(
        np.arange(len(train_df)), test_size=VAL_SIZE, random_state=SEED, stratify=y_all
    )
    y_tr = y_all[idx_tr]
    y_va = y_all[idx_va]
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    print(f"Таблиці прочитано заново: train {len(train_df)}, val {len(idx_va)}, test {len(test_df)}.")

if "train_ids" not in globals():
    train_ids = train_df[id_col].tolist()
if "test_ids" not in globals():
    test_ids = test_df[test_id_col].tolist()

if len(idx_va) != 1287:
    print(f"Увага: у валідації {len(idx_va)} рядків, у минулому прогоні було 1287.")
else:
    print("Розбиття збігається з минулим прогоном: 1287 валідаційних рядків.")

if "text_clean" in train_df.columns and "text_clean" in test_df.columns:
    print("Очищений опис уже є, повторно не чищу.")
else:
    NEGATION_TOKENS = {
        "no", "not", "nor", "never", "none", "nobody", "nothing",
        "neither", "nowhere", "without", "cannot",
    }
    CONTRACTIONS = {
        "ain't": "is not",
        "aren't": "are not",
        "can't": "cannot",
        "couldn't": "could not",
        "didn't": "did not",
        "doesn't": "does not",
        "don't": "do not",
        "hadn't": "had not",
        "hasn't": "has not",
        "haven't": "have not",
        "he's": "he is",
        "here's": "here is",
        "how's": "how is",
        "i'm": "i am",
        "i've": "i have",
        "isn't": "is not",
        "it's": "it is",
        "let's": "let us",
        "mustn't": "must not",
        "she's": "she is",
        "that's": "that is",
        "there's": "there is",
        "they're": "they are",
        "they've": "they have",
        "we're": "we are",
        "we've": "we have",
        "weren't": "were not",
        "what's": "what is",
        "where's": "where is",
        "who's": "who is",
        "won't": "will not",
        "wouldn't": "would not",
        "you're": "you are",
        "you've": "you have",
    }
    CONTRACTION_RE = re.compile(
        r"\b(" + "|".join(sorted(map(re.escape, CONTRACTIONS), key=len, reverse=True)) + r")\b"
    )
    FALLBACK_STOPWORDS = set(
        "a about above after again against all am an and any are as at be because been "
        "before being below between both but by can did do does doing down during each "
        "few for from further had has have having he her here hers herself him himself "
        "his how i if in into is it its itself just me more most my myself of off on "
        "once only or other our ours ourselves out over own same she should so some "
        "such than that the their theirs them themselves then there these they this "
        "those through to too under until up very was we were what when where which "
        "while who whom why will with you your yours yourself yourselves".split()
    )

    def load_stopwords():
        try:
            import nltk
            from nltk.corpus import stopwords

            nltk.download("stopwords", quiet=True)
            words = set(stopwords.words("english"))
            print("Стоп-слова: NLTK, english")
        except Exception as exc:
            print(f"Стоп-слова NLTK недоступні ({type(exc).__name__}: {exc}). Використовую вбудований список.")
            words = set(FALLBACK_STOPWORDS)
        return words - NEGATION_TOKENS

    STOPWORDS = load_stopwords()

    def clean_text(value):
        if not isinstance(value, str):
            return ""
        text = value.lower().replace("’", "'").replace("`", "'")
        text = CONTRACTION_RE.sub(lambda match: CONTRACTIONS[match.group(0)], text)
        text = re.sub(r"https?://\S+|www\.\S+", " ", text)
        text = re.sub(r"[^a-z\s]", " ", text)
        tokens = [token for token in text.split() if len(token) > 1 and token not in STOPWORDS]
        return " ".join(tokens)

    train_df["text_clean"] = train_df[text_col].fillna("").astype(str).map(clean_text)
    test_df["text_clean"] = test_df[text_col].fillna("").astype(str).map(clean_text)
    print("Порожніх описів після очищення:", float(train_df["text_clean"].eq("").mean()))

print("Пристрій:", device)


In [ ]:
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision.models import EfficientNet_B0_Weights, efficientnet_b0
from tqdm.auto import tqdm

PHOTO_CAP = 3
FEATURE_BATCH_SIZE = 64
CACHE_NAME = "efficientnet_b0_pet_features.npz"

reuse_backbone = (
    globals().get("IMAGE_BACKBONE") == "efficientnet_b0"
    and "image_combined" in globals()
    and "image_combined_test" in globals()
    and len(image_combined) == len(train_df)
    and len(image_combined_test) == len(test_ids)
)
if reuse_backbone:
    print(
        "Вектори EfficientNet-B0 уже в пам'яті:",
        tuple(image_combined.shape),
        tuple(image_combined_test.shape),
    )
else:
    def list_image_files(image_dir):
        files = []
        for path in Path(image_dir).rglob("*"):
            if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS:
                files.append(path)
        return files

    def link_all_images(pet_ids, files):
        pet_set = set(pet_ids)
        grouped = {pet_id: [] for pet_id in pet_set}
        for path in files:
            stem = path.stem
            pet_id = None
            order = 0
            if stem in pet_set:
                pet_id = stem
            else:
                for separator in ("-", "_"):
                    if separator not in stem:
                        continue
                    base, rest = stem.rsplit(separator, 1)
                    if base in pet_set:
                        pet_id = base
                        order = int(rest) if rest.isdigit() else 10**6
                        break
            if pet_id is not None:
                grouped[pet_id].append((order, path))
        linked = {}
        for pet_id, items in grouped.items():
            items.sort(key=lambda item: (item[0], item[1].name))
            linked[pet_id] = [item_path for _, item_path in items]
        return linked

    class PetImageDataset(Dataset):
        def __init__(self, records, transform):
            self.records = records
            self.transform = transform

        def __len__(self):
            return len(self.records)

        def __getitem__(self, index):
            pet_id, path = self.records[index]
            try:
                with Image.open(path) as image:
                    tensor = self.transform(image.convert("RGB"))
                ok = 1
            except Exception:
                tensor = torch.zeros(3, 224, 224)
                ok = 0
            return tensor, ok, pet_id

    def extract_features(paths_by_pet, backbone, transform, dim):
        records = [(pet_id, path) for pet_id, paths in paths_by_pet.items() for path in paths[:PHOTO_CAP]]
        sums = {pet_id: np.zeros(dim, dtype=np.float32) for pet_id in paths_by_pet}
        counts = {pet_id: 0 for pet_id in paths_by_pet}
        if records:
            loader = DataLoader(
                PetImageDataset(records, transform),
                batch_size=FEATURE_BATCH_SIZE,
                shuffle=False,
                num_workers=NUM_WORKERS,
            )
            backbone.eval()
            for tensors, oks, pet_ids_batch in tqdm(loader, desc="EfficientNet-B0"):
                features = backbone(tensors.to(device)).detach().cpu().numpy()
                oks = oks.numpy()
                for index, pet_id in enumerate(pet_ids_batch):
                    if int(oks[index]) == 1:
                        sums[pet_id] += features[index]
                        counts[pet_id] += 1
        extracted = {}
        for pet_id in paths_by_pet:
            if counts[pet_id] > 0:
                extracted[pet_id] = (sums[pet_id] / counts[pet_id], counts[pet_id])
            else:
                extracted[pet_id] = (np.zeros(dim, dtype=np.float32), 0)
        return extracted

    def stack_features(pet_ids, feature_map, dim):
        vectors = np.zeros((len(pet_ids), dim), dtype=np.float32)
        missing = np.ones((len(pet_ids), 1), dtype=np.float32)
        for index, pet_id in enumerate(pet_ids):
            vector, count = feature_map[pet_id]
            if count > 0:
                vectors[index] = vector
                missing[index, 0] = 0.0
        return np.concatenate([vectors, missing], axis=1)

    def locate_cache():
        direct = WORKING / CACHE_NAME
        if direct.exists():
            return direct
        root = Path("/kaggle/input")
        if root.exists():
            found = sorted(root.rglob(CACHE_NAME))
            if found:
                print(f"Кеш знайдено серед вхідних даних: {found[0]}")
                return found[0]
        return None

    def load_feature_map(cache_path):
        cached = np.load(cache_path, allow_pickle=False)
        cached_ids = [normalize_id(pet_id) for pet_id in cached["ids"].tolist()]
        counts = np.array(cached["counts"], copy=True)
        matrix = np.array(cached["features"], dtype=np.float32, copy=True)
        cached.close()
        feature_map = {}
        for index, pet_id in enumerate(cached_ids):
            feature_map[pet_id] = (matrix[index].copy(), int(counts[index]))
        return feature_map, int(matrix.shape[1])

    needed = set(train_ids) | set(test_ids)
    cache_path = locate_cache()
    feature_map = None
    feat_dim = None
    if cache_path is not None:
        loaded, feat_dim = load_feature_map(cache_path)
        if set(loaded) >= needed:
            feature_map = loaded
            print(f"Ознаки EfficientNet-B0 завантажено з кешу: {cache_path}, розмір вектора {feat_dim}")
        else:
            print("Кеш EfficientNet-B0 не покриває ідентифікатори, рахую заново.")
    if feature_map is None:
        print("Кешу EfficientNet-B0 немає. Рахую заморожені ознаки, голову не навчаю.")
        try:
            weights = EfficientNet_B0_Weights.IMAGENET1K_V1
            backbone = efficientnet_b0(weights=weights)
            transform = weights.transforms()
        except Exception as exc:
            raise RuntimeError(
                "Не вдалося завантажити ваги EfficientNet-B0. Увімкніть Internet у налаштуваннях ноутбука."
            ) from exc
        backbone.classifier = nn.Identity()
        backbone.to(device)
        backbone.eval()
        for parameter in backbone.parameters():
            parameter.requires_grad = False
        with torch.no_grad():
            feat_dim = int(backbone(torch.zeros(1, 3, 224, 224, device=device)).shape[1])
        image_files = list_image_files(IMAGE_DIR)
        all_ids = list(dict.fromkeys(list(train_ids) + list(test_ids)))
        linked = link_all_images(all_ids, image_files)
        paths_by_pet = {pet_id: linked[pet_id][:PHOTO_CAP] for pet_id in all_ids}
        print(f"Файлів зображень: {len(image_files)}. У середнє входить до {PHOTO_CAP} фото.")
        feature_map = extract_features(paths_by_pet, backbone, transform, feat_dim)
        ids_array = np.array(list(feature_map.keys()))
        cache_out = WORKING / CACHE_NAME
        np.savez_compressed(
            cache_out,
            ids=ids_array,
            features=np.stack([feature_map[pet_id][0] for pet_id in ids_array]),
            counts=np.array([feature_map[pet_id][1] for pet_id in ids_array], dtype=np.int32),
        )
        print(f"Ознаки збережено: {cache_out}")
        del backbone
        if device.type == "cuda":
            torch.cuda.empty_cache()

    image_combined = stack_features(train_ids, feature_map, feat_dim)
    image_combined_test = stack_features(test_ids, feature_map, feat_dim)
    IMAGE_BACKBONE = "efficientnet_b0"

print("Матриця EfficientNet train/test:", tuple(image_combined.shape), tuple(image_combined_test.shape))


In [ ]:
import copy

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, Dataset

if globals().get("IMAGE_BACKBONE") != "efficientnet_b0":
    raise RuntimeError("Спочатку виконайте комірку ознак EfficientNet-B0. Вектори ResNet для цього кроку не годяться.")

SVD_DIM = 128
JOINT_HIDDEN = 128
JOINT_DROPOUT = 0.3
JOINT_LR = 1e-3
JOINT_WEIGHT_DECAY = 1e-4
JOINT_MAX_EPOCHS = 8
JOINT_PATIENCE = 2
JOINT_MIN_DELTA = 0.001
JOINT_BATCH = 64
EXP5_VAL_QWK = 0.4639
EXP9_VAL_QWK = 0.4017
SUBMISSION_MARGIN = 0.02
TFIDF_MAX_FEATURES = 20000
TFIDF_MIN_DF = 2


class JointDataset(Dataset):
    def __init__(self, features, labels=None):
        self.features = torch.tensor(np.asarray(features), dtype=torch.float32)
        self.labels = None if labels is None else torch.tensor(np.asarray(labels), dtype=torch.long)

    def __len__(self):
        return self.features.size(0)

    def __getitem__(self, index):
        if self.labels is None:
            return self.features[index]
        return self.features[index], self.labels[index]


class JointHead(nn.Module):
    def __init__(self, in_dim, hidden, n_classes, dropout=JOINT_DROPOUT):
        super().__init__()
        self.net = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, n_classes),
        )

    def forward(self, features):
        return self.net(features)


def text_matrices(fit_text, groups):
    vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=TFIDF_MIN_DF, max_features=TFIDF_MAX_FEATURES)
    sparse_fit = vectorizer.fit_transform(fit_text)
    svd_dim = min(SVD_DIM, sparse_fit.shape[1] - 1, sparse_fit.shape[0] - 1)
    svd = TruncatedSVD(n_components=svd_dim, random_state=SEED)
    svd.fit(sparse_fit)
    matrices = [svd.transform(vectorizer.transform(rows)).astype(np.float32) for rows in groups]
    explained = float(svd.explained_variance_ratio_.sum())
    return matrices, explained, sparse_fit.shape[1], svd_dim


def decode_probabilities(probs, n_classes):
    class_index = np.arange(n_classes, dtype=np.float64)
    expectation = np.clip(np.rint(probs @ class_index), 0, n_classes - 1).astype(int)
    return {"argmax": probs.argmax(axis=1).astype(int), "expectation": expectation}


def fit_joint_head(features_train, labels_train, features_val, labels_val):
    torch.manual_seed(SEED)
    model = JointHead(features_train.shape[1], JOINT_HIDDEN, N_CLASSES).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=JOINT_LR, weight_decay=JOINT_WEIGHT_DECAY)
    criterion = nn.CrossEntropyLoss()
    train_loader = DataLoader(
        JointDataset(features_train, labels_train),
        batch_size=JOINT_BATCH,
        shuffle=True,
        generator=torch.Generator().manual_seed(SEED),
    )
    val_loader = DataLoader(JointDataset(features_val, labels_val), batch_size=256, shuffle=False)
    best_qwk = -1.0
    best_epoch = 1
    best_decoder = "argmax"
    best_state = copy.deepcopy(model.state_dict())
    wait = 0
    epochs_ran = 0
    for epoch in range(1, JOINT_MAX_EPOCHS + 1):
        epochs_ran = epoch
        model.train()
        total_loss, seen = 0.0, 0
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            optimizer.zero_grad(set_to_none=True)
            logits = model(batch_x)
            loss = criterion(logits, batch_y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            total_loss += loss.item() * batch_y.size(0)
            seen += batch_y.size(0)
        model.eval()
        prob_chunks, true_chunks = [], []
        val_loss, val_seen = 0.0, 0
        with torch.no_grad():
            for batch_x, batch_y in val_loader:
                batch_x, batch_y = batch_x.to(device), batch_y.to(device)
                logits = model(batch_x)
                val_loss += criterion(logits, batch_y).item() * batch_y.size(0)
                val_seen += batch_y.size(0)
                prob_chunks.append(torch.softmax(logits, dim=1).cpu().numpy())
                true_chunks.append(batch_y.cpu().numpy())
        probs = np.concatenate(prob_chunks)
        y_true = np.concatenate(true_chunks)
        decoded = decode_probabilities(probs, N_CLASSES)
        qwk_argmax = quadratic_weighted_kappa(y_true, decoded["argmax"])
        qwk_expectation = quadratic_weighted_kappa(y_true, decoded["expectation"])
        epoch_decoder = "expectation" if qwk_expectation > qwk_argmax else "argmax"
        epoch_qwk = qwk_expectation if epoch_decoder == "expectation" else qwk_argmax
        print(
            f"Епоха {epoch:02d}/{JOINT_MAX_EPOCHS} | train loss {total_loss / max(seen, 1):.4f} | "
            f"val loss {val_loss / max(val_seen, 1):.4f} | "
            f"QWK argmax {qwk_argmax:.4f} | QWK expectation {qwk_expectation:.4f}"
        )
        if epoch_qwk > best_qwk + JOINT_MIN_DELTA:
            best_qwk = epoch_qwk
            best_epoch = epoch
            best_decoder = epoch_decoder
            best_state = copy.deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1
            if wait >= JOINT_PATIENCE:
                break
    model.load_state_dict(best_state)
    print(f"Зупинка після епохи {epochs_ran}. Найкраща епоха {best_epoch}, QWK {best_qwk:.4f} ({best_decoder}).")
    return model, best_epoch, best_decoder, epochs_ran


def predict_joint(model, features):
    model.eval()
    loader = DataLoader(JointDataset(features), batch_size=256, shuffle=False)
    chunks = []
    with torch.no_grad():
        for batch_x in loader:
            logits = model(batch_x.to(device))
            chunks.append(torch.softmax(logits, dim=1).cpu().numpy())
    return np.concatenate(chunks)


(text_train_matrix,), explained, n_tfidf, svd_dim = text_matrices(
    train_df.iloc[idx_tr]["text_clean"],
    [train_df["text_clean"]],
)
print(
    f"TF-IDF ознак {n_tfidf}, компонент SVD {svd_dim}, пояснена дисперсія {explained:.3f}. "
    "Ваг класів немає. Кількість фото в модель не входить."
)
scaler = StandardScaler()
fused_tr = scaler.fit_transform(
    np.concatenate([image_combined[idx_tr], text_train_matrix[idx_tr]], axis=1)
).astype(np.float32)
fused_va = scaler.transform(
    np.concatenate([image_combined[idx_va], text_train_matrix[idx_va]], axis=1)
).astype(np.float32)
print("Спільна матриця train/val:", fused_tr.shape, fused_va.shape)
print("Параметрів голови:", int(fused_tr.shape[1] * JOINT_HIDDEN + JOINT_HIDDEN + JOINT_HIDDEN * N_CLASSES + N_CLASSES))

joint_model, joint_best_epoch, joint_decoder, joint_epochs = fit_joint_head(fused_tr, y_tr, fused_va, y_va)
val_probs = predict_joint(joint_model, fused_va)
val_decoded = decode_probabilities(val_probs, N_CLASSES)
val_scores = {name: quadratic_weighted_kappa(y_va, pred) for name, pred in val_decoded.items()}
chosen_pred = val_decoded[joint_decoder]
print(
    f"Exp10 QWK argmax {val_scores['argmax']:.4f}, expectation {val_scores['expectation']:.4f}. "
    f"Обрано {joint_decoder}, accuracy {accuracy_score(y_va, chosen_pred):.4f}."
)
print("Прогнози на валідації:")
print(pd.Series(chosen_pred).map(IDX_TO_LABEL).value_counts().sort_index())
print(
    f"Порівняння: Exp5 {EXP5_VAL_QWK:.4f}, Exp9 {EXP9_VAL_QWK:.4f}, "
    f"EfficientNet-B0 {val_scores[joint_decoder]:.4f}."
)

exp10_name = "Exp10. EfficientNet-B0 заморожений + SVD 128"
exp10_notes = (
    f"без ваг класів; без кількості фото; svd {svd_dim}; пояснена дисперсія {explained:.3f}; "
    f"argmax {val_scores['argmax']:.4f}; expectation {val_scores['expectation']:.4f}"
)
if "log_experiment" in globals():
    log_experiment(exp10_name, val_scores[joint_decoder], joint_decoder, joint_epochs, joint_best_epoch, exp10_notes)
else:
    print(exp10_name, round(val_scores[joint_decoder], 4), joint_decoder, "епох", joint_epochs, "найкраща", joint_best_epoch)

if val_scores[joint_decoder] > EXP5_VAL_QWK + SUBMISSION_MARGIN:
    print("EfficientNet-B0 обігнав Exp5 на валідації більше ніж на 0.02. Навчаю голову на повному train і пишу submission.csv.")
    (text_full, text_test), _, _, _ = text_matrices(
        train_df["text_clean"],
        [train_df["text_clean"], test_df["text_clean"]],
    )
    final_scaler = StandardScaler()
    fused_all = final_scaler.fit_transform(np.concatenate([image_combined, text_full], axis=1)).astype(np.float32)
    fused_test = final_scaler.transform(np.concatenate([image_combined_test, text_test], axis=1)).astype(np.float32)
    torch.manual_seed(SEED)
    final_joint = JointHead(fused_all.shape[1], JOINT_HIDDEN, N_CLASSES).to(device)
    final_optimizer = torch.optim.AdamW(final_joint.parameters(), lr=JOINT_LR, weight_decay=JOINT_WEIGHT_DECAY)
    final_criterion = nn.CrossEntropyLoss()
    final_loader = DataLoader(
        JointDataset(fused_all, y_all),
        batch_size=JOINT_BATCH,
        shuffle=True,
        generator=torch.Generator().manual_seed(SEED),
    )
    final_epochs = max(int(joint_best_epoch), 1)
    final_joint.train()
    for epoch in range(1, final_epochs + 1):
        total_loss, seen = 0.0, 0
        for batch_x, batch_y in final_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            final_optimizer.zero_grad(set_to_none=True)
            logits = final_joint(batch_x)
            loss = final_criterion(logits, batch_y)
            loss.backward()
            nn.utils.clip_grad_norm_(final_joint.parameters(), 5.0)
            final_optimizer.step()
            total_loss += loss.item() * batch_y.size(0)
            seen += batch_y.size(0)
        print(f"Повне навчання голови, епоха {epoch}/{final_epochs}, train loss {total_loss / max(seen, 1):.4f}")
    test_probs = predict_joint(final_joint, fused_test)
    test_idx = decode_probabilities(test_probs, N_CLASSES)[joint_decoder]
    submission = pd.DataFrame({
        "PetID": [normalize_id(pet_id) for pet_id in test_ids],
        "AdoptionSpeed": [IDX_TO_LABEL[int(index)] for index in test_idx],
    })
    submission = submission.drop_duplicates("PetID", keep="first")
    submission["AdoptionSpeed"] = submission["AdoptionSpeed"].astype(int)
    submission = submission[["PetID", "AdoptionSpeed"]]
    submission.to_csv(WORKING / "submission.csv", index=False)
    print(f"Збережено submission.csv, рядків {len(submission)}.")
    print(submission["AdoptionSpeed"].value_counts().sort_index())
else:
    print(
        "Новий submission.csv не записано: val QWK не вищий за Exp5 більше ніж на 0.02. "
        "Публічний файл зі score 0.6676 лишається чинним."
    )


### Висновки експерименту 10

Заморожений EfficientNet-B0 і SVD на 128 дають спільну матрицю 1409 ознак і голову на 180 996 параметрів. На збережених вагах 4-ї епохи val QWK 0.474 за округленим очікуванням класу і 0.454 за argmax. Accuracy 0.414. По епохах argmax ішов 0.451, 0.463, 0.454, 0.454, 0.448, 0.474, очікування — 0.408, 0.451, 0.451, 0.474, 0.430, 0.467. Зупинка після 6-ї епохи, найкраща — 4-та. На 6-й епосі argmax теж сягнув 0.474, але на інших вагах і без приросту 0.001 до вже збереженого результату, тож ці ваги не лишились.

Val loss найнижчий на 3-й епосі, 1.179. На 4-й, де очікування дало 0.474, val loss уже 1.210, а train loss падав далі і на 6-й був 0.912.

Argmax цієї моделі, 0.454, нижчий за Exp5 (0.464). Число 0.474 належить декодеру очікування. Він ставить клас 1 у 86 випадках із 1287 (6.7% за частки 18.6% у train), клас 2 — 494 (38.4% за 27.6%), клас 3 — 519 (40.3% за 20.6%), клас 4 — 188 (14.6% за 33.2%). Прогнози зібрані в середині шкали, тому точних збігів мало, а далеких помилок теж менше, і каппа на цьому розбитті трохи вища.

Приріст над Exp5 — 0.010, менший за поріг 0.02. Такий самий за розміром приріст валідації в Exp6 на публічному тесті дав 0.6524 замість 0.6676. Новий файл не записано. Чинним лишається Exp5.

Наступний блок не міняє енкодер. Спочатку навчається голова на замороженому ResNet, як у Exp5, і лише потім останній блок отримує кілька кроків зі швидкістю 1e-5. У Exp9 блок і ще порожня голова вчилися разом, тому ознаки зіпсувались уже на першій епосі.


## 18. Експеримент 11. Спочатку голова, потім останній блок

У EfficientNet-B0 число 0.474 належить декодеру очікування. Argmax на тих самих вагах — 0.454, нижче за 0.464 у Exp5. Очікування зібрало прогнози в класах 2 і 3, а val loss у мить цього рекорду вже ріс. Приріст 0.010 менший за поріг, і такий самий приріст у Exp6 на публічному тесті не підтвердився. Чинний файл лишається Exp5, score 0.6676.

Exp9 зіпсував ознаки, бо останній блок і ще порожня голова вчилися разом. Тут порядок інший. Спочатку голова вчиться на замороженому ResNet18 і SVD на 128, як у Exp5. Потім той самий останній блок отримує не більше 4 епох зі швидкістю 1e-5, голова — 1e-4. Якщо ці кроки не піднімають val QWK над замороженою головою цього ж прогону, у підсумок іде вона.

Як запускати на Kaggle: **не натискайте Run All**. Не перезапускайте комірки навчання попередніх розділів. Виконайте лише цей розділ зверху вниз. Друга комірка бере кеш `resnet18_pet_features.npz` або один раз рахує заморожені ознаки. Третя комірка після цього читає фото лише на етапі донавчання блока. Потрібні GPU і Internet, якщо кешу ResNet немає.

Поріг сабміту: підсумковий val QWK вищий за 0.4639 більше ніж на 0.02. Інакше чинним лишається файл зі score 0.6676.


In [ ]:
import os
import random
import re
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import cohen_kappa_score
from sklearn.model_selection import train_test_split

SEED = 42
VAL_SIZE = 0.2
MAX_IMAGES = 3
FEATURE_BATCH_SIZE = 64
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
DATA_ROOT = Path("/kaggle/input/competitions/deep-learning-for-computer-vision-and-nlp-2026-10")
IMAGE_DIR = DATA_ROOT / "images"
WORKING = Path("/kaggle/working")
WORKING.mkdir(parents=True, exist_ok=True)
NUM_WORKERS = 2 if os.name != "nt" else 0
if "device" not in globals():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

needs_tables = "train_df" not in globals() or "idx_tr" not in globals() or "y_all" not in globals()
if not needs_tables:
    print(f"Таблиці вже в пам'яті: train {len(train_df)}, val {len(idx_va)}. Повторно не читаю.")
else:
    TRAIN_CSV = DATA_ROOT / "train.csv"
    TEST_CSV = DATA_ROOT / "test.csv"
    SAMPLE_CSV = DATA_ROOT / "sample_submission.csv"

    def find_column(columns, options):
        lookup = {str(column).lower(): column for column in columns}
        for option in options:
            if option.lower() in lookup:
                return lookup[option.lower()]
        return None

    def normalize_id(value):
        if pd.isna(value):
            return ""
        text = str(value).strip()
        if text.endswith(".0"):
            text = text[:-2]
        return text

    def read_table(path):
        frame = pd.read_csv(path)
        frame.columns = [str(column).strip() for column in frame.columns]
        return frame

    def quadratic_weighted_kappa(y_true, y_pred):
        y_true = np.asarray(y_true).astype(int)
        y_pred = np.asarray(y_pred).astype(int)
        if len(y_true) == 0:
            return 0.0
        score = cohen_kappa_score(y_true, y_pred, weights="quadratic")
        if score != score:
            return 0.0
        return float(score)

    train_df = read_table(TRAIN_CSV)
    test_df = read_table(TEST_CSV)
    sample_df = read_table(SAMPLE_CSV)
    id_col = find_column(train_df.columns, ["PetID", "pet_id", "id"])
    target_col = find_column(train_df.columns, ["AdoptionSpeed", "adoption_speed", "label", "target"])
    text_col = find_column(train_df.columns, ["Description", "description", "desc", "text", "comments"])
    test_id_col = find_column(test_df.columns, ["PetID", "pet_id", "id"]) or id_col
    train_df[id_col] = train_df[id_col].map(normalize_id)
    test_df[test_id_col] = test_df[test_id_col].map(normalize_id)
    if int(train_df[id_col].duplicated().sum()):
        train_df = train_df.drop_duplicates(id_col, keep="first")
    train_df = train_df.reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)
    if target_col in test_df.columns:
        test_df = test_df.drop(columns=[target_col])
    test_text_col = find_column(test_df.columns, ["Description", "description", "desc", "text", "comments"])
    if text_col is None:
        text_col = "Description"
        train_df[text_col] = ""
        test_df[text_col] = ""
    elif test_text_col is None:
        test_df[text_col] = ""
    elif test_text_col != text_col:
        test_df[text_col] = test_df[test_text_col]
    raw_target = train_df[target_col]
    if pd.api.types.is_numeric_dtype(raw_target):
        numeric_target = raw_target.astype(float)
        if np.allclose(numeric_target.to_numpy(), np.rint(numeric_target.to_numpy())):
            train_df[target_col] = numeric_target.round().astype(int)
    classes = sorted(train_df[target_col].dropna().unique().tolist())
    LABEL_TO_IDX = {label: index for index, label in enumerate(classes)}
    IDX_TO_LABEL = {index: label for label, index in LABEL_TO_IDX.items()}
    N_CLASSES = len(classes)
    train_df = train_df.dropna(subset=[target_col]).reset_index(drop=True)
    y_all = train_df[target_col].map(LABEL_TO_IDX).to_numpy(dtype=int)
    idx_tr, idx_va = train_test_split(
        np.arange(len(train_df)), test_size=VAL_SIZE, random_state=SEED, stratify=y_all
    )
    y_tr = y_all[idx_tr]
    y_va = y_all[idx_va]
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    print(f"Таблиці прочитано заново: train {len(train_df)}, val {len(idx_va)}, test {len(test_df)}.")

if "train_ids" not in globals():
    train_ids = train_df[id_col].tolist()
if "test_ids" not in globals():
    test_ids = test_df[test_id_col].tolist()

if len(idx_va) != 1287:
    print(f"Увага: у валідації {len(idx_va)} рядків, у минулому прогоні було 1287.")
else:
    print("Розбиття збігається з минулим прогоном: 1287 валідаційних рядків.")

if "text_clean" in train_df.columns and "text_clean" in test_df.columns:
    print("Очищений опис уже є, повторно не чищу.")
else:
    NEGATION_TOKENS = {
        "no", "not", "nor", "never", "none", "nobody", "nothing",
        "neither", "nowhere", "without", "cannot",
    }
    CONTRACTIONS = {
        "ain't": "is not",
        "aren't": "are not",
        "can't": "cannot",
        "couldn't": "could not",
        "didn't": "did not",
        "doesn't": "does not",
        "don't": "do not",
        "hadn't": "had not",
        "hasn't": "has not",
        "haven't": "have not",
        "he's": "he is",
        "here's": "here is",
        "how's": "how is",
        "i'm": "i am",
        "i've": "i have",
        "isn't": "is not",
        "it's": "it is",
        "let's": "let us",
        "mustn't": "must not",
        "she's": "she is",
        "that's": "that is",
        "there's": "there is",
        "they're": "they are",
        "they've": "they have",
        "we're": "we are",
        "we've": "we have",
        "weren't": "were not",
        "what's": "what is",
        "where's": "where is",
        "who's": "who is",
        "won't": "will not",
        "wouldn't": "would not",
        "you're": "you are",
        "you've": "you have",
    }
    CONTRACTION_RE = re.compile(
        r"\b(" + "|".join(sorted(map(re.escape, CONTRACTIONS), key=len, reverse=True)) + r")\b"
    )
    FALLBACK_STOPWORDS = set(
        "a about above after again against all am an and any are as at be because been "
        "before being below between both but by can did do does doing down during each "
        "few for from further had has have having he her here hers herself him himself "
        "his how i if in into is it its itself just me more most my myself of off on "
        "once only or other our ours ourselves out over own same she should so some "
        "such than that the their theirs them themselves then there these they this "
        "those through to too under until up very was we were what when where which "
        "while who whom why will with you your yours yourself yourselves".split()
    )

    def load_stopwords():
        try:
            import nltk
            from nltk.corpus import stopwords

            nltk.download("stopwords", quiet=True)
            words = set(stopwords.words("english"))
            print("Стоп-слова: NLTK, english")
        except Exception as exc:
            print(f"Стоп-слова NLTK недоступні ({type(exc).__name__}: {exc}). Використовую вбудований список.")
            words = set(FALLBACK_STOPWORDS)
        return words - NEGATION_TOKENS

    STOPWORDS = load_stopwords()

    def clean_text(value):
        if not isinstance(value, str):
            return ""
        text = value.lower().replace("’", "'").replace("`", "'")
        text = CONTRACTION_RE.sub(lambda match: CONTRACTIONS[match.group(0)], text)
        text = re.sub(r"https?://\S+|www\.\S+", " ", text)
        text = re.sub(r"[^a-z\s]", " ", text)
        tokens = [token for token in text.split() if len(token) > 1 and token not in STOPWORDS]
        return " ".join(tokens)

    train_df["text_clean"] = train_df[text_col].fillna("").astype(str).map(clean_text)
    test_df["text_clean"] = test_df[text_col].fillna("").astype(str).map(clean_text)
    print("Порожніх описів після очищення:", float(train_df["text_clean"].eq("").mean()))

print("Пристрій:", device)


In [ ]:
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision.models import ResNet18_Weights, resnet18
from tqdm.auto import tqdm

PHOTO_CAP = 3
FEATURE_BATCH_SIZE = 64
CACHE_NAME = "resnet18_pet_features.npz"


def list_image_files(image_dir):
    files = []
    for path in Path(image_dir).rglob("*"):
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS:
            files.append(path)
    return files


def link_all_images(pet_ids, files):
    pet_set = set(pet_ids)
    grouped = {pet_id: [] for pet_id in pet_set}
    for path in files:
        stem = path.stem
        pet_id = None
        order = 0
        if stem in pet_set:
            pet_id = stem
        else:
            for separator in ("-", "_"):
                if separator not in stem:
                    continue
                base, rest = stem.rsplit(separator, 1)
                if base in pet_set:
                    pet_id = base
                    order = int(rest) if rest.isdigit() else 10**6
                    break
        if pet_id is not None:
            grouped[pet_id].append((order, path))
    linked = {}
    for pet_id, items in grouped.items():
        items.sort(key=lambda item: (item[0], item[1].name))
        linked[pet_id] = [item_path for _, item_path in items]
    return linked


image_files = list_image_files(IMAGE_DIR)
all_ids = list(dict.fromkeys(list(train_ids) + list(test_ids)))
linked_photos = link_all_images(all_ids, image_files)
train_photo_rows = [linked_photos[pet_id][:PHOTO_CAP] for pet_id in train_ids]
test_photo_rows = [linked_photos[pet_id][:PHOTO_CAP] for pet_id in test_ids]
print(f"Файлів зображень: {len(image_files)}. У середнє і в донавчання входить до {PHOTO_CAP} фото.")

reuse_resnet = (
    "resnet_combined" in globals()
    and "resnet_combined_test" in globals()
    and getattr(resnet_combined, "shape", (0, 0))[1] == 513
    and len(resnet_combined) == len(train_df)
    and len(resnet_combined_test) == len(test_ids)
)
if reuse_resnet:
    print("Вектори ResNet18 на 3 фото вже в пам'яті:", tuple(resnet_combined.shape))
else:
    class PetImageDataset(Dataset):
        def __init__(self, records, transform):
            self.records = records
            self.transform = transform

        def __len__(self):
            return len(self.records)

        def __getitem__(self, index):
            pet_id, path = self.records[index]
            try:
                with Image.open(path) as image:
                    tensor = self.transform(image.convert("RGB"))
                ok = 1
            except Exception:
                tensor = torch.zeros(3, 224, 224)
                ok = 0
            return tensor, ok, pet_id

    def extract_features(paths_by_pet, backbone, transform, dim):
        records = [(pet_id, path) for pet_id, paths in paths_by_pet.items() for path in paths]
        sums = {pet_id: np.zeros(dim, dtype=np.float32) for pet_id in paths_by_pet}
        counts = {pet_id: 0 for pet_id in paths_by_pet}
        if records:
            loader = DataLoader(
                PetImageDataset(records, transform),
                batch_size=FEATURE_BATCH_SIZE,
                shuffle=False,
                num_workers=NUM_WORKERS,
            )
            backbone.eval()
            for tensors, oks, pet_ids_batch in tqdm(loader, desc="ResNet18"):
                features = backbone(tensors.to(device)).detach().cpu().numpy()
                oks = oks.numpy()
                for index, pet_id in enumerate(pet_ids_batch):
                    if int(oks[index]) == 1:
                        sums[pet_id] += features[index]
                        counts[pet_id] += 1
        extracted = {}
        for pet_id in paths_by_pet:
            if counts[pet_id] > 0:
                extracted[pet_id] = (sums[pet_id] / counts[pet_id], counts[pet_id])
            else:
                extracted[pet_id] = (np.zeros(dim, dtype=np.float32), 0)
        return extracted

    def stack_features(pet_ids, feature_map, dim):
        vectors = np.zeros((len(pet_ids), dim), dtype=np.float32)
        missing = np.ones((len(pet_ids), 1), dtype=np.float32)
        for index, pet_id in enumerate(pet_ids):
            vector, count = feature_map[pet_id]
            if count > 0:
                vectors[index] = vector
                missing[index, 0] = 0.0
        return np.concatenate([vectors, missing], axis=1)

    def locate_cache():
        direct = WORKING / CACHE_NAME
        if direct.exists():
            return direct
        root = Path("/kaggle/input")
        if root.exists():
            found = sorted(root.rglob(CACHE_NAME))
            if found:
                print(f"Кеш знайдено серед вхідних даних: {found[0]}")
                return found[0]
        return None

    def load_feature_map(cache_path):
        cached = np.load(cache_path, allow_pickle=False)
        cached_ids = [normalize_id(pet_id) for pet_id in cached["ids"].tolist()]
        counts = np.array(cached["counts"], copy=True)
        matrix = np.array(cached["features"], dtype=np.float32, copy=True)
        cached.close()
        feature_map = {}
        for index, pet_id in enumerate(cached_ids):
            feature_map[pet_id] = (matrix[index].copy(), int(counts[index]))
        return feature_map, int(matrix.shape[1])

    needed = set(train_ids) | set(test_ids)
    cache_path = locate_cache()
    feature_map = None
    feat_dim = None
    if cache_path is not None:
        loaded, feat_dim = load_feature_map(cache_path)
        if feat_dim == 512 and set(loaded) >= needed:
            feature_map = loaded
            print(f"Ознаки ResNet18 завантажено з кешу: {cache_path}")
        else:
            print("Кеш ResNet не підходить для цього кроку, рахую заново.")
    if feature_map is None:
        print("Кешу ResNet18 немає. Рахую заморожені ознаки трьох фото.")
        try:
            weights = ResNet18_Weights.IMAGENET1K_V1
            backbone = resnet18(weights=weights)
            transform = weights.transforms()
        except Exception as exc:
            raise RuntimeError(
                "Не вдалося завантажити ваги ResNet18. Увімкніть Internet у налаштуваннях ноутбука."
            ) from exc
        backbone.fc = nn.Identity()
        backbone.to(device)
        backbone.eval()
        for parameter in backbone.parameters():
            parameter.requires_grad = False
        with torch.no_grad():
            feat_dim = int(backbone(torch.zeros(1, 3, 224, 224, device=device)).shape[1])
        paths_by_pet = {pet_id: linked_photos[pet_id][:PHOTO_CAP] for pet_id in all_ids}
        feature_map = extract_features(paths_by_pet, backbone, transform, feat_dim)
        ids_array = np.array(list(feature_map.keys()))
        cache_out = WORKING / CACHE_NAME
        np.savez_compressed(
            cache_out,
            ids=ids_array,
            features=np.stack([feature_map[pet_id][0] for pet_id in ids_array]),
            counts=np.array([feature_map[pet_id][1] for pet_id in ids_array], dtype=np.int32),
        )
        print(f"Ознаки збережено: {cache_out}")
        del backbone
        if device.type == "cuda":
            torch.cuda.empty_cache()
    resnet_combined = stack_features(train_ids, feature_map, feat_dim)
    resnet_combined_test = stack_features(test_ids, feature_map, feat_dim)

print("Матриця ResNet train/test:", tuple(resnet_combined.shape), tuple(resnet_combined_test.shape))


In [ ]:
import copy

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, Dataset
from torchvision.models import ResNet18_Weights, resnet18

if "resnet_combined" not in globals() or "train_photo_rows" not in globals():
    raise RuntimeError("Спочатку виконайте комірку ознак ResNet цього розділу.")
if resnet_combined.shape[1] != 513:
    raise RuntimeError(f"Очікував вектор 513, отримав {resnet_combined.shape[1]}.")

SVD_DIM = 128
JOINT_HIDDEN = 128
JOINT_DROPOUT = 0.3
HEAD_LR = 1e-3
FINE_HEAD_LR = 1e-4
FINE_BLOCK_LR = 1e-5
WEIGHT_DECAY = 1e-4
HEAD_EPOCHS = 8
FINE_EPOCHS = 4
PATIENCE = 2
MIN_DELTA = 0.001
HEAD_BATCH = 64
FINE_BATCH = 16
PHOTO_CAP = 3
EXP5_VAL_QWK = 0.4639
SUBMISSION_MARGIN = 0.02
TFIDF_MAX_FEATURES = 20000
TFIDF_MIN_DF = 2


class VecDataset(Dataset):
    def __init__(self, features, labels=None):
        self.features = torch.tensor(np.asarray(features), dtype=torch.float32)
        self.labels = None if labels is None else torch.tensor(np.asarray(labels), dtype=torch.long)

    def __len__(self):
        return self.features.size(0)

    def __getitem__(self, index):
        if self.labels is None:
            return self.features[index]
        return self.features[index], self.labels[index]


class JointHead(nn.Module):
    def __init__(self, in_dim, hidden, n_classes, dropout=JOINT_DROPOUT):
        super().__init__()
        self.net = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, n_classes),
        )

    def forward(self, features):
        return self.net(features)


class PetTuneDataset(Dataset):
    def __init__(self, row_index, text_matrix, path_rows, transform, labels=None):
        self.row_index = np.asarray(row_index, dtype=int)
        self.text_matrix = np.asarray(text_matrix, dtype=np.float32)
        self.path_rows = path_rows
        self.transform = transform
        self.labels = None if labels is None else np.asarray(labels, dtype=np.int64)

    def __len__(self):
        return len(self.row_index)

    def __getitem__(self, index):
        row = int(self.row_index[index])
        frames = []
        for path in self.path_rows[row]:
            try:
                with Image.open(path) as image:
                    frames.append(self.transform(image.convert("RGB")))
            except Exception:
                continue
        valid = len(frames)
        while len(frames) < PHOTO_CAP:
            frames.append(torch.zeros(3, 224, 224))
        item = {
            "images": torch.stack(frames[:PHOTO_CAP]),
            "valid": valid,
            "text": torch.tensor(self.text_matrix[row]),
        }
        if self.labels is not None:
            item["label"] = int(self.labels[row])
        return item


def text_matrices(fit_text, groups):
    vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=TFIDF_MIN_DF, max_features=TFIDF_MAX_FEATURES)
    sparse_fit = vectorizer.fit_transform(fit_text)
    svd_dim = min(SVD_DIM, sparse_fit.shape[1] - 1, sparse_fit.shape[0] - 1)
    svd = TruncatedSVD(n_components=svd_dim, random_state=SEED)
    svd.fit(sparse_fit)
    matrices = [svd.transform(vectorizer.transform(rows)).astype(np.float32) for rows in groups]
    explained = float(svd.explained_variance_ratio_.sum())
    return matrices, explained, sparse_fit.shape[1], svd_dim


def decode_probabilities(probs, n_classes):
    class_index = np.arange(n_classes, dtype=np.float64)
    expectation = np.clip(np.rint(probs @ class_index), 0, n_classes - 1).astype(int)
    return {"argmax": probs.argmax(axis=1).astype(int), "expectation": expectation}


def fit_frozen_head(features_train, labels_train, features_val, labels_val):
    torch.manual_seed(SEED)
    model = JointHead(features_train.shape[1], JOINT_HIDDEN, N_CLASSES).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=HEAD_LR, weight_decay=WEIGHT_DECAY)
    criterion = nn.CrossEntropyLoss()
    train_loader = DataLoader(
        VecDataset(features_train, labels_train),
        batch_size=HEAD_BATCH,
        shuffle=True,
        generator=torch.Generator().manual_seed(SEED),
    )
    val_loader = DataLoader(VecDataset(features_val, labels_val), batch_size=256, shuffle=False)
    best_qwk = -1.0
    best_epoch = 1
    best_decoder = "argmax"
    best_state = copy.deepcopy(model.state_dict())
    wait = 0
    epochs_ran = 0
    for epoch in range(1, HEAD_EPOCHS + 1):
        epochs_ran = epoch
        model.train()
        total_loss, seen = 0.0, 0
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            optimizer.zero_grad(set_to_none=True)
            logits = model(batch_x)
            loss = criterion(logits, batch_y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            total_loss += loss.item() * batch_y.size(0)
            seen += batch_y.size(0)
        model.eval()
        prob_chunks, true_chunks = [], []
        val_loss, val_seen = 0.0, 0
        with torch.no_grad():
            for batch_x, batch_y in val_loader:
                batch_x, batch_y = batch_x.to(device), batch_y.to(device)
                logits = model(batch_x)
                val_loss += criterion(logits, batch_y).item() * batch_y.size(0)
                val_seen += batch_y.size(0)
                prob_chunks.append(torch.softmax(logits, dim=1).cpu().numpy())
                true_chunks.append(batch_y.cpu().numpy())
        probs = np.concatenate(prob_chunks)
        y_true = np.concatenate(true_chunks)
        decoded = decode_probabilities(probs, N_CLASSES)
        qwk_argmax = quadratic_weighted_kappa(y_true, decoded["argmax"])
        qwk_expectation = quadratic_weighted_kappa(y_true, decoded["expectation"])
        epoch_decoder = "expectation" if qwk_expectation > qwk_argmax else "argmax"
        epoch_qwk = qwk_expectation if epoch_decoder == "expectation" else qwk_argmax
        print(
            f"Голова {epoch:02d}/{HEAD_EPOCHS} | train loss {total_loss / max(seen, 1):.4f} | "
            f"val loss {val_loss / max(val_seen, 1):.4f} | "
            f"QWK argmax {qwk_argmax:.4f} | QWK expectation {qwk_expectation:.4f}"
        )
        if epoch_qwk > best_qwk + MIN_DELTA:
            best_qwk = epoch_qwk
            best_epoch = epoch
            best_decoder = epoch_decoder
            best_state = copy.deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1
            if wait >= PATIENCE:
                break
    model.load_state_dict(best_state)
    print(f"Заморожена голова: зупинка після {epochs_ran}, найкраща {best_epoch}, QWK {best_qwk:.4f} ({best_decoder}).")
    return model, best_qwk, best_epoch, best_decoder, epochs_ran


def predict_vectors(model, features):
    model.eval()
    loader = DataLoader(VecDataset(features), batch_size=256, shuffle=False)
    chunks = []
    with torch.no_grad():
        for batch_x in loader:
            logits = model(batch_x.to(device))
            chunks.append(torch.softmax(logits, dim=1).cpu().numpy())
    return np.concatenate(chunks)


def make_backbone():
    weights = ResNet18_Weights.IMAGENET1K_V1
    backbone = resnet18(weights=weights)
    transform = weights.transforms()
    backbone.fc = nn.Identity()
    for parameter in backbone.parameters():
        parameter.requires_grad = False
    for parameter in backbone.layer4[-1].parameters():
        parameter.requires_grad = True
    return backbone.to(device), transform


def pool_images(backbone, images, valid):
    batch, slots, channels, height, width = images.shape
    flat = images.reshape(batch * slots, channels, height, width)
    features = backbone(flat).reshape(batch, slots, -1)
    positions = torch.arange(slots, device=images.device).view(1, slots)
    mask = positions < valid.view(batch, 1)
    features = features * mask.unsqueeze(-1)
    pooled = features.sum(dim=1) / valid.clamp(min=1).view(batch, 1).float()
    missing = (valid == 0).float().unsqueeze(1)
    pooled = pooled * (1.0 - missing)
    return torch.cat([pooled, missing], dim=1)


def batch_logits(backbone, head, batch, scale_mean, scale_div):
    images = batch["images"].to(device)
    valid = batch["valid"].to(device)
    text = batch["text"].to(device)
    visual = pool_images(backbone, images, valid)
    raw = torch.cat([visual, text], dim=1)
    scaled = (raw - scale_mean) / scale_div
    return head(scaled)


def scores_from_loader(backbone, head, loader, scale_mean, scale_div):
    backbone.eval()
    head.eval()
    prob_chunks, true_chunks = [], []
    with torch.no_grad():
        for batch in loader:
            logits = batch_logits(backbone, head, batch, scale_mean, scale_div)
            prob_chunks.append(torch.softmax(logits, dim=1).cpu().numpy())
            true_chunks.append(batch["label"].numpy())
    probs = np.concatenate(prob_chunks)
    y_true = np.concatenate(true_chunks)
    decoded = decode_probabilities(probs, N_CLASSES)
    qwk_argmax = quadratic_weighted_kappa(y_true, decoded["argmax"])
    qwk_expectation = quadratic_weighted_kappa(y_true, decoded["expectation"])
    decoder = "expectation" if qwk_expectation > qwk_argmax else "argmax"
    qwk = qwk_expectation if decoder == "expectation" else qwk_argmax
    return qwk, decoder, qwk_argmax, qwk_expectation, probs


(text_train_matrix,), explained, n_tfidf, svd_dim = text_matrices(
    train_df.iloc[idx_tr]["text_clean"],
    [train_df["text_clean"]],
)
print(
    f"TF-IDF ознак {n_tfidf}, компонент SVD {svd_dim}, пояснена дисперсія {explained:.3f}. "
    "Ваг класів немає."
)
scaler = StandardScaler()
fused_tr = scaler.fit_transform(
    np.concatenate([resnet_combined[idx_tr], text_train_matrix[idx_tr]], axis=1)
).astype(np.float32)
fused_va = scaler.transform(
    np.concatenate([resnet_combined[idx_va], text_train_matrix[idx_va]], axis=1)
).astype(np.float32)
frozen_model, frozen_qwk, frozen_epoch, frozen_decoder, frozen_epochs = fit_frozen_head(
    fused_tr, y_tr, fused_va, y_va
)
frozen_probs = predict_vectors(frozen_model, fused_va)
frozen_decoded = decode_probabilities(frozen_probs, N_CLASSES)
print(
    "Заморожена голова, accuracy",
    f"{accuracy_score(y_va, frozen_decoded[frozen_decoder]):.4f}.",
)

scale_mean = torch.tensor(scaler.mean_, dtype=torch.float32, device=device)
scale_div = torch.tensor(np.clip(scaler.scale_, 1e-6, None), dtype=torch.float32, device=device)
backbone, image_transform = make_backbone()
tune_head = JointHead(fused_tr.shape[1], JOINT_HIDDEN, N_CLASSES).to(device)
tune_head.load_state_dict(frozen_model.state_dict())
block_params = sum(parameter.numel() for parameter in backbone.layer4[-1].parameters() if parameter.requires_grad)
print(f"Донавчаю останній блок, {block_params} параметрів, швидкість {FINE_BLOCK_LR}.")
tune_loader = DataLoader(
    PetTuneDataset(idx_tr, text_train_matrix, train_photo_rows, image_transform, y_all),
    batch_size=FINE_BATCH,
    shuffle=True,
    num_workers=NUM_WORKERS,
    generator=torch.Generator().manual_seed(SEED),
)
tune_val_loader = DataLoader(
    PetTuneDataset(idx_va, text_train_matrix, train_photo_rows, image_transform, y_all),
    batch_size=FINE_BATCH,
    shuffle=False,
    num_workers=NUM_WORKERS,
)
optimizer = torch.optim.AdamW(
    [
        {"params": backbone.layer4[-1].parameters(), "lr": FINE_BLOCK_LR},
        {"params": tune_head.parameters(), "lr": FINE_HEAD_LR},
    ],
    weight_decay=WEIGHT_DECAY,
)
criterion = nn.CrossEntropyLoss()
best_tune_qwk = frozen_qwk
best_tune_decoder = frozen_decoder
best_tune_epoch = 0
best_block = None
best_head = None
wait = 0
tune_epochs = 0
for epoch in range(1, FINE_EPOCHS + 1):
    tune_epochs = epoch
    backbone.eval()
    backbone.layer4[-1].train()
    tune_head.train()
    total_loss, seen = 0.0, 0
    for batch in tune_loader:
        labels = batch["label"].to(device)
        optimizer.zero_grad(set_to_none=True)
        logits = batch_logits(backbone, tune_head, batch, scale_mean, scale_div)
        loss = criterion(logits, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(
            [parameter for parameter in list(backbone.layer4[-1].parameters()) + list(tune_head.parameters()) if parameter.requires_grad],
            5.0,
        )
        optimizer.step()
        total_loss += loss.item() * labels.size(0)
        seen += labels.size(0)
    epoch_qwk, epoch_decoder, qwk_argmax, qwk_expectation, _ = scores_from_loader(
        backbone, tune_head, tune_val_loader, scale_mean, scale_div
    )
    print(
        f"Блок {epoch:02d}/{FINE_EPOCHS} | train loss {total_loss / max(seen, 1):.4f} | "
        f"QWK argmax {qwk_argmax:.4f} | QWK expectation {qwk_expectation:.4f}"
    )
    if epoch_qwk > best_tune_qwk + MIN_DELTA:
        best_tune_qwk = epoch_qwk
        best_tune_decoder = epoch_decoder
        best_tune_epoch = epoch
        best_block = copy.deepcopy(backbone.layer4[-1].state_dict())
        best_head = copy.deepcopy(tune_head.state_dict())
        wait = 0
    else:
        wait += 1
        if wait >= PATIENCE:
            break

use_tuned = best_block is not None
if use_tuned:
    backbone.layer4[-1].load_state_dict(best_block)
    tune_head.load_state_dict(best_head)
    final_qwk = best_tune_qwk
    final_decoder = best_tune_decoder
    print(f"Блок обігнав заморожену голову: {frozen_qwk:.4f} -> {final_qwk:.4f}, епоха {best_tune_epoch}.")
else:
    final_qwk = frozen_qwk
    final_decoder = frozen_decoder
    print(f"Блок не обігнав заморожену голову {frozen_qwk:.4f}. У підсумок іде вона.")

print(f"Exp11 підсумок QWK {final_qwk:.4f}, декодер {final_decoder}.")
exp11_name = "Exp11. Голова на замороженому ResNet, потім останній блок"
exp11_notes = (
    f"заморожена {frozen_qwk:.4f}; блок {best_tune_qwk:.4f}; обрано "
    f"{'блок' if use_tuned else 'голова'}; декодер {final_decoder}"
)
if "log_experiment" in globals():
    log_experiment(
        exp11_name,
        final_qwk,
        final_decoder,
        tune_epochs if use_tuned else frozen_epochs,
        best_tune_epoch if use_tuned else frozen_epoch,
        exp11_notes,
    )
else:
    print(exp11_name, round(final_qwk, 4), final_decoder, exp11_notes)

if final_qwk > EXP5_VAL_QWK + SUBMISSION_MARGIN:
    print("Підсумок вищий за Exp5 більше ніж на 0.02. Навчаю на повному train і пишу submission.csv.")
    (text_full, text_test), _, _, _ = text_matrices(
        train_df["text_clean"],
        [train_df["text_clean"], test_df["text_clean"]],
    )
    final_scaler = StandardScaler()
    fused_all = final_scaler.fit_transform(np.concatenate([resnet_combined, text_full], axis=1)).astype(np.float32)
    fused_test = final_scaler.transform(np.concatenate([resnet_combined_test, text_test], axis=1)).astype(np.float32)
    torch.manual_seed(SEED)
    full_head = JointHead(fused_all.shape[1], JOINT_HIDDEN, N_CLASSES).to(device)
    full_optimizer = torch.optim.AdamW(full_head.parameters(), lr=HEAD_LR, weight_decay=WEIGHT_DECAY)
    full_criterion = nn.CrossEntropyLoss()
    full_loader = DataLoader(
        VecDataset(fused_all, y_all),
        batch_size=HEAD_BATCH,
        shuffle=True,
        generator=torch.Generator().manual_seed(SEED),
    )
    head_epochs = max(int(frozen_epoch), 1)
    full_head.train()
    for epoch in range(1, head_epochs + 1):
        total_loss, seen = 0.0, 0
        for batch_x, batch_y in full_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            full_optimizer.zero_grad(set_to_none=True)
            logits = full_head(batch_x)
            loss = full_criterion(logits, batch_y)
            loss.backward()
            nn.utils.clip_grad_norm_(full_head.parameters(), 5.0)
            full_optimizer.step()
            total_loss += loss.item() * batch_y.size(0)
            seen += batch_y.size(0)
        print(f"Повне навчання голови, епоха {epoch}/{head_epochs}, train loss {total_loss / max(seen, 1):.4f}")
    if use_tuned:
        full_backbone, full_transform = make_backbone()
        full_mean = torch.tensor(final_scaler.mean_, dtype=torch.float32, device=device)
        full_div = torch.tensor(np.clip(final_scaler.scale_, 1e-6, None), dtype=torch.float32, device=device)
        full_tune_optimizer = torch.optim.AdamW(
            [
                {"params": full_backbone.layer4[-1].parameters(), "lr": FINE_BLOCK_LR},
                {"params": full_head.parameters(), "lr": FINE_HEAD_LR},
            ],
            weight_decay=WEIGHT_DECAY,
        )
        full_tune_loader = DataLoader(
            PetTuneDataset(np.arange(len(train_df)), text_full, train_photo_rows, full_transform, y_all),
            batch_size=FINE_BATCH,
            shuffle=True,
            num_workers=NUM_WORKERS,
            generator=torch.Generator().manual_seed(SEED),
        )
        for epoch in range(1, max(int(best_tune_epoch), 1) + 1):
            full_backbone.eval()
            full_backbone.layer4[-1].train()
            full_head.train()
            total_loss, seen = 0.0, 0
            for batch in full_tune_loader:
                labels = batch["label"].to(device)
                full_tune_optimizer.zero_grad(set_to_none=True)
                logits = batch_logits(full_backbone, full_head, batch, full_mean, full_div)
                loss = full_criterion(logits, labels)
                loss.backward()
                nn.utils.clip_grad_norm_(
                    [parameter for parameter in list(full_backbone.layer4[-1].parameters()) + list(full_head.parameters()) if parameter.requires_grad],
                    5.0,
                )
                full_tune_optimizer.step()
                total_loss += loss.item() * labels.size(0)
                seen += labels.size(0)
            print(f"Повне донавчання блока, епоха {epoch}/{best_tune_epoch}, train loss {total_loss / max(seen, 1):.4f}")
        test_loader = DataLoader(
            PetTuneDataset(np.arange(len(test_ids)), text_test, test_photo_rows, full_transform, None),
            batch_size=FINE_BATCH,
            shuffle=False,
            num_workers=NUM_WORKERS,
        )
        full_backbone.eval()
        full_head.eval()
        test_chunks = []
        with torch.no_grad():
            for batch in test_loader:
                logits = batch_logits(full_backbone, full_head, batch, full_mean, full_div)
                test_chunks.append(torch.softmax(logits, dim=1).cpu().numpy())
        test_probs = np.concatenate(test_chunks)
    else:
        test_probs = predict_vectors(full_head, fused_test)
    test_idx = decode_probabilities(test_probs, N_CLASSES)[final_decoder]
    submission = pd.DataFrame({
        "PetID": [normalize_id(pet_id) for pet_id in test_ids],
        "AdoptionSpeed": [IDX_TO_LABEL[int(index)] for index in test_idx],
    })
    submission = submission.drop_duplicates("PetID", keep="first")
    submission["AdoptionSpeed"] = submission["AdoptionSpeed"].astype(int)
    submission = submission[["PetID", "AdoptionSpeed"]]
    submission.to_csv(WORKING / "submission.csv", index=False)
    print(f"Збережено submission.csv, рядків {len(submission)}.")
    print(submission["AdoptionSpeed"].value_counts().sort_index())
else:
    print(
        "Новий submission.csv не записано: підсумок не вищий за Exp5 більше ніж на 0.02. "
        "Публічний файл зі score 0.6676 лишається чинним."
    )


### Висновки експерименту 11

Заморожена голова повторила Exp5. По епохах argmax ішов 0.436, 0.464, 0.460, 0.455, очікування — 0.367, 0.394, 0.396, 0.410. Зупинка після 4-ї епохи, найкраща — 2-га, val QWK 0.4639 за argmax, accuracy 0.4949. Val loss найнижчий на 3-й епосі, 1.176, а каппа вже на 2-й, де val loss 1.182. Це той самий режим, що дав публічний score 0.6676.

Останній блок, 4 720 640 параметрів, ішов зі швидкістю 1e-5 після вже навченої голови. Дві епохи: argmax 0.4475 і 0.4629, очікування 0.400 і 0.429. Жодна не перевищила заморожену голову на 0.001, тому донавчання зупинилось. У рядок журналу для блока потрапило 0.4639: це поріг замороженої голови, від якого рахувався приріст, а не каппа окремої епохи блока. Краща епоха блока — 0.4629.

У підсумок пішла заморожена голова, декодер argmax, QWK 0.4639. Новий submission.csv не записано: підсумок збігається з Exp5 і не вищий за 0.4639 більше ніж на 0.02. Публічний файл зі score 0.6676 лишається чинним.

Теплий старт не підняв розмороження. У Exp9 блок зі швидкістю 1e-4 і ще порожньою головою опустив каппу до 0.402. Тут швидкість удесятеро менша і голова вже вміла читати ознаки, а перша епоха блока все одно дала 0.448, друга повернулась лише до 0.463. Донавчання згорткових ваг на цій вибірці далі не продовжуємо. Чинна модель — Exp5.


## 19. Експеримент 12. Цифри в описі і сирий текст

Очищення Exp5 вирізає все, крім літер. Фраза «2 months» втрачає число, а токен довжини 1 відкидається навіть якби цифра лишилась у рядку. Віку окремою колонкою в таблиці немає, тож величина віку може бути лише в описі. TF-IDF Exp1 і Exp5 цього числа не бачили.

У цьому розділі дві голови на одних і тих самих векторах ResNet18 (до 3 фото). Решта голови як у Exp5: SVD на 128, прихований шар 128, без ваг класів.

1. **Цифри.** Те саме очищення, той самий список стоп-слів і розкриття скорочень, але цифри лишаються. Одноцифровий токен теж лишається, інакше «2 months» знову зникне. У векторизаторі шаблон токена пропускає слово з однієї цифри.
2. **Сирий опис.** Нижній регістр і прибрані URL. Стоп-слова не вирізаються, пунктуація не зчищається заздалегідь, скорочення не розкриваються. Векторизатор сам ділить рядок на слова і теж бачить окремі цифри.

Дві голови розділяють причини. Якщо зростуть обидві, новий сигнал у числах. Якщо зросте лише сирий опис, заважало решта очищення. Якщо зростуть лише цифри, стоп-слова в сирому тексті розбавили вік.

Як запускати на Kaggle: **не натискайте Run All**. Не перезапускайте комірки навчання попередніх розділів. Виконайте лише цей розділ зверху вниз. Друга комірка бере кеш `resnet18_pet_features.npz` або один раз рахує заморожені ознаки. Третя комірка фото більше не читає: дві голови вчаться на вже готовій матриці. Потрібні GPU і Internet, якщо кешу ResNet немає.

Поріг сабміту: кращий із двох val QWK вищий за 0.4639 більше ніж на 0.02. Інакше чинним лишається файл зі score 0.6676.


In [ ]:
import os
import random
import re
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import cohen_kappa_score
from sklearn.model_selection import train_test_split

SEED = 42
VAL_SIZE = 0.2
MAX_IMAGES = 3
FEATURE_BATCH_SIZE = 64
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
DATA_ROOT = Path("/kaggle/input/competitions/deep-learning-for-computer-vision-and-nlp-2026-10")
IMAGE_DIR = DATA_ROOT / "images"
WORKING = Path("/kaggle/working")
WORKING.mkdir(parents=True, exist_ok=True)
NUM_WORKERS = 2 if os.name != "nt" else 0
if "device" not in globals():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

needs_tables = "train_df" not in globals() or "idx_tr" not in globals() or "y_all" not in globals()
if not needs_tables:
    print(f"Таблиці вже в пам'яті: train {len(train_df)}, val {len(idx_va)}. Повторно не читаю.")
else:
    TRAIN_CSV = DATA_ROOT / "train.csv"
    TEST_CSV = DATA_ROOT / "test.csv"
    SAMPLE_CSV = DATA_ROOT / "sample_submission.csv"

    def find_column(columns, options):
        lookup = {str(column).lower(): column for column in columns}
        for option in options:
            if option.lower() in lookup:
                return lookup[option.lower()]
        return None

    def normalize_id(value):
        if pd.isna(value):
            return ""
        text = str(value).strip()
        if text.endswith(".0"):
            text = text[:-2]
        return text

    def read_table(path):
        frame = pd.read_csv(path)
        frame.columns = [str(column).strip() for column in frame.columns]
        return frame

    def quadratic_weighted_kappa(y_true, y_pred):
        y_true = np.asarray(y_true).astype(int)
        y_pred = np.asarray(y_pred).astype(int)
        if len(y_true) == 0:
            return 0.0
        score = cohen_kappa_score(y_true, y_pred, weights="quadratic")
        if score != score:
            return 0.0
        return float(score)

    train_df = read_table(TRAIN_CSV)
    test_df = read_table(TEST_CSV)
    sample_df = read_table(SAMPLE_CSV)
    id_col = find_column(train_df.columns, ["PetID", "pet_id", "id"])
    target_col = find_column(train_df.columns, ["AdoptionSpeed", "adoption_speed", "label", "target"])
    text_col = find_column(train_df.columns, ["Description", "description", "desc", "text", "comments"])
    test_id_col = find_column(test_df.columns, ["PetID", "pet_id", "id"]) or id_col
    train_df[id_col] = train_df[id_col].map(normalize_id)
    test_df[test_id_col] = test_df[test_id_col].map(normalize_id)
    if int(train_df[id_col].duplicated().sum()):
        train_df = train_df.drop_duplicates(id_col, keep="first")
    train_df = train_df.reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)
    if target_col in test_df.columns:
        test_df = test_df.drop(columns=[target_col])
    test_text_col = find_column(test_df.columns, ["Description", "description", "desc", "text", "comments"])
    if text_col is None:
        text_col = "Description"
        train_df[text_col] = ""
        test_df[text_col] = ""
    elif test_text_col is None:
        test_df[text_col] = ""
    elif test_text_col != text_col:
        test_df[text_col] = test_df[test_text_col]
    raw_target = train_df[target_col]
    if pd.api.types.is_numeric_dtype(raw_target):
        numeric_target = raw_target.astype(float)
        if np.allclose(numeric_target.to_numpy(), np.rint(numeric_target.to_numpy())):
            train_df[target_col] = numeric_target.round().astype(int)
    classes = sorted(train_df[target_col].dropna().unique().tolist())
    LABEL_TO_IDX = {label: index for index, label in enumerate(classes)}
    IDX_TO_LABEL = {index: label for label, index in LABEL_TO_IDX.items()}
    N_CLASSES = len(classes)
    train_df = train_df.dropna(subset=[target_col]).reset_index(drop=True)
    y_all = train_df[target_col].map(LABEL_TO_IDX).to_numpy(dtype=int)
    idx_tr, idx_va = train_test_split(
        np.arange(len(train_df)), test_size=VAL_SIZE, random_state=SEED, stratify=y_all
    )
    y_tr = y_all[idx_tr]
    y_va = y_all[idx_va]
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    print(f"Таблиці прочитано заново: train {len(train_df)}, val {len(idx_va)}, test {len(test_df)}.")

if "train_ids" not in globals():
    train_ids = train_df[id_col].tolist()
if "test_ids" not in globals():
    test_ids = test_df[test_id_col].tolist()

if len(idx_va) != 1287:
    print(f"Увага: у валідації {len(idx_va)} рядків, у минулому прогоні було 1287.")
else:
    print("Розбиття збігається з минулим прогоном: 1287 валідаційних рядків.")

if "text_clean" in train_df.columns and "text_clean" in test_df.columns:
    print("Очищений опис уже є, повторно не чищу.")
else:
    NEGATION_TOKENS = {
        "no", "not", "nor", "never", "none", "nobody", "nothing",
        "neither", "nowhere", "without", "cannot",
    }
    CONTRACTIONS = {
        "ain't": "is not",
        "aren't": "are not",
        "can't": "cannot",
        "couldn't": "could not",
        "didn't": "did not",
        "doesn't": "does not",
        "don't": "do not",
        "hadn't": "had not",
        "hasn't": "has not",
        "haven't": "have not",
        "he's": "he is",
        "here's": "here is",
        "how's": "how is",
        "i'm": "i am",
        "i've": "i have",
        "isn't": "is not",
        "it's": "it is",
        "let's": "let us",
        "mustn't": "must not",
        "she's": "she is",
        "that's": "that is",
        "there's": "there is",
        "they're": "they are",
        "they've": "they have",
        "we're": "we are",
        "we've": "we have",
        "weren't": "were not",
        "what's": "what is",
        "where's": "where is",
        "who's": "who is",
        "won't": "will not",
        "wouldn't": "would not",
        "you're": "you are",
        "you've": "you have",
    }
    CONTRACTION_RE = re.compile(
        r"\b(" + "|".join(sorted(map(re.escape, CONTRACTIONS), key=len, reverse=True)) + r")\b"
    )
    FALLBACK_STOPWORDS = set(
        "a about above after again against all am an and any are as at be because been "
        "before being below between both but by can did do does doing down during each "
        "few for from further had has have having he her here hers herself him himself "
        "his how i if in into is it its itself just me more most my myself of off on "
        "once only or other our ours ourselves out over own same she should so some "
        "such than that the their theirs them themselves then there these they this "
        "those through to too under until up very was we were what when where which "
        "while who whom why will with you your yours yourself yourselves".split()
    )

    def load_stopwords():
        try:
            import nltk
            from nltk.corpus import stopwords

            nltk.download("stopwords", quiet=True)
            words = set(stopwords.words("english"))
            print("Стоп-слова: NLTK, english")
        except Exception as exc:
            print(f"Стоп-слова NLTK недоступні ({type(exc).__name__}: {exc}). Використовую вбудований список.")
            words = set(FALLBACK_STOPWORDS)
        return words - NEGATION_TOKENS

    STOPWORDS = load_stopwords()

    def clean_text(value):
        if not isinstance(value, str):
            return ""
        text = value.lower().replace("’", "'").replace("`", "'")
        text = CONTRACTION_RE.sub(lambda match: CONTRACTIONS[match.group(0)], text)
        text = re.sub(r"https?://\S+|www\.\S+", " ", text)
        text = re.sub(r"[^a-z\s]", " ", text)
        tokens = [token for token in text.split() if len(token) > 1 and token not in STOPWORDS]
        return " ".join(tokens)

    train_df["text_clean"] = train_df[text_col].fillna("").astype(str).map(clean_text)
    test_df["text_clean"] = test_df[text_col].fillna("").astype(str).map(clean_text)
    print("Порожніх описів після очищення:", float(train_df["text_clean"].eq("").mean()))

print("Пристрій:", device)


In [ ]:
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision.models import ResNet18_Weights, resnet18
from tqdm.auto import tqdm

PHOTO_CAP = 3
FEATURE_BATCH_SIZE = 64
CACHE_NAME = "resnet18_pet_features.npz"


def list_image_files(image_dir):
    files = []
    for path in Path(image_dir).rglob("*"):
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS:
            files.append(path)
    return files


def link_all_images(pet_ids, files):
    pet_set = set(pet_ids)
    grouped = {pet_id: [] for pet_id in pet_set}
    for path in files:
        stem = path.stem
        pet_id = None
        order = 0
        if stem in pet_set:
            pet_id = stem
        else:
            for separator in ("-", "_"):
                if separator not in stem:
                    continue
                base, rest = stem.rsplit(separator, 1)
                if base in pet_set:
                    pet_id = base
                    order = int(rest) if rest.isdigit() else 10**6
                    break
        if pet_id is not None:
            grouped[pet_id].append((order, path))
    linked = {}
    for pet_id, items in grouped.items():
        items.sort(key=lambda item: (item[0], item[1].name))
        linked[pet_id] = [item_path for _, item_path in items]
    return linked


image_files = list_image_files(IMAGE_DIR)
all_ids = list(dict.fromkeys(list(train_ids) + list(test_ids)))
linked_photos = link_all_images(all_ids, image_files)
train_photo_rows = [linked_photos[pet_id][:PHOTO_CAP] for pet_id in train_ids]
test_photo_rows = [linked_photos[pet_id][:PHOTO_CAP] for pet_id in test_ids]
print(f"Файлів зображень: {len(image_files)}. У середнє і в донавчання входить до {PHOTO_CAP} фото.")

reuse_resnet = (
    "resnet_combined" in globals()
    and "resnet_combined_test" in globals()
    and getattr(resnet_combined, "shape", (0, 0))[1] == 513
    and len(resnet_combined) == len(train_df)
    and len(resnet_combined_test) == len(test_ids)
)
if reuse_resnet:
    print("Вектори ResNet18 на 3 фото вже в пам'яті:", tuple(resnet_combined.shape))
else:
    class PetImageDataset(Dataset):
        def __init__(self, records, transform):
            self.records = records
            self.transform = transform

        def __len__(self):
            return len(self.records)

        def __getitem__(self, index):
            pet_id, path = self.records[index]
            try:
                with Image.open(path) as image:
                    tensor = self.transform(image.convert("RGB"))
                ok = 1
            except Exception:
                tensor = torch.zeros(3, 224, 224)
                ok = 0
            return tensor, ok, pet_id

    def extract_features(paths_by_pet, backbone, transform, dim):
        records = [(pet_id, path) for pet_id, paths in paths_by_pet.items() for path in paths]
        sums = {pet_id: np.zeros(dim, dtype=np.float32) for pet_id in paths_by_pet}
        counts = {pet_id: 0 for pet_id in paths_by_pet}
        if records:
            loader = DataLoader(
                PetImageDataset(records, transform),
                batch_size=FEATURE_BATCH_SIZE,
                shuffle=False,
                num_workers=NUM_WORKERS,
            )
            backbone.eval()
            for tensors, oks, pet_ids_batch in tqdm(loader, desc="ResNet18"):
                features = backbone(tensors.to(device)).detach().cpu().numpy()
                oks = oks.numpy()
                for index, pet_id in enumerate(pet_ids_batch):
                    if int(oks[index]) == 1:
                        sums[pet_id] += features[index]
                        counts[pet_id] += 1
        extracted = {}
        for pet_id in paths_by_pet:
            if counts[pet_id] > 0:
                extracted[pet_id] = (sums[pet_id] / counts[pet_id], counts[pet_id])
            else:
                extracted[pet_id] = (np.zeros(dim, dtype=np.float32), 0)
        return extracted

    def stack_features(pet_ids, feature_map, dim):
        vectors = np.zeros((len(pet_ids), dim), dtype=np.float32)
        missing = np.ones((len(pet_ids), 1), dtype=np.float32)
        for index, pet_id in enumerate(pet_ids):
            vector, count = feature_map[pet_id]
            if count > 0:
                vectors[index] = vector
                missing[index, 0] = 0.0
        return np.concatenate([vectors, missing], axis=1)

    def locate_cache():
        direct = WORKING / CACHE_NAME
        if direct.exists():
            return direct
        root = Path("/kaggle/input")
        if root.exists():
            found = sorted(root.rglob(CACHE_NAME))
            if found:
                print(f"Кеш знайдено серед вхідних даних: {found[0]}")
                return found[0]
        return None

    def load_feature_map(cache_path):
        cached = np.load(cache_path, allow_pickle=False)
        cached_ids = [normalize_id(pet_id) for pet_id in cached["ids"].tolist()]
        counts = np.array(cached["counts"], copy=True)
        matrix = np.array(cached["features"], dtype=np.float32, copy=True)
        cached.close()
        feature_map = {}
        for index, pet_id in enumerate(cached_ids):
            feature_map[pet_id] = (matrix[index].copy(), int(counts[index]))
        return feature_map, int(matrix.shape[1])

    needed = set(train_ids) | set(test_ids)
    cache_path = locate_cache()
    feature_map = None
    feat_dim = None
    if cache_path is not None:
        loaded, feat_dim = load_feature_map(cache_path)
        if feat_dim == 512 and set(loaded) >= needed:
            feature_map = loaded
            print(f"Ознаки ResNet18 завантажено з кешу: {cache_path}")
        else:
            print("Кеш ResNet не підходить для цього кроку, рахую заново.")
    if feature_map is None:
        print("Кешу ResNet18 немає. Рахую заморожені ознаки трьох фото.")
        try:
            weights = ResNet18_Weights.IMAGENET1K_V1
            backbone = resnet18(weights=weights)
            transform = weights.transforms()
        except Exception as exc:
            raise RuntimeError(
                "Не вдалося завантажити ваги ResNet18. Увімкніть Internet у налаштуваннях ноутбука."
            ) from exc
        backbone.fc = nn.Identity()
        backbone.to(device)
        backbone.eval()
        for parameter in backbone.parameters():
            parameter.requires_grad = False
        with torch.no_grad():
            feat_dim = int(backbone(torch.zeros(1, 3, 224, 224, device=device)).shape[1])
        paths_by_pet = {pet_id: linked_photos[pet_id][:PHOTO_CAP] for pet_id in all_ids}
        feature_map = extract_features(paths_by_pet, backbone, transform, feat_dim)
        ids_array = np.array(list(feature_map.keys()))
        cache_out = WORKING / CACHE_NAME
        np.savez_compressed(
            cache_out,
            ids=ids_array,
            features=np.stack([feature_map[pet_id][0] for pet_id in ids_array]),
            counts=np.array([feature_map[pet_id][1] for pet_id in ids_array], dtype=np.int32),
        )
        print(f"Ознаки збережено: {cache_out}")
        del backbone
        if device.type == "cuda":
            torch.cuda.empty_cache()
    resnet_combined = stack_features(train_ids, feature_map, feat_dim)
    resnet_combined_test = stack_features(test_ids, feature_map, feat_dim)

print("Матриця ResNet train/test:", tuple(resnet_combined.shape), tuple(resnet_combined_test.shape))


In [ ]:
import copy
import re

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, Dataset

SVD_DIM = 128
JOINT_HIDDEN = 128
JOINT_DROPOUT = 0.3
JOINT_LR = 1e-3
JOINT_WEIGHT_DECAY = 1e-4
JOINT_MAX_EPOCHS = 8
JOINT_PATIENCE = 2
JOINT_MIN_DELTA = 0.001
JOINT_BATCH = 64
EXP5_VAL_QWK = 0.4639
SUBMISSION_MARGIN = 0.02
TFIDF_MAX_FEATURES = 20000
TFIDF_MIN_DF = 2
TOKEN_PATTERN = r"(?u)\b\w+\b"
AGE_RE = re.compile(
    r"\b\d+\s*(?:month|months|week|weeks|year|years|day|days|yr|yrs|mth|mths)\b",
    re.IGNORECASE,
)

assert "resnet_combined" in globals() and resnet_combined.shape[1] == 513, (
    "Потрібна матриця ResNet18 на 3 фото. Спочатку виконайте попередню комірку цього розділу."
)
assert len(resnet_combined) == len(train_df)
assert len(resnet_combined_test) == len(test_ids)

if "STOPWORDS" not in globals():
    NEGATION_TOKENS = {"no", "not", "nor", "never", "none", "nobody", "nothing", "neither"}
    FALLBACK_STOPWORDS = set(
        "a about above after again against all am an and any are as at be because been "
        "before being below between both but by can did do does doing down during each "
        "few for from further had has have having he her here hers herself him himself "
        "his how i if in into is it its itself just me more most my myself of off on "
        "once only or other our ours ourselves out over own same she should so some "
        "such than that the their theirs them themselves then there these they this "
        "those through to too under until up very was we were what when where which "
        "while who whom why will with you your yours yourself yourselves".split()
    )

    def load_stopwords():
        try:
            import nltk
            from nltk.corpus import stopwords

            nltk.download("stopwords", quiet=True)
            words = set(stopwords.words("english"))
            print("Стоп-слова: NLTK, english")
        except Exception as exc:
            print(f"Стоп-слова NLTK недоступні ({type(exc).__name__}: {exc}). Використовую вбудований список.")
            words = set(FALLBACK_STOPWORDS)
        return words - NEGATION_TOKENS

    STOPWORDS = load_stopwords()


def clean_text_keep_digits(value):
    if not isinstance(value, str):
        return ""
    text = value.lower().replace("\u2019", "'").replace("`", "'")
    if "CONTRACTION_RE" in globals() and "CONTRACTIONS" in globals():
        text = CONTRACTION_RE.sub(lambda match: CONTRACTIONS[match.group(0)], text)
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    tokens = [
        token
        for token in text.split()
        if (len(token) > 1 or token.isdigit()) and token not in STOPWORDS
    ]
    return " ".join(tokens)


def normalize_raw_text(value):
    if not isinstance(value, str):
        return ""
    text = value.lower().replace("\u2019", "'").replace("`", "'")
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    return re.sub(r"\s+", " ", text).strip()


class JointDataset(Dataset):
    def __init__(self, features, labels=None):
        self.features = torch.tensor(np.asarray(features), dtype=torch.float32)
        self.labels = None if labels is None else torch.tensor(np.asarray(labels), dtype=torch.long)

    def __len__(self):
        return self.features.size(0)

    def __getitem__(self, index):
        if self.labels is None:
            return self.features[index]
        return self.features[index], self.labels[index]


class JointHead(nn.Module):
    def __init__(self, in_dim, hidden, n_classes, dropout=JOINT_DROPOUT):
        super().__init__()
        self.net = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, n_classes),
        )

    def forward(self, features):
        return self.net(features)


def decode_probabilities(probs, n_classes):
    class_index = np.arange(n_classes, dtype=np.float64)
    expectation = np.clip(np.rint(probs @ class_index), 0, n_classes - 1).astype(int)
    return {"argmax": probs.argmax(axis=1).astype(int), "expectation": expectation}


def fit_joint_head(features_train, labels_train, features_val, labels_val, prefix):
    torch.manual_seed(SEED)
    model = JointHead(features_train.shape[1], JOINT_HIDDEN, N_CLASSES).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=JOINT_LR, weight_decay=JOINT_WEIGHT_DECAY)
    criterion = nn.CrossEntropyLoss()
    train_loader = DataLoader(
        JointDataset(features_train, labels_train),
        batch_size=JOINT_BATCH,
        shuffle=True,
        generator=torch.Generator().manual_seed(SEED),
    )
    val_loader = DataLoader(JointDataset(features_val, labels_val), batch_size=256, shuffle=False)
    best_qwk = -1.0
    best_epoch = 1
    best_decoder = "argmax"
    best_state = copy.deepcopy(model.state_dict())
    wait = 0
    epochs_ran = 0
    for epoch in range(1, JOINT_MAX_EPOCHS + 1):
        epochs_ran = epoch
        model.train()
        total_loss, seen = 0.0, 0
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            optimizer.zero_grad(set_to_none=True)
            logits = model(batch_x)
            loss = criterion(logits, batch_y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            total_loss += loss.item() * batch_y.size(0)
            seen += batch_y.size(0)
        model.eval()
        prob_chunks, true_chunks = [], []
        val_loss, val_seen = 0.0, 0
        with torch.no_grad():
            for batch_x, batch_y in val_loader:
                batch_x, batch_y = batch_x.to(device), batch_y.to(device)
                logits = model(batch_x)
                val_loss += criterion(logits, batch_y).item() * batch_y.size(0)
                val_seen += batch_y.size(0)
                prob_chunks.append(torch.softmax(logits, dim=1).cpu().numpy())
                true_chunks.append(batch_y.cpu().numpy())
        probs = np.concatenate(prob_chunks)
        y_true = np.concatenate(true_chunks)
        decoded = decode_probabilities(probs, N_CLASSES)
        qwk_argmax = quadratic_weighted_kappa(y_true, decoded["argmax"])
        qwk_expectation = quadratic_weighted_kappa(y_true, decoded["expectation"])
        epoch_decoder = "expectation" if qwk_expectation > qwk_argmax else "argmax"
        epoch_qwk = qwk_expectation if epoch_decoder == "expectation" else qwk_argmax
        print(
            f"{prefix} {epoch:02d}/{JOINT_MAX_EPOCHS} | train loss {total_loss / max(seen, 1):.4f} | "
            f"val loss {val_loss / max(val_seen, 1):.4f} | "
            f"QWK argmax {qwk_argmax:.4f} | QWK expectation {qwk_expectation:.4f}"
        )
        if epoch_qwk > best_qwk + JOINT_MIN_DELTA:
            best_qwk = epoch_qwk
            best_epoch = epoch
            best_decoder = epoch_decoder
            best_state = copy.deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1
            if wait >= JOINT_PATIENCE:
                break
    model.load_state_dict(best_state)
    n_params = sum(parameter.numel() for parameter in model.parameters())
    print(
        f"{prefix}: зупинка після {epochs_ran}, найкраща {best_epoch}, "
        f"QWK {best_qwk:.4f} ({best_decoder}), параметрів {n_params}."
    )
    return model, best_qwk, best_epoch, best_decoder, epochs_ran


def predict_joint(model, features):
    model.eval()
    loader = DataLoader(JointDataset(features), batch_size=256, shuffle=False)
    chunks = []
    with torch.no_grad():
        for batch_x in loader:
            logits = model(batch_x.to(device))
            chunks.append(torch.softmax(logits, dim=1).cpu().numpy())
    return np.concatenate(chunks)


def text_svd(frame_fit, frames_transform):
    vectorizer = TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=TFIDF_MIN_DF,
        max_features=TFIDF_MAX_FEATURES,
        token_pattern=TOKEN_PATTERN,
    )
    sparse_fit = vectorizer.fit_transform(frame_fit)
    svd_dim = min(SVD_DIM, sparse_fit.shape[1] - 1, sparse_fit.shape[0] - 1)
    svd = TruncatedSVD(n_components=svd_dim, random_state=SEED)
    dense_fit = svd.fit_transform(sparse_fit)
    transformed = [svd.transform(vectorizer.transform(frame)) for frame in frames_transform]
    explained = float(svd.explained_variance_ratio_.sum())
    names = vectorizer.get_feature_names_out()
    digit_names = [name for name in names if any(character.isdigit() for character in name)]
    return dense_fit, transformed, explained, sparse_fit.shape[1], svd_dim, digit_names


def run_variant(prefix, train_text, test_text):
    text_tr, (text_va,), explained, n_tfidf, svd_dim, digit_names = text_svd(
        train_text.iloc[idx_tr],
        [train_text.iloc[idx_va]],
    )
    print(
        f"{prefix} | TF-IDF ознак {n_tfidf}, компонент SVD {svd_dim}, "
        f"пояснена дисперсія {explained:.3f}. Ознак із цифрою {len(digit_names)}. Ваг класів немає."
    )
    print(f"{prefix} | приклади ознак із цифрою: {digit_names[:25]}")
    scaler = StandardScaler()
    fused_tr = scaler.fit_transform(np.concatenate([resnet_combined[idx_tr], text_tr], axis=1)).astype(np.float32)
    fused_va = scaler.transform(np.concatenate([resnet_combined[idx_va], text_va], axis=1)).astype(np.float32)
    print(f"{prefix} | спільна матриця train/val:", fused_tr.shape, fused_va.shape)
    model, best_qwk, best_epoch, decoder, epochs_ran = fit_joint_head(fused_tr, y_tr, fused_va, y_va, prefix)
    val_probs = predict_joint(model, fused_va)
    val_decoded = decode_probabilities(val_probs, N_CLASSES)
    val_scores = {name: quadratic_weighted_kappa(y_va, pred) for name, pred in val_decoded.items()}
    chosen_pred = val_decoded[decoder]
    print(
        f"{prefix} QWK argmax {val_scores['argmax']:.4f}, expectation {val_scores['expectation']:.4f}. "
        f"Обрано {decoder}, accuracy {accuracy_score(y_va, chosen_pred):.4f}."
    )
    print(f"{prefix} | прогнози на валідації:")
    print(pd.Series(chosen_pred).map(IDX_TO_LABEL).value_counts().sort_index())
    return {
        "prefix": prefix,
        "train_text": train_text,
        "test_text": test_text,
        "explained": explained,
        "n_tfidf": n_tfidf,
        "svd_dim": svd_dim,
        "n_digit_features": len(digit_names),
        "qwk": val_scores[decoder],
        "scores": val_scores,
        "decoder": decoder,
        "accuracy": float(accuracy_score(y_va, chosen_pred)),
        "best_epoch": best_epoch,
        "epochs_ran": epochs_ran,
        "best_qwk_seen": best_qwk,
    }


raw_train = train_df[text_col].fillna("").astype(str)
raw_test = test_df[text_col].fillna("").astype(str)
digits_train = raw_train.map(clean_text_keep_digits)
digits_test = raw_test.map(clean_text_keep_digits)
plain_train = raw_train.map(normalize_raw_text)
plain_test = raw_test.map(normalize_raw_text)

age_hits = int(raw_train.str.contains(AGE_RE).sum())
print(f"Описів train із фразою віку на кшталт «2 months»: {age_hits} з {len(raw_train)}.")
if "text_clean" in train_df.columns:
    changed = int((digits_train.to_numpy() != train_df["text_clean"].fillna("").astype(str).to_numpy()).sum())
    print(f"Рядків, де текст із цифрами відрізняється від очищеного Exp5: {changed} з {len(train_df)}.")
shown = 0
for raw_value, digit_value in zip(raw_train.tolist(), digits_train.tolist()):
    if AGE_RE.search(raw_value) is None:
        continue
    print("Сирий фрагмент:", " ".join(raw_value.split())[:180])
    print("Текст із цифрами:", digit_value[:180])
    shown += 1
    if shown >= 5:
        break

results = [
    run_variant("Цифри", digits_train, digits_test),
    run_variant("Сирий", plain_train, plain_test),
]
for row in results:
    notes = (
        f"svd {row['svd_dim']}; пояснена дисперсія {row['explained']:.3f}; "
        f"ознак із цифрою {row['n_digit_features']}; "
        f"argmax {row['scores']['argmax']:.4f}; expectation {row['scores']['expectation']:.4f}"
    )
    exp_name = f"Exp12. {row['prefix']} + ResNet, SVD 128, без ваг класів"
    if "log_experiment" in globals():
        log_experiment(exp_name, row["qwk"], row["decoder"], row["epochs_ran"], row["best_epoch"], notes)
    else:
        print(exp_name, round(row["qwk"], 4), row["decoder"], "епох", row["epochs_ran"], "найкраща", row["best_epoch"])

winner = max(results, key=lambda row: (row["qwk"], row["scores"]["argmax"]))
print(
    "Exp12 підсумок: "
    + "; ".join(
        f"{row['prefix']} {row['qwk']:.4f} {row['decoder']}"
        for row in results
    )
    + f". Обрано {winner['prefix']}."
)
print(
    f"Порівняння з Exp5 {EXP5_VAL_QWK:.4f}: кращий варіант {winner['prefix']} "
    f"{winner['qwk']:.4f} ({winner['decoder']})."
)

if winner["qwk"] > EXP5_VAL_QWK + SUBMISSION_MARGIN:
    print(f"Варіант «{winner['prefix']}» вищий за Exp5 більше ніж на 0.02. Навчаю на повному train.")
    text_all, (text_test,), _, _, _, _ = text_svd(
        winner["train_text"],
        [winner["test_text"]],
    )
    final_scaler = StandardScaler()
    fused_all = final_scaler.fit_transform(
        np.concatenate([resnet_combined, text_all], axis=1)
    ).astype(np.float32)
    fused_test = final_scaler.transform(
        np.concatenate([resnet_combined_test, text_test], axis=1)
    ).astype(np.float32)
    torch.manual_seed(SEED)
    final_joint = JointHead(fused_all.shape[1], JOINT_HIDDEN, N_CLASSES).to(device)
    final_optimizer = torch.optim.AdamW(final_joint.parameters(), lr=JOINT_LR, weight_decay=JOINT_WEIGHT_DECAY)
    final_criterion = nn.CrossEntropyLoss()
    final_loader = DataLoader(
        JointDataset(fused_all, y_all),
        batch_size=JOINT_BATCH,
        shuffle=True,
        generator=torch.Generator().manual_seed(SEED),
    )
    final_epochs = max(int(winner["best_epoch"]), 1)
    final_joint.train()
    for epoch in range(1, final_epochs + 1):
        total_loss, seen = 0.0, 0
        for batch_x, batch_y in final_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            final_optimizer.zero_grad(set_to_none=True)
            logits = final_joint(batch_x)
            loss = final_criterion(logits, batch_y)
            loss.backward()
            nn.utils.clip_grad_norm_(final_joint.parameters(), 5.0)
            final_optimizer.step()
            total_loss += loss.item() * batch_y.size(0)
            seen += batch_y.size(0)
        print(
            f"Повне навчання, {winner['prefix']}, епоха {epoch}/{final_epochs}, "
            f"train loss {total_loss / max(seen, 1):.4f}"
        )
    test_probs = predict_joint(final_joint, fused_test)
    test_idx = decode_probabilities(test_probs, N_CLASSES)[winner["decoder"]]
    submission = pd.DataFrame({
        "PetID": [normalize_id(pet_id) for pet_id in test_ids],
        "AdoptionSpeed": [IDX_TO_LABEL[int(index)] for index in test_idx],
    })
    submission = submission.drop_duplicates("PetID", keep="first")
    submission["AdoptionSpeed"] = submission["AdoptionSpeed"].astype(int)
    submission = submission[["PetID", "AdoptionSpeed"]]
    submission.to_csv(WORKING / "submission.csv", index=False)
    print(f"Збережено submission.csv, рядків {len(submission)}.")
    print(submission["AdoptionSpeed"].value_counts().sort_index())
else:
    print(
        "Новий submission.csv не записано: кращий варіант не вищий за Exp5 більше ніж на 0.02. "
        "Публічний файл зі score 0.6676 лишається чинним."
    )


### Висновки експерименту 12

Фраза віку на кшталт «2 months» є в 1508 описах із 6431. Текст зі збереженими цифрами відрізняється від очищення Exp5 у 2958 рядках. У цих рядках вік доходить до моделі: у надрукованих прикладах лишились «6 weeks», «3 months», «1 year», «6 months». Поруч у словник потрапляють дози, години і розбиті дроби. Перші 25 ознак із цифрою за алфавітом — «0 5ml», «1 30pm», «1 5kg», «1 1», «1 able», а не вікові біграми: зріз іде з початку словника. Усього таких ознак 1041 із 20 000. Пояснена дисперсія SVD 0.226, матриця 641, голова 82 692 параметри.

Варіант «цифри». Argmax по епохах: 0.433, 0.467, 0.435, 0.454. Очікування: 0.355, 0.399, 0.393, 0.412. Зупинка після 4-ї епохи, найкраща — 2-га, val QWK 0.467 за argmax, accuracy 0.483. Val loss на 2-й і 3-й майже однаковий: 1.183 і 1.182. Прогнози на 1287 рядках: клас 1 — 246, 2 — 351, 3 — 155, 4 — 535. Клас 3 лишається близько 12%, як в Exp5. Приріст над 0.464 становить 0.003, accuracy нижча за 0.495 в Exp5. Поріг 0.02 не взято.

Варіант «сирий». Пояснена дисперсія 0.252, ознак із цифрою 887, серед прикладів є біграми зі службовими словами: «1 and», «1 for», «1 has». Argmax: 0.415, 0.449, 0.447, 0.454, 0.447, 0.450. Очікування: 0.359, 0.385, 0.399, 0.404, 0.407, 0.420. Зупинка після 6-ї, найкраща 4-та, val QWK 0.454 за argmax, accuracy 0.481. Val loss найнижчий на 3-й епосі, 1.176, на 4-й уже 1.188, на 6-й — 1.203. Прогнози: 1 — 266, 2 — 324, 3 — 194, 4 — 503. Клас 3 тут 15.1% за частки 20.6% у train, каппа на 0.010 нижча за Exp5.

Обрано «цифри», декодер argmax, QWK 0.467. Новий submission.csv не записано: приріст 0.003 менший за 0.02. Публічний файл зі score 0.6676 лишається чинним.

Канал усіх цифр майже не зрушив каппу, бо вік у ньому змішаний із дозами, годинами і датами. Сирий опис підняв пояснену дисперсію і знизив каппу, тож для цієї голови список стоп-слів і вирізання пунктуації лишаються доречними. Текстова частина чинної моделі — очищення Exp5.


## 20. Експеримент 13. Кумулятивна втрата

У Exp5 cross-entropy карає сусідню помилку так само, як стрибок через усю шкалу. Quadratic weighted kappa влаштована інакше: чим далі класи, тим дорожча помилка. На валідації Exp5 клас 3 лишається близько 12% за частки 20.6% у train.

Матриця та сама, що дала публічний score 0.6676: заморожений ResNet18, до 3 фото, SVD на 128 від очищеного тексту, прихований шар 128, без ваг класів. Останній шар видає три логіти. Вони відповідають на питання «клас більший за 1?», «більший за 2?», «більший за 3?». Втрата — бінарна крос-ентропія цих трьох відповідей. Помилка на сусідній клас зриває одну відповідь, помилка з 1 на 4 — усі три.

Прогноз за порогом: скільки з трьох імовірностей перевищує 0.5. Поруч рахується округлене очікування класу з того самого розподілу. Число втрати з cross-entropy Exp5 не порівнюється: тут інша шкала.

Як запускати на Kaggle: **не натискайте Run All**. Не перезапускайте комірки навчання попередніх розділів. Виконайте лише цей розділ зверху вниз. Друга комірка бере кеш `resnet18_pet_features.npz` або один раз рахує заморожені ознаки. Третя комірка фото більше не читає. Потрібні GPU і Internet, якщо кешу ResNet немає.

Поріг сабміту: val QWK вищий за 0.4639 більше ніж на 0.02. Інакше чинним лишається файл зі score 0.6676.


In [ ]:
import os
import random
import re
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import cohen_kappa_score
from sklearn.model_selection import train_test_split

SEED = 42
VAL_SIZE = 0.2
MAX_IMAGES = 3
FEATURE_BATCH_SIZE = 64
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
DATA_ROOT = Path("/kaggle/input/competitions/deep-learning-for-computer-vision-and-nlp-2026-10")
IMAGE_DIR = DATA_ROOT / "images"
WORKING = Path("/kaggle/working")
WORKING.mkdir(parents=True, exist_ok=True)
NUM_WORKERS = 2 if os.name != "nt" else 0
if "device" not in globals():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

needs_tables = "train_df" not in globals() or "idx_tr" not in globals() or "y_all" not in globals()
if not needs_tables:
    print(f"Таблиці вже в пам'яті: train {len(train_df)}, val {len(idx_va)}. Повторно не читаю.")
else:
    TRAIN_CSV = DATA_ROOT / "train.csv"
    TEST_CSV = DATA_ROOT / "test.csv"
    SAMPLE_CSV = DATA_ROOT / "sample_submission.csv"

    def find_column(columns, options):
        lookup = {str(column).lower(): column for column in columns}
        for option in options:
            if option.lower() in lookup:
                return lookup[option.lower()]
        return None

    def normalize_id(value):
        if pd.isna(value):
            return ""
        text = str(value).strip()
        if text.endswith(".0"):
            text = text[:-2]
        return text

    def read_table(path):
        frame = pd.read_csv(path)
        frame.columns = [str(column).strip() for column in frame.columns]
        return frame

    def quadratic_weighted_kappa(y_true, y_pred):
        y_true = np.asarray(y_true).astype(int)
        y_pred = np.asarray(y_pred).astype(int)
        if len(y_true) == 0:
            return 0.0
        score = cohen_kappa_score(y_true, y_pred, weights="quadratic")
        if score != score:
            return 0.0
        return float(score)

    train_df = read_table(TRAIN_CSV)
    test_df = read_table(TEST_CSV)
    sample_df = read_table(SAMPLE_CSV)
    id_col = find_column(train_df.columns, ["PetID", "pet_id", "id"])
    target_col = find_column(train_df.columns, ["AdoptionSpeed", "adoption_speed", "label", "target"])
    text_col = find_column(train_df.columns, ["Description", "description", "desc", "text", "comments"])
    test_id_col = find_column(test_df.columns, ["PetID", "pet_id", "id"]) or id_col
    train_df[id_col] = train_df[id_col].map(normalize_id)
    test_df[test_id_col] = test_df[test_id_col].map(normalize_id)
    if int(train_df[id_col].duplicated().sum()):
        train_df = train_df.drop_duplicates(id_col, keep="first")
    train_df = train_df.reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)
    if target_col in test_df.columns:
        test_df = test_df.drop(columns=[target_col])
    test_text_col = find_column(test_df.columns, ["Description", "description", "desc", "text", "comments"])
    if text_col is None:
        text_col = "Description"
        train_df[text_col] = ""
        test_df[text_col] = ""
    elif test_text_col is None:
        test_df[text_col] = ""
    elif test_text_col != text_col:
        test_df[text_col] = test_df[test_text_col]
    raw_target = train_df[target_col]
    if pd.api.types.is_numeric_dtype(raw_target):
        numeric_target = raw_target.astype(float)
        if np.allclose(numeric_target.to_numpy(), np.rint(numeric_target.to_numpy())):
            train_df[target_col] = numeric_target.round().astype(int)
    classes = sorted(train_df[target_col].dropna().unique().tolist())
    LABEL_TO_IDX = {label: index for index, label in enumerate(classes)}
    IDX_TO_LABEL = {index: label for label, index in LABEL_TO_IDX.items()}
    N_CLASSES = len(classes)
    train_df = train_df.dropna(subset=[target_col]).reset_index(drop=True)
    y_all = train_df[target_col].map(LABEL_TO_IDX).to_numpy(dtype=int)
    idx_tr, idx_va = train_test_split(
        np.arange(len(train_df)), test_size=VAL_SIZE, random_state=SEED, stratify=y_all
    )
    y_tr = y_all[idx_tr]
    y_va = y_all[idx_va]
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    print(f"Таблиці прочитано заново: train {len(train_df)}, val {len(idx_va)}, test {len(test_df)}.")

if "train_ids" not in globals():
    train_ids = train_df[id_col].tolist()
if "test_ids" not in globals():
    test_ids = test_df[test_id_col].tolist()

if len(idx_va) != 1287:
    print(f"Увага: у валідації {len(idx_va)} рядків, у минулому прогоні було 1287.")
else:
    print("Розбиття збігається з минулим прогоном: 1287 валідаційних рядків.")

if "text_clean" in train_df.columns and "text_clean" in test_df.columns:
    print("Очищений опис уже є, повторно не чищу.")
else:
    NEGATION_TOKENS = {
        "no", "not", "nor", "never", "none", "nobody", "nothing",
        "neither", "nowhere", "without", "cannot",
    }
    CONTRACTIONS = {
        "ain't": "is not",
        "aren't": "are not",
        "can't": "cannot",
        "couldn't": "could not",
        "didn't": "did not",
        "doesn't": "does not",
        "don't": "do not",
        "hadn't": "had not",
        "hasn't": "has not",
        "haven't": "have not",
        "he's": "he is",
        "here's": "here is",
        "how's": "how is",
        "i'm": "i am",
        "i've": "i have",
        "isn't": "is not",
        "it's": "it is",
        "let's": "let us",
        "mustn't": "must not",
        "she's": "she is",
        "that's": "that is",
        "there's": "there is",
        "they're": "they are",
        "they've": "they have",
        "we're": "we are",
        "we've": "we have",
        "weren't": "were not",
        "what's": "what is",
        "where's": "where is",
        "who's": "who is",
        "won't": "will not",
        "wouldn't": "would not",
        "you're": "you are",
        "you've": "you have",
    }
    CONTRACTION_RE = re.compile(
        r"\b(" + "|".join(sorted(map(re.escape, CONTRACTIONS), key=len, reverse=True)) + r")\b"
    )
    FALLBACK_STOPWORDS = set(
        "a about above after again against all am an and any are as at be because been "
        "before being below between both but by can did do does doing down during each "
        "few for from further had has have having he her here hers herself him himself "
        "his how i if in into is it its itself just me more most my myself of off on "
        "once only or other our ours ourselves out over own same she should so some "
        "such than that the their theirs them themselves then there these they this "
        "those through to too under until up very was we were what when where which "
        "while who whom why will with you your yours yourself yourselves".split()
    )

    def load_stopwords():
        try:
            import nltk
            from nltk.corpus import stopwords

            nltk.download("stopwords", quiet=True)
            words = set(stopwords.words("english"))
            print("Стоп-слова: NLTK, english")
        except Exception as exc:
            print(f"Стоп-слова NLTK недоступні ({type(exc).__name__}: {exc}). Використовую вбудований список.")
            words = set(FALLBACK_STOPWORDS)
        return words - NEGATION_TOKENS

    STOPWORDS = load_stopwords()

    def clean_text(value):
        if not isinstance(value, str):
            return ""
        text = value.lower().replace("’", "'").replace("`", "'")
        text = CONTRACTION_RE.sub(lambda match: CONTRACTIONS[match.group(0)], text)
        text = re.sub(r"https?://\S+|www\.\S+", " ", text)
        text = re.sub(r"[^a-z\s]", " ", text)
        tokens = [token for token in text.split() if len(token) > 1 and token not in STOPWORDS]
        return " ".join(tokens)

    train_df["text_clean"] = train_df[text_col].fillna("").astype(str).map(clean_text)
    test_df["text_clean"] = test_df[text_col].fillna("").astype(str).map(clean_text)
    print("Порожніх описів після очищення:", float(train_df["text_clean"].eq("").mean()))

print("Пристрій:", device)


In [ ]:
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision.models import ResNet18_Weights, resnet18
from tqdm.auto import tqdm

PHOTO_CAP = 3
FEATURE_BATCH_SIZE = 64
CACHE_NAME = "resnet18_pet_features.npz"


def list_image_files(image_dir):
    files = []
    for path in Path(image_dir).rglob("*"):
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS:
            files.append(path)
    return files


def link_all_images(pet_ids, files):
    pet_set = set(pet_ids)
    grouped = {pet_id: [] for pet_id in pet_set}
    for path in files:
        stem = path.stem
        pet_id = None
        order = 0
        if stem in pet_set:
            pet_id = stem
        else:
            for separator in ("-", "_"):
                if separator not in stem:
                    continue
                base, rest = stem.rsplit(separator, 1)
                if base in pet_set:
                    pet_id = base
                    order = int(rest) if rest.isdigit() else 10**6
                    break
        if pet_id is not None:
            grouped[pet_id].append((order, path))
    linked = {}
    for pet_id, items in grouped.items():
        items.sort(key=lambda item: (item[0], item[1].name))
        linked[pet_id] = [item_path for _, item_path in items]
    return linked


image_files = list_image_files(IMAGE_DIR)
all_ids = list(dict.fromkeys(list(train_ids) + list(test_ids)))
linked_photos = link_all_images(all_ids, image_files)
train_photo_rows = [linked_photos[pet_id][:PHOTO_CAP] for pet_id in train_ids]
test_photo_rows = [linked_photos[pet_id][:PHOTO_CAP] for pet_id in test_ids]
print(f"Файлів зображень: {len(image_files)}. У середнє і в донавчання входить до {PHOTO_CAP} фото.")

reuse_resnet = (
    "resnet_combined" in globals()
    and "resnet_combined_test" in globals()
    and getattr(resnet_combined, "shape", (0, 0))[1] == 513
    and len(resnet_combined) == len(train_df)
    and len(resnet_combined_test) == len(test_ids)
)
if reuse_resnet:
    print("Вектори ResNet18 на 3 фото вже в пам'яті:", tuple(resnet_combined.shape))
else:
    class PetImageDataset(Dataset):
        def __init__(self, records, transform):
            self.records = records
            self.transform = transform

        def __len__(self):
            return len(self.records)

        def __getitem__(self, index):
            pet_id, path = self.records[index]
            try:
                with Image.open(path) as image:
                    tensor = self.transform(image.convert("RGB"))
                ok = 1
            except Exception:
                tensor = torch.zeros(3, 224, 224)
                ok = 0
            return tensor, ok, pet_id

    def extract_features(paths_by_pet, backbone, transform, dim):
        records = [(pet_id, path) for pet_id, paths in paths_by_pet.items() for path in paths]
        sums = {pet_id: np.zeros(dim, dtype=np.float32) for pet_id in paths_by_pet}
        counts = {pet_id: 0 for pet_id in paths_by_pet}
        if records:
            loader = DataLoader(
                PetImageDataset(records, transform),
                batch_size=FEATURE_BATCH_SIZE,
                shuffle=False,
                num_workers=NUM_WORKERS,
            )
            backbone.eval()
            for tensors, oks, pet_ids_batch in tqdm(loader, desc="ResNet18"):
                features = backbone(tensors.to(device)).detach().cpu().numpy()
                oks = oks.numpy()
                for index, pet_id in enumerate(pet_ids_batch):
                    if int(oks[index]) == 1:
                        sums[pet_id] += features[index]
                        counts[pet_id] += 1
        extracted = {}
        for pet_id in paths_by_pet:
            if counts[pet_id] > 0:
                extracted[pet_id] = (sums[pet_id] / counts[pet_id], counts[pet_id])
            else:
                extracted[pet_id] = (np.zeros(dim, dtype=np.float32), 0)
        return extracted

    def stack_features(pet_ids, feature_map, dim):
        vectors = np.zeros((len(pet_ids), dim), dtype=np.float32)
        missing = np.ones((len(pet_ids), 1), dtype=np.float32)
        for index, pet_id in enumerate(pet_ids):
            vector, count = feature_map[pet_id]
            if count > 0:
                vectors[index] = vector
                missing[index, 0] = 0.0
        return np.concatenate([vectors, missing], axis=1)

    def locate_cache():
        direct = WORKING / CACHE_NAME
        if direct.exists():
            return direct
        root = Path("/kaggle/input")
        if root.exists():
            found = sorted(root.rglob(CACHE_NAME))
            if found:
                print(f"Кеш знайдено серед вхідних даних: {found[0]}")
                return found[0]
        return None

    def load_feature_map(cache_path):
        cached = np.load(cache_path, allow_pickle=False)
        cached_ids = [normalize_id(pet_id) for pet_id in cached["ids"].tolist()]
        counts = np.array(cached["counts"], copy=True)
        matrix = np.array(cached["features"], dtype=np.float32, copy=True)
        cached.close()
        feature_map = {}
        for index, pet_id in enumerate(cached_ids):
            feature_map[pet_id] = (matrix[index].copy(), int(counts[index]))
        return feature_map, int(matrix.shape[1])

    needed = set(train_ids) | set(test_ids)
    cache_path = locate_cache()
    feature_map = None
    feat_dim = None
    if cache_path is not None:
        loaded, feat_dim = load_feature_map(cache_path)
        if feat_dim == 512 and set(loaded) >= needed:
            feature_map = loaded
            print(f"Ознаки ResNet18 завантажено з кешу: {cache_path}")
        else:
            print("Кеш ResNet не підходить для цього кроку, рахую заново.")
    if feature_map is None:
        print("Кешу ResNet18 немає. Рахую заморожені ознаки трьох фото.")
        try:
            weights = ResNet18_Weights.IMAGENET1K_V1
            backbone = resnet18(weights=weights)
            transform = weights.transforms()
        except Exception as exc:
            raise RuntimeError(
                "Не вдалося завантажити ваги ResNet18. Увімкніть Internet у налаштуваннях ноутбука."
            ) from exc
        backbone.fc = nn.Identity()
        backbone.to(device)
        backbone.eval()
        for parameter in backbone.parameters():
            parameter.requires_grad = False
        with torch.no_grad():
            feat_dim = int(backbone(torch.zeros(1, 3, 224, 224, device=device)).shape[1])
        paths_by_pet = {pet_id: linked_photos[pet_id][:PHOTO_CAP] for pet_id in all_ids}
        feature_map = extract_features(paths_by_pet, backbone, transform, feat_dim)
        ids_array = np.array(list(feature_map.keys()))
        cache_out = WORKING / CACHE_NAME
        np.savez_compressed(
            cache_out,
            ids=ids_array,
            features=np.stack([feature_map[pet_id][0] for pet_id in ids_array]),
            counts=np.array([feature_map[pet_id][1] for pet_id in ids_array], dtype=np.int32),
        )
        print(f"Ознаки збережено: {cache_out}")
        del backbone
        if device.type == "cuda":
            torch.cuda.empty_cache()
    resnet_combined = stack_features(train_ids, feature_map, feat_dim)
    resnet_combined_test = stack_features(test_ids, feature_map, feat_dim)

print("Матриця ResNet train/test:", tuple(resnet_combined.shape), tuple(resnet_combined_test.shape))


In [ ]:
import copy

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, Dataset

SVD_DIM = 128
JOINT_HIDDEN = 128
JOINT_DROPOUT = 0.3
JOINT_LR = 1e-3
JOINT_WEIGHT_DECAY = 1e-4
JOINT_MAX_EPOCHS = 8
JOINT_PATIENCE = 2
JOINT_MIN_DELTA = 0.001
JOINT_BATCH = 64
EXP5_VAL_QWK = 0.4639
SUBMISSION_MARGIN = 0.02
TFIDF_MAX_FEATURES = 20000
TFIDF_MIN_DF = 2

assert "resnet_combined" in globals() and resnet_combined.shape[1] == 513, (
    "Потрібна матриця ResNet18 на 3 фото. Спочатку виконайте попередню комірку цього розділу."
)
assert len(resnet_combined) == len(train_df)
assert len(resnet_combined_test) == len(test_ids)
assert "text_clean" in train_df.columns and "text_clean" in test_df.columns
assert N_CLASSES == 4, "Кумулятивні питання розраховані на класи 1, 2, 3, 4."


class JointDataset(Dataset):
    def __init__(self, features, labels=None):
        self.features = torch.tensor(np.asarray(features), dtype=torch.float32)
        self.labels = None if labels is None else torch.tensor(np.asarray(labels), dtype=torch.long)

    def __len__(self):
        return self.features.size(0)

    def __getitem__(self, index):
        if self.labels is None:
            return self.features[index]
        return self.features[index], self.labels[index]


class OrdinalHead(nn.Module):
    def __init__(self, in_dim, hidden, n_classes, dropout=JOINT_DROPOUT):
        super().__init__()
        self.net = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, n_classes - 1),
        )

    def forward(self, features):
        return self.net(features)


def cumulative_targets(labels, n_classes):
    levels = torch.arange(n_classes - 1, device=labels.device)
    return (labels.unsqueeze(1) > levels).float()


def class_distribution(logits):
    cumulative = 1.0 / (1.0 + np.exp(-logits))
    cumulative = np.minimum.accumulate(cumulative, axis=1)
    n_classes = cumulative.shape[1] + 1
    probabilities = np.empty((len(cumulative), n_classes), dtype=np.float64)
    probabilities[:, 0] = 1.0 - cumulative[:, 0]
    for level in range(1, n_classes - 1):
        probabilities[:, level] = cumulative[:, level - 1] - cumulative[:, level]
    probabilities[:, -1] = cumulative[:, -1]
    return probabilities


def decode_ordinal(logits):
    cumulative = 1.0 / (1.0 + np.exp(-logits))
    threshold = (cumulative > 0.5).sum(axis=1).astype(int)
    probabilities = class_distribution(logits)
    class_index = np.arange(probabilities.shape[1], dtype=np.float64)
    expectation = np.clip(np.rint(probabilities @ class_index), 0, probabilities.shape[1] - 1).astype(int)
    nonmonotone = float((np.diff(cumulative, axis=1) > 1e-6).any(axis=1).mean()) if len(cumulative) else 0.0
    return {"поріг": threshold, "очікування": expectation}, nonmonotone


def fit_ordinal_head(features_train, labels_train, features_val, labels_val):
    torch.manual_seed(SEED)
    model = OrdinalHead(features_train.shape[1], JOINT_HIDDEN, N_CLASSES).to(device)
    n_params = sum(parameter.numel() for parameter in model.parameters())
    optimizer = torch.optim.AdamW(model.parameters(), lr=JOINT_LR, weight_decay=JOINT_WEIGHT_DECAY)
    criterion = nn.BCEWithLogitsLoss()
    train_loader = DataLoader(
        JointDataset(features_train, labels_train),
        batch_size=JOINT_BATCH,
        shuffle=True,
        generator=torch.Generator().manual_seed(SEED),
    )
    val_loader = DataLoader(JointDataset(features_val, labels_val), batch_size=256, shuffle=False)
    best_qwk = -1.0
    best_epoch = 1
    best_decoder = "поріг"
    best_state = copy.deepcopy(model.state_dict())
    wait = 0
    epochs_ran = 0
    for epoch in range(1, JOINT_MAX_EPOCHS + 1):
        epochs_ran = epoch
        model.train()
        total_loss, seen = 0.0, 0
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            optimizer.zero_grad(set_to_none=True)
            logits = model(batch_x)
            loss = criterion(logits, cumulative_targets(batch_y, N_CLASSES))
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            total_loss += loss.item() * batch_y.size(0)
            seen += batch_y.size(0)
        model.eval()
        logit_chunks, true_chunks = [], []
        val_loss, val_seen = 0.0, 0
        with torch.no_grad():
            for batch_x, batch_y in val_loader:
                batch_x, batch_y = batch_x.to(device), batch_y.to(device)
                logits = model(batch_x)
                val_loss += criterion(logits, cumulative_targets(batch_y, N_CLASSES)).item() * batch_y.size(0)
                val_seen += batch_y.size(0)
                logit_chunks.append(logits.cpu().numpy())
                true_chunks.append(batch_y.cpu().numpy())
        decoded, _ = decode_ordinal(np.concatenate(logit_chunks))
        y_true = np.concatenate(true_chunks)
        qwk_threshold = quadratic_weighted_kappa(y_true, decoded["поріг"])
        qwk_expectation = quadratic_weighted_kappa(y_true, decoded["очікування"])
        epoch_decoder = "очікування" if qwk_expectation > qwk_threshold else "поріг"
        epoch_qwk = qwk_expectation if epoch_decoder == "очікування" else qwk_threshold
        print(
            f"Епоха {epoch:02d}/{JOINT_MAX_EPOCHS} | train loss {total_loss / max(seen, 1):.4f} | "
            f"val loss {val_loss / max(val_seen, 1):.4f} | "
            f"QWK поріг {qwk_threshold:.4f} | QWK очікування {qwk_expectation:.4f}"
        )
        if epoch_qwk > best_qwk + JOINT_MIN_DELTA:
            best_qwk = epoch_qwk
            best_epoch = epoch
            best_decoder = epoch_decoder
            best_state = copy.deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1
            if wait >= JOINT_PATIENCE:
                break
    model.load_state_dict(best_state)
    print(
        f"Зупинка після епохи {epochs_ran}. Найкраща епоха {best_epoch}, "
        f"QWK {best_qwk:.4f} ({best_decoder}). Параметрів {n_params}."
    )
    return model, best_qwk, best_epoch, best_decoder, epochs_ran, n_params


def predict_logits(model, features):
    model.eval()
    loader = DataLoader(JointDataset(features), batch_size=256, shuffle=False)
    chunks = []
    with torch.no_grad():
        for batch_x in loader:
            chunks.append(model(batch_x.to(device)).cpu().numpy())
    return np.concatenate(chunks)


def text_svd(frame_fit, frames_transform):
    vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=TFIDF_MIN_DF, max_features=TFIDF_MAX_FEATURES)
    sparse_fit = vectorizer.fit_transform(frame_fit)
    svd_dim = min(SVD_DIM, sparse_fit.shape[1] - 1, sparse_fit.shape[0] - 1)
    svd = TruncatedSVD(n_components=svd_dim, random_state=SEED)
    dense_fit = svd.fit_transform(sparse_fit)
    transformed = [svd.transform(vectorizer.transform(frame)) for frame in frames_transform]
    explained = float(svd.explained_variance_ratio_.sum())
    return dense_fit, transformed, explained, sparse_fit.shape[1], svd_dim


text_tr, (text_va,), explained, n_tfidf, svd_dim = text_svd(
    train_df.iloc[idx_tr]["text_clean"],
    [train_df.iloc[idx_va]["text_clean"]],
)
print(
    f"TF-IDF ознак {n_tfidf}, компонент SVD {svd_dim}, "
    f"пояснена дисперсія {explained:.3f}. Ваг класів немає. Втрата кумулятивна."
)
scaler = StandardScaler()
fused_tr = scaler.fit_transform(np.concatenate([resnet_combined[idx_tr], text_tr], axis=1)).astype(np.float32)
fused_va = scaler.transform(np.concatenate([resnet_combined[idx_va], text_va], axis=1)).astype(np.float32)
print("Спільна матриця train/val:", fused_tr.shape, fused_va.shape)

ordinal_model, ordinal_qwk, ordinal_best_epoch, ordinal_decoder, ordinal_epochs, ordinal_params = fit_ordinal_head(
    fused_tr, y_tr, fused_va, y_va
)
val_logits = predict_logits(ordinal_model, fused_va)
val_decoded, nonmonotone_share = decode_ordinal(val_logits)
val_scores = {name: quadratic_weighted_kappa(y_va, pred) for name, pred in val_decoded.items()}
chosen_pred = val_decoded[ordinal_decoder]
print(
    f"Exp13 QWK поріг {val_scores['поріг']:.4f}, очікування {val_scores['очікування']:.4f}. "
    f"Обрано {ordinal_decoder}, accuracy {accuracy_score(y_va, chosen_pred):.4f}."
)
print(f"Частка валідації, де P(клас > k) не спадає: {nonmonotone_share:.3f}.")
print("Прогнози на валідації:")
print(pd.Series(chosen_pred).map(IDX_TO_LABEL).value_counts().sort_index())
print(f"Порівняння: Exp5 {EXP5_VAL_QWK:.4f}, кумулятивна голова {val_scores[ordinal_decoder]:.4f}.")

exp13_name = "Exp13. Кумулятивна втрата + ResNet, SVD 128"
exp13_notes = (
    f"без ваг класів; svd {svd_dim}; пояснена дисперсія {explained:.3f}; "
    f"параметрів {ordinal_params}; поріг {val_scores['поріг']:.4f}; "
    f"очікування {val_scores['очікування']:.4f}"
)
if "log_experiment" in globals():
    log_experiment(
        exp13_name,
        val_scores[ordinal_decoder],
        ordinal_decoder,
        ordinal_epochs,
        ordinal_best_epoch,
        exp13_notes,
    )
else:
    print(
        exp13_name,
        round(val_scores[ordinal_decoder], 4),
        ordinal_decoder,
        "епох",
        ordinal_epochs,
        "найкраща",
        ordinal_best_epoch,
    )

if val_scores[ordinal_decoder] > EXP5_VAL_QWK + SUBMISSION_MARGIN:
    print("Кумулятивна голова вища за Exp5 більше ніж на 0.02. Навчаю її на повному train.")
    text_all, (text_test,), _, _, _ = text_svd(
        train_df["text_clean"],
        [test_df["text_clean"]],
    )
    final_scaler = StandardScaler()
    fused_all = final_scaler.fit_transform(np.concatenate([resnet_combined, text_all], axis=1)).astype(np.float32)
    fused_test = final_scaler.transform(np.concatenate([resnet_combined_test, text_test], axis=1)).astype(np.float32)
    torch.manual_seed(SEED)
    final_model = OrdinalHead(fused_all.shape[1], JOINT_HIDDEN, N_CLASSES).to(device)
    final_optimizer = torch.optim.AdamW(final_model.parameters(), lr=JOINT_LR, weight_decay=JOINT_WEIGHT_DECAY)
    final_criterion = nn.BCEWithLogitsLoss()
    final_loader = DataLoader(
        JointDataset(fused_all, y_all),
        batch_size=JOINT_BATCH,
        shuffle=True,
        generator=torch.Generator().manual_seed(SEED),
    )
    final_epochs = max(int(ordinal_best_epoch), 1)
    final_model.train()
    for epoch in range(1, final_epochs + 1):
        total_loss, seen = 0.0, 0
        for batch_x, batch_y in final_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            final_optimizer.zero_grad(set_to_none=True)
            logits = final_model(batch_x)
            loss = final_criterion(logits, cumulative_targets(batch_y, N_CLASSES))
            loss.backward()
            nn.utils.clip_grad_norm_(final_model.parameters(), 5.0)
            final_optimizer.step()
            total_loss += loss.item() * batch_y.size(0)
            seen += batch_y.size(0)
        print(f"Повне навчання, епоха {epoch}/{final_epochs}, train loss {total_loss / max(seen, 1):.4f}")
    test_logits = predict_logits(final_model, fused_test)
    test_idx = decode_ordinal(test_logits)[0][ordinal_decoder]
    submission = pd.DataFrame({
        "PetID": [normalize_id(pet_id) for pet_id in test_ids],
        "AdoptionSpeed": [IDX_TO_LABEL[int(index)] for index in test_idx],
    })
    submission = submission.drop_duplicates("PetID", keep="first")
    submission["AdoptionSpeed"] = submission["AdoptionSpeed"].astype(int)
    submission = submission[["PetID", "AdoptionSpeed"]]
    submission.to_csv(WORKING / "submission.csv", index=False)
    print(f"Збережено submission.csv, рядків {len(submission)}.")
    print(submission["AdoptionSpeed"].value_counts().sort_index())
else:
    print(
        "Новий submission.csv не записано: val QWK не вищий за Exp5 більше ніж на 0.02. "
        "Публічний файл зі score 0.6676 лишається чинним."
    )


### Висновки експерименту 13

Матриця та сама, що в Exp5: 641 ознака, SVD пояснює 0.227, голова 82 563 параметри. Ваг класів немає. Поріг по епохах: 0.421, 0.439, 0.458, 0.478, 0.469, 0.465. Очікування: 0.325, 0.375, 0.396, 0.414, 0.407, 0.419. Зупинка після 6-ї епохи, найкраща — 4-та. На збережених вагах val QWK 0.478 за порогом і 0.414 за очікуванням, accuracy 0.448. У 1.2% рядків валідації ймовірності «клас > k» не спадають, тож три відповіді майже скрізь узгоджені.

Val loss найнижчий на 3-й епосі, 0.505. На 4-й, де поріг дав 0.478, він уже 0.508. Train loss падав далі і на 6-й був 0.437. Це шкала бінарної крос-ентропії трьох відповідей.

Поріг ставить клас 1 у 150 випадках із 1287 (11.7% за частки 18.6% у train), клас 2 — 430 (33.4% за 27.6%), клас 3 — 398 (30.9% за 20.6%), клас 4 — 309 (24.0% за 33.2%). Клас 3, який у Exp5 був близько 12%, тут зібрав майже третину прогнозів. Точних збігів менше, ніж в Exp5, де accuracy 0.495, а далеких помилок теж менше, тому каппа вища.

Приріст над Exp5 — 0.014, менший за поріг 0.02. Приріст того самого порядку в Exp6 на публічному тесті дав 0.6524 замість 0.6676. Новий файл не записано. Чинним лишається Exp5.

Кумулятивна втрата зрушила прогнози з країв у класи 2 і 3 і підняла каппу, не змінивши енкодер. Для заміни файлу зі score 0.6676 цього замало.


## 21. Експеримент 14. Лише текст: нормалізація і поля з опису

Фото лишається як у Exp5, публічний score 0.6676. Цей розділ фото не читає і `submission.csv` не пише. Питання вузьке: чи піднімає val QWK самого тексту інше очищення або кілька чисел, які TF-IDF викидає.

Орієнтир — Exp1, 0.273 за argmax: уніграми й біграми очищеного опису, 20 000 ознак, логістична регресія з вагами класів `balanced`. Усі варіанти нижче мають ту саму регресію і те саме розбиття. Міняється лише текст.

1. **Очищений.** Повторення Exp1.
2. **Цифри.** Те саме очищення, але числа й одноцифрові токени лишаються.
3. **Сирий.** Нижній регістр і без URL. Стоп-слова не вирізаються, пунктуація не зчищається заздалегідь.
4. **Лише поля.** Без мішка слів: вік у місяцях, кількість тварин в оголошенні, довжина опису в словах, прапорці вакцинації, стерилізації, дегельмінтизації, терміновості, цуценяти або кошеняти, вагітності.
5. **Очищений і поля.** Мішок слів Exp1 плюс ці поля. Вік і кількість заповнюються медіаною лише тренувальної частини розбиття.

Як запускати на Kaggle: **не натискайте Run All**. Виконайте лише цей розділ зверху вниз. Друга комірка читає таблиці, якщо їх ще немає в пам'яті. Третя комірка вчить п'ять регресій. GPU не потрібен.


In [ ]:
import os
import random
import re
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import cohen_kappa_score
from sklearn.model_selection import train_test_split

SEED = 42
VAL_SIZE = 0.2
MAX_IMAGES = 3
FEATURE_BATCH_SIZE = 64
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
DATA_ROOT = Path("/kaggle/input/competitions/deep-learning-for-computer-vision-and-nlp-2026-10")
IMAGE_DIR = DATA_ROOT / "images"
WORKING = Path("/kaggle/working")
WORKING.mkdir(parents=True, exist_ok=True)
NUM_WORKERS = 2 if os.name != "nt" else 0
if "device" not in globals():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

needs_tables = "train_df" not in globals() or "idx_tr" not in globals() or "y_all" not in globals()
if not needs_tables:
    print(f"Таблиці вже в пам'яті: train {len(train_df)}, val {len(idx_va)}. Повторно не читаю.")
else:
    TRAIN_CSV = DATA_ROOT / "train.csv"
    TEST_CSV = DATA_ROOT / "test.csv"
    SAMPLE_CSV = DATA_ROOT / "sample_submission.csv"

    def find_column(columns, options):
        lookup = {str(column).lower(): column for column in columns}
        for option in options:
            if option.lower() in lookup:
                return lookup[option.lower()]
        return None

    def normalize_id(value):
        if pd.isna(value):
            return ""
        text = str(value).strip()
        if text.endswith(".0"):
            text = text[:-2]
        return text

    def read_table(path):
        frame = pd.read_csv(path)
        frame.columns = [str(column).strip() for column in frame.columns]
        return frame

    def quadratic_weighted_kappa(y_true, y_pred):
        y_true = np.asarray(y_true).astype(int)
        y_pred = np.asarray(y_pred).astype(int)
        if len(y_true) == 0:
            return 0.0
        score = cohen_kappa_score(y_true, y_pred, weights="quadratic")
        if score != score:
            return 0.0
        return float(score)

    train_df = read_table(TRAIN_CSV)
    test_df = read_table(TEST_CSV)
    sample_df = read_table(SAMPLE_CSV)
    id_col = find_column(train_df.columns, ["PetID", "pet_id", "id"])
    target_col = find_column(train_df.columns, ["AdoptionSpeed", "adoption_speed", "label", "target"])
    text_col = find_column(train_df.columns, ["Description", "description", "desc", "text", "comments"])
    test_id_col = find_column(test_df.columns, ["PetID", "pet_id", "id"]) or id_col
    train_df[id_col] = train_df[id_col].map(normalize_id)
    test_df[test_id_col] = test_df[test_id_col].map(normalize_id)
    if int(train_df[id_col].duplicated().sum()):
        train_df = train_df.drop_duplicates(id_col, keep="first")
    train_df = train_df.reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)
    if target_col in test_df.columns:
        test_df = test_df.drop(columns=[target_col])
    test_text_col = find_column(test_df.columns, ["Description", "description", "desc", "text", "comments"])
    if text_col is None:
        text_col = "Description"
        train_df[text_col] = ""
        test_df[text_col] = ""
    elif test_text_col is None:
        test_df[text_col] = ""
    elif test_text_col != text_col:
        test_df[text_col] = test_df[test_text_col]
    raw_target = train_df[target_col]
    if pd.api.types.is_numeric_dtype(raw_target):
        numeric_target = raw_target.astype(float)
        if np.allclose(numeric_target.to_numpy(), np.rint(numeric_target.to_numpy())):
            train_df[target_col] = numeric_target.round().astype(int)
    classes = sorted(train_df[target_col].dropna().unique().tolist())
    LABEL_TO_IDX = {label: index for index, label in enumerate(classes)}
    IDX_TO_LABEL = {index: label for label, index in LABEL_TO_IDX.items()}
    N_CLASSES = len(classes)
    train_df = train_df.dropna(subset=[target_col]).reset_index(drop=True)
    y_all = train_df[target_col].map(LABEL_TO_IDX).to_numpy(dtype=int)
    idx_tr, idx_va = train_test_split(
        np.arange(len(train_df)), test_size=VAL_SIZE, random_state=SEED, stratify=y_all
    )
    y_tr = y_all[idx_tr]
    y_va = y_all[idx_va]
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    print(f"Таблиці прочитано заново: train {len(train_df)}, val {len(idx_va)}, test {len(test_df)}.")

if "train_ids" not in globals():
    train_ids = train_df[id_col].tolist()
if "test_ids" not in globals():
    test_ids = test_df[test_id_col].tolist()

if len(idx_va) != 1287:
    print(f"Увага: у валідації {len(idx_va)} рядків, у минулому прогоні було 1287.")
else:
    print("Розбиття збігається з минулим прогоном: 1287 валідаційних рядків.")

if "text_clean" in train_df.columns and "text_clean" in test_df.columns:
    print("Очищений опис уже є, повторно не чищу.")
else:
    NEGATION_TOKENS = {
        "no", "not", "nor", "never", "none", "nobody", "nothing",
        "neither", "nowhere", "without", "cannot",
    }
    CONTRACTIONS = {
        "ain't": "is not",
        "aren't": "are not",
        "can't": "cannot",
        "couldn't": "could not",
        "didn't": "did not",
        "doesn't": "does not",
        "don't": "do not",
        "hadn't": "had not",
        "hasn't": "has not",
        "haven't": "have not",
        "he's": "he is",
        "here's": "here is",
        "how's": "how is",
        "i'm": "i am",
        "i've": "i have",
        "isn't": "is not",
        "it's": "it is",
        "let's": "let us",
        "mustn't": "must not",
        "she's": "she is",
        "that's": "that is",
        "there's": "there is",
        "they're": "they are",
        "they've": "they have",
        "we're": "we are",
        "we've": "we have",
        "weren't": "were not",
        "what's": "what is",
        "where's": "where is",
        "who's": "who is",
        "won't": "will not",
        "wouldn't": "would not",
        "you're": "you are",
        "you've": "you have",
    }
    CONTRACTION_RE = re.compile(
        r"\b(" + "|".join(sorted(map(re.escape, CONTRACTIONS), key=len, reverse=True)) + r")\b"
    )
    FALLBACK_STOPWORDS = set(
        "a about above after again against all am an and any are as at be because been "
        "before being below between both but by can did do does doing down during each "
        "few for from further had has have having he her here hers herself him himself "
        "his how i if in into is it its itself just me more most my myself of off on "
        "once only or other our ours ourselves out over own same she should so some "
        "such than that the their theirs them themselves then there these they this "
        "those through to too under until up very was we were what when where which "
        "while who whom why will with you your yours yourself yourselves".split()
    )

    def load_stopwords():
        try:
            import nltk
            from nltk.corpus import stopwords

            nltk.download("stopwords", quiet=True)
            words = set(stopwords.words("english"))
            print("Стоп-слова: NLTK, english")
        except Exception as exc:
            print(f"Стоп-слова NLTK недоступні ({type(exc).__name__}: {exc}). Використовую вбудований список.")
            words = set(FALLBACK_STOPWORDS)
        return words - NEGATION_TOKENS

    STOPWORDS = load_stopwords()

    def clean_text(value):
        if not isinstance(value, str):
            return ""
        text = value.lower().replace("’", "'").replace("`", "'")
        text = CONTRACTION_RE.sub(lambda match: CONTRACTIONS[match.group(0)], text)
        text = re.sub(r"https?://\S+|www\.\S+", " ", text)
        text = re.sub(r"[^a-z\s]", " ", text)
        tokens = [token for token in text.split() if len(token) > 1 and token not in STOPWORDS]
        return " ".join(tokens)

    train_df["text_clean"] = train_df[text_col].fillna("").astype(str).map(clean_text)
    test_df["text_clean"] = test_df[text_col].fillna("").astype(str).map(clean_text)
    print("Порожніх описів після очищення:", float(train_df["text_clean"].eq("").mean()))

print("Пристрій:", device)


In [ ]:
import re

import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler

TFIDF_MAX_FEATURES = 20000
TFIDF_MIN_DF = 2
TEXT_BASELINE_QWK = 0.2730
AGE_RE = re.compile(
    r"(\d+(?:\.\d+)?)\s*(months?|weeks?|years?|days?|yrs?|mths?)",
    re.IGNORECASE,
)
COUNT_RE = re.compile(
    r"\b(\d{1,2})\s+(?:\w+\s+){0,2}?(?:kittens?|puppies|puppys|cats?|dogs?|tabbys?|babies|pups?|pets?)\b",
    re.IGNORECASE,
)
TOTAL_RE = re.compile(r"\btotal of\s+(\d{1,2})\b", re.IGNORECASE)
FLAG_RES = {
    "vaccinated": re.compile(r"vaccinat", re.IGNORECASE),
    "sterilized": re.compile(r"steril|neuter|spay", re.IGNORECASE),
    "dewormed": re.compile(r"deworm", re.IGNORECASE),
    "urgent": re.compile(r"urgent", re.IGNORECASE),
    "young": re.compile(r"kitten|puppy", re.IGNORECASE),
    "pregnant": re.compile(r"pregnant", re.IGNORECASE),
}

assert "text_clean" in train_df.columns
assert N_CLASSES == 4


def decode_probabilities(probs, n_classes):
    class_index = np.arange(n_classes, dtype=np.float64)
    expectation = np.clip(np.rint(probs @ class_index), 0, n_classes - 1).astype(int)
    return {"argmax": probs.argmax(axis=1).astype(int), "expectation": expectation}


def clean_text_keep_digits(value):
    if not isinstance(value, str):
        return ""
    text = value.lower().replace("\u2019", "'").replace("`", "'")
    if "CONTRACTION_RE" in globals() and "CONTRACTIONS" in globals():
        text = CONTRACTION_RE.sub(lambda match: CONTRACTIONS[match.group(0)], text)
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    stopwords = STOPWORDS if "STOPWORDS" in globals() else set()
    tokens = [
        token
        for token in text.split()
        if (len(token) > 1 or token.isdigit()) and token not in stopwords
    ]
    return " ".join(tokens)


def normalize_raw_text(value):
    if not isinstance(value, str):
        return ""
    text = value.lower().replace("\u2019", "'").replace("`", "'")
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def parse_age_months(value):
    if not isinstance(value, str):
        return np.nan
    match = AGE_RE.search(value)
    if match is None:
        return np.nan
    number = float(match.group(1))
    unit = match.group(2).lower()
    if unit.startswith("week"):
        return number / 4.345
    if unit.startswith("year") or unit.startswith("yr"):
        return number * 12.0
    if unit.startswith("day"):
        return number / 30.0
    return number


def parse_pet_count(value):
    if not isinstance(value, str):
        return np.nan
    found = [int(item) for item in COUNT_RE.findall(value)]
    total = TOTAL_RE.search(value)
    if total is not None:
        found.append(int(total.group(1)))
    found = [item for item in found if 1 <= item <= 30]
    if not found:
        return np.nan
    return float(max(found))


def score_classifier(name, x_train, x_val):
    classifier = LogisticRegression(
        max_iter=500,
        class_weight="balanced",
        random_state=SEED,
    )
    classifier.fit(x_train, y_tr)
    raw_proba = classifier.predict_proba(x_val)
    probabilities = np.zeros((len(y_va), N_CLASSES), dtype=np.float64)
    for column, class_index in enumerate(classifier.classes_):
        probabilities[:, int(class_index)] = raw_proba[:, column]
    decoded = decode_probabilities(probabilities, N_CLASSES)
    scores = {
        decoder: quadratic_weighted_kappa(y_va, prediction)
        for decoder, prediction in decoded.items()
    }
    chosen = "expectation" if scores["expectation"] > scores["argmax"] else "argmax"
    prediction = decoded[chosen]
    print(
        f"{name} | ознак {x_train.shape[1]} | "
        f"QWK argmax {scores['argmax']:.4f} | QWK expectation {scores['expectation']:.4f} | "
        f"обрано {chosen} | accuracy {accuracy_score(y_va, prediction):.4f}"
    )
    print(pd.Series(prediction).map(IDX_TO_LABEL).value_counts().sort_index())
    notes = f"argmax {scores['argmax']:.4f}; expectation {scores['expectation']:.4f}; ознак {x_train.shape[1]}"
    exp_name = f"Exp14. {name}"
    if "log_experiment" in globals():
        log_experiment(exp_name, scores[chosen], chosen, 0, 0, notes)
    else:
        print(exp_name, round(scores[chosen], 4), chosen)
    return {"name": name, "qwk": scores[chosen], "decoder": chosen, "scores": scores}


def tfidf_matrices(documents):
    vectorizer = TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=TFIDF_MIN_DF,
        max_features=TFIDF_MAX_FEATURES,
        token_pattern=r"(?u)\b\w+\b",
    )
    return (
        vectorizer.fit_transform(documents.iloc[idx_tr]),
        vectorizer.transform(documents.iloc[idx_va]),
    )


raw_train = train_df[text_col].fillna("").astype(str)
digits_train = raw_train.map(clean_text_keep_digits)
plain_train = raw_train.map(normalize_raw_text)

age = raw_train.map(parse_age_months).to_numpy(dtype=float)
pet_count = raw_train.map(parse_pet_count).to_numpy(dtype=float)
word_count = raw_train.str.split().map(len).to_numpy(dtype=float)
flag_frame = pd.DataFrame({
    name: raw_train.map(lambda text, pattern=pattern: float(pattern.search(text) is not None))
    for name, pattern in FLAG_RES.items()
})
print(
    f"Вік розпізнано в {int(np.isfinite(age).sum())} із {len(age)} описів, "
    f"медіана {np.nanmedian(age):.1f} міс."
)
print(f"Кількість тварин розпізнано в {int(np.isfinite(pet_count).sum())} із {len(pet_count)} описів.")
print("Частки прапорців:")
print(flag_frame.mean().round(3))

field_values = np.column_stack([age, pet_count, word_count, flag_frame.to_numpy(dtype=float)])
imputer = SimpleImputer(strategy="median")
scaler = StandardScaler()
fields_tr = scaler.fit_transform(imputer.fit_transform(field_values[idx_tr]))
fields_va = scaler.transform(imputer.transform(field_values[idx_va]))

results = []
clean_tr, clean_va = tfidf_matrices(train_df["text_clean"])
results.append(score_classifier("Очищений", clean_tr, clean_va))
digit_tr, digit_va = tfidf_matrices(digits_train)
results.append(score_classifier("Цифри", digit_tr, digit_va))
raw_tr, raw_va = tfidf_matrices(plain_train)
results.append(score_classifier("Сирий", raw_tr, raw_va))
results.append(score_classifier("Лише поля", fields_tr, fields_va))
results.append(score_classifier(
    "Очищений і поля",
    sparse.hstack([clean_tr, sparse.csr_matrix(fields_tr)], format="csr"),
    sparse.hstack([clean_va, sparse.csr_matrix(fields_va)], format="csr"),
))

best = max(results, key=lambda row: (row["qwk"], row["scores"]["argmax"]))
print(
    "Exp14 підсумок: "
    + "; ".join(f"{row['name']} {row['qwk']:.4f} {row['decoder']}" for row in results)
    + f". Кращий за текстом: {best['name']}."
)
print(
    f"Орієнтир Exp1 {TEXT_BASELINE_QWK:.4f}. "
    "submission.csv не записано: фото й файл зі score 0.6676 лишаються чинними."
)


### Висновки експерименту 14

Очищений варіант повторив Exp1: val QWK 0.273 за argmax, очікування 0.228, accuracy 0.424. Прогнози на 1287 рядках: клас 1 — 276, 2 — 280, 3 — 258, 4 — 473.

Цифри знизили каппу до 0.266, accuracy 0.417. Сирий опис підняв каппу до 0.279 при accuracy 0.416. Прогнози сирого: 1 — 266, 2 — 298, 3 — 273, 4 — 450. Приріст над очищеним — 0.006.

Вік у місяцях розпізнано в 1509 із 6431 описів, медіана 3 місяці. Кількість тварин — у 769. Прапорці: вакцинація 15.3%, стерилізація 21.9%, дегельмінтизація 9.0%, терміновість 2.9%, цуценя або кошеня 31.1%, вагітність 2.1%. Самі 9 полів дають val QWK 0.149 і accuracy 0.345. Клас 2 — 39 прогнозів, клас 4 — 718.

Кращий текст прогону — очищений мішок слів разом із полями: val QWK 0.283 за argmax, очікування 0.233, accuracy 0.416. Прогнози: 1 — 277, 2 — 282, 3 — 252, 4 — 476. Приріст над Exp1 — 0.010. Гістограма майже та сама, що в очищеного тексту.

submission.csv не записано. Файл зі score 0.6676 лишається. Приріст 0.010 належить лише тексту і не змінює фотомодель.
